In [ ]:
# %load ../../notebooks/init.ipy
%reload_ext autoreload
%autoreload 2

# Builtin packages
from datetime import datetime
from importlib import reload
import logging
import os
from pathlib import Path
import sys
import warnings
from copy import copy
# standard secondary packages
import astropy as ap
import h5py
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import scipy as sp
import scipy.stats
import tqdm.notebook as tqdm


# development packages
import kalepy as kale
import kalepy.utils
import kalepy.plot

# --- Holodeck ----
import holodeck as holo
from holodeck import cosmo, utils, plot
from holodeck.constants import MSOL, PC, YR, MPC, GYR, SPLC, NWTG, SCHW
from holodeck.utils import _AGE_UNIVERSE_GYR
import holodeck.sams
import holodeck.gravwaves
from holodeck.hardening import allowed_param_range
#import holodeck.evolution
#import holodeck.population

# Silence annoying numpy errors
np.seterr(divide='ignore', invalid='ignore', over='ignore')
warnings.filterwarnings("ignore", category=UserWarning)

# Plotting settings
mpl.rc('font', **{'family': 'serif', 'sans-serif': ['Times'], 'size': 15})
mpl.rc('lines', solid_capstyle='round')
mpl.rc('mathtext', fontset='cm')
mpl.style.use('default')   # avoid dark backgrounds from dark theme vscode
plt.rcParams.update({'grid.alpha': 0.5})

# Load log and set logging level
log = holo.log
# log.setLevel(logging.INFO)
#log.setLevel(logging.DEBUG)
#log.setLevel(log.DEBUG)
#log.level, log.DEBUG, log.INFO


# ---- Define filepath containing simulation galaxy merger data files ----#
# ---- (if using files not in _PATH_DATA) ---- #
_HOME_PATH = Path('~/').expanduser()
p = os.path.join(_HOME_PATH, 'cosmo_sim_merger_data')
if os.path.exists(p):
    _SIM_MERGER_PATH = p
else:
    p = os.path.join(_HOME_PATH, 'nanograv/cosmo_sim_merger_data')
    if os.path.exists(p):
        _SIM_MERGER_PATH = p
    else:
        _SIM_MERGER_PATH = _PATH_DATA
#_SIM_MERGER_PATH = _PATH_DATA
print(f"{_SIM_MERGER_PATH=}")
# ------------------------------------------------------------------------ #

#SPEED_LIMIT = 0.1 * SPLC

freqs, freqs_edges = utils.pta_freqs()


In [ ]:
def get_sam_dadt(nrads=100, shape=None, gsmf_flag=2, gpf_flag=0, tau=1.0, 
                 ainit=1.0e4, rc=100.0, nu_in=-1.0, nu_out=+2.5,
                 calc_gwb=False, nreals=10, nloud=1):
    
    if gsmf_flag == 1:
        _gsmf = holo.sams.GSMF_Schechter()
    elif gsmf_flag == 2:
        _gsmf = holo.sams.GSMF_Double_Schechter()
    else: 
        raise ValueError('keyword `gsmf_flag` must be 1 for single Schecter or 2 for double Schechter')
        
    if gpf_flag: 
        _gpf = holo.sams.GPF_Power_Law()
    else:
        _gpf = None
        
    sam = holo.sams.Semi_Analytic_Model(
        mtot = (1.0e4*MSOL, 1.0e12*MSOL, 91),
        #mtot = (1.0e5*MSOL, 1.0e11*MSOL, 91),
        #mtot = (1.0e9*MSOL, 1.0e10*MSOL, 91),
        shape = shape,
        gsmf = _gsmf,
        gpf = _gpf
    )
    
    hard = holo.hardening.Fixed_Time_2PL_SAM(
        sam, tau * GYR,
        sepa_init = ainit * PC,
        rchar = rc * PC,
        gamma_inner = nu_in,
        gamma_outer = nu_out
    )
    #print("checking status of class hard")
    #print(hard)
    #print(dir(hard))
    #print(vars(hard))
    #print(issubclass(hard,holo.hardening._Hardening))

    #print(f"before defining radii: {sam.mtot.shape=} {sam.mrat.shape=} {sam.redz.shape=}")
    # () start from the hardening model's initial separation
    rmax = hard._sepa_init
    # (M,) end at the ISCO
    rmin = utils.rad_isco(sam.mtot)
    # rmin = hard._TIME_TOTAL_RMIN * np.ones_like(sam.mtot)
    # Choose steps for each binary, log-spaced between rmin and rmax
    extr = np.log10([rmax * np.ones_like(rmin), rmin])
    radii = np.linspace(0.0, 1.0, nrads)[np.newaxis, :]
    # (M, X)
    radii = extr[0][:, np.newaxis] + (extr[1] - extr[0])[:, np.newaxis] * radii
    radii = 10.0 ** radii
    # (M, Q, Z, X)
    mt, mr, rz, rads = np.broadcast_arrays(
        sam.mtot[:, np.newaxis, np.newaxis, np.newaxis],
        sam.mrat[np.newaxis, :, np.newaxis, np.newaxis],
        sam.redz[np.newaxis, np.newaxis, :, np.newaxis],
        radii[:, np.newaxis, np.newaxis, :]
    )
    # (X, M*Q*Z)
    #mt, mr, rz, rads = [mm.reshape(-1, STEPS).T for mm in [mt, mr, rz, rads]]
    #print(f'{sam.mtot.shape=}, {sam.mrat.shape=}, {sam.redz.shape=}, {radii.shape=}')
    #print(f'{mt.shape=}, {mr.shape=}, {rz.shape=}, {rads.shape=}')
    #print('Mtot=', sam.mtot/MSOL)
    #print('q=', sam.mrat)
    #print('redz=', sam.redz)

    # old: (X, M*Q*Z) --- `Fixed_Time.dadt` will only accept this shape
    # new: (M, Q, Z, X) is shape of input and output arrays
    dadt = hard.dadt(mt, mr, rads)
    #print(f"{mt.shape=}, {mr.shape=}, {dadt.shape=}, {rads.shape=}")

    if calc_gwb:
        freqs, freqs_edges = utils.pta_freqs()
        gwb = sam.gwb(freqs_edges, hard, realize=nreals, loudest=nloud, params=True)
        # returns hc_ss, hc_bg, sspar, bgpar as tuple
        return sam, hard, rads, dadt, gwb
    else:
        return sam, hard, rads, dadt

    
def get_sam_newhard_dadt(nrads=100, shape=None, gsmf_flag=2, gpf_flag=0,
                         tau_outer=1.0, rch9=100, alphach=-1, nu_in=-1.0, 
                         dadt_rch=None, gwc_units='rg', 
                         rgw9=1e3, alphagw=0, betagw=0,
                         in_time=None,
                         in_mod_type=0,
                         mtot_range=None, mrat_range=None,
                         calc_gwb=False, nreals=10, nloud=1):
    if gpf_flag: 
        _gpf = sams.GPF_Power_Law()
    else:
        _gpf = None
    
    if gsmf_flag == 1:
        _gsmf = holo.sams.GSMF_Schechter()
    elif gsmf_flag == 2:
        _gsmf = holo.sams.GSMF_Double_Schechter()
    else: 
        raise ValueError('keyword `gsmf_flag` must be 1 for single Schecter or 2 for double Schechter')
        
    # if `shape` is set, will override the hard-coded value here:
    if mtot_range is None:
        _mtot = (1.0e4*MSOL, 1.0e12*MSOL, 91)
    else:
        if len(mtot_range) != 2:
            raise ValueError("keyword `mtot_range` must have length 2.")
        _mtot = (mtot_range[0], mtot_range[1], 91)

    # if `shape` is set, will override the hard-coded value here:
    if mrat_range is None:
        _mrat=(1e-3, 1.0, 81)
    else:
        if len(mrat_range) != 2:
            raise ValueError("keyword `mrat_range` must have length 2.")
        _mrat = (mrat_range[0], mrat_range[1], 81)
    
    sam = holo.sams.Semi_Analytic_Model(
        mtot = _mtot,
        mrat = _mrat,
        shape = shape,
        gsmf = _gsmf,
        gpf = _gpf
    )
    #OLD:
    #newhard = holo.hardening.FixedOuterTime_InnerPL_SAM(
    #    sam, 
    #    tau_outer * GYR,
    #    rchar = rch * PC,
    #    gamma_inner = nu_in,
    #    x_gw_crit = xcrit,
    #    num_steps = nrads
    #)   
    #NEW:
    #self, sam, num_steps=300, outer_time=1.0*GYR, rchar=100.0*PC, 
    #             gamma_inner=-1.0, x_gw_crit=1e3, gw_crit_units='rg',
    #             r_gw_crit_9=0.01*PC, alpha_gw_crit=0.75, dadt_rchar=None, 
    #             inner_time=None,
    #             inner_model_type=0):
    print(f"{nrads=}")
    newhard = holo.hardening.FixedOuterTime_InnerPL_SAM(
        sam, 
        num_steps = nrads,
        outer_time = tau_outer * GYR,
        rchar_9 = rch9 * PC,
        alpha_char = alphach,
        nu_inner = nu_in,
        gw_crit_units = gwc_units,
        r_gw_crit_9 = rgw9, 
        alpha_gw_crit = alphagw, 
        beta_gw_crit = betagw,
        dadt_rchar=dadt_rch,
        inner_time = in_time,
        inner_model_type=in_mod_type,
        fobs_min=None
    )
    #print("checking status of class newhard")
    #print(newhard)
    #print(dir(newhard))
    #print(vars(newhard))
    #print(issubclass(newhard,holo.hardening._Hardening))

    # () start from the inner hardening model's initial separation
    rmax = newhard._rchar_9 * (sam.mtot/(1.0e9*MSOL))**(newhard._alpha_char+1)
    print(f"{newhard._rchar_9=} {newhard._rchar_9/PC=}")
    # (M,) end at the ISCO
    rmin = utils.rad_isco(sam.mtot)
    # Choose steps for each binary, log-spaced between rmin and rmax
    extr = np.log10([rmax * np.ones_like(rmin), rmin])
    radii = np.linspace(0.0, 1.0, nrads)[np.newaxis, :]
    # (M, X)
    radii = extr[0][:, np.newaxis] + (extr[1] - extr[0])[:, np.newaxis] * radii
    radii = 10.0 ** radii
    # (M, Q, Z, X)
    mt, mr, rz, rads = np.broadcast_arrays(
        sam.mtot[:, np.newaxis, np.newaxis, np.newaxis],
        sam.mrat[np.newaxis, :, np.newaxis, np.newaxis],
        sam.redz[np.newaxis, np.newaxis, :, np.newaxis],
        radii[:, np.newaxis, np.newaxis, :]
    )

    # old: (X, M*Q*Z) --- `Fixed_Time.dadt` will only accept this shape
    # new: (M, Q, Z, X) is shape of input and output arrays
    dadt, agw_crit, rz_char, rz_final = newhard.dadt(mt, mr, rz, rads)
    print(f"{mt.shape=}, {mr.shape=}, {rz.shape=}, {dadt.shape=}, {rads.shape=}")
    print(f"{dadt.shape=}, {rz_char.shape=}, {rz_final.shape=}")
    print(f"*** {rads.max()=} {rads.min()=} ***")
    if calc_gwb:
        freqs, freqs_edges = utils.pta_freqs()
        gwb = sam.gwb(freqs_edges, hard=newhard, realize=nreals, loudest=nloud, params=True)
        # returns hc_ss, hc_bg, sspar, bgpar as tuple
        
        return sam, newhard, rads, dadt, agw_crit, rz_char, rz_final, gwb
    else:
        return sam, newhard, rads, dadt, agw_crit, rz_char, rz_final
    

def sepa_emit(mtot, fgw):
    """
    separation of an equal-mass circular binary with total mass mtot emitting GWs at frequency fgw
    
    assumes mtot in cgs and fgw in Hz, returns separation in cm
    """
    #print(f'{MSOL=}, {mtot=}, {fgw=}, {NWTG=}')
    return ( NWTG * mtot / (fgw * np.pi) **2 )**(1.0/3)

def freq_emit(mtot, sepa):
    """
    emitted GW frequency of an equal-mass circular binary with total mass mtot at separation sepa
    
    assumes mtot in cgs and separation in cm, returns fgw in Hz
    """
    #print(f'{MSOL=}, {mtot=}, {fgw=}, {NWTG=}')
    return np.sqrt( NWTG * mtot / (sepa**3 * np.pi**2) )

### HERE
def calc_and_plot_dadt(sam_data, distance_units='pc', fixedTime='total', 
                       Tobs_yr=None, fobs_min=1.0e-9,
                       max_to_plot = 4, extra_panels=False, verbose=False):
    
    if distance_units == 'pc':
        xlim=[1e-8,1e5]
        xlbl = f'binary separation [{distance_units}]' 
        inv_axis = True
    elif distance_units == 'rg':
        xlim=[1,1e13]
        xlbl = f'binary separation [{distance_units}]'        
        inv_axis = True
    elif distance_units == 'forb':
        xlim=[1.0e-12,1.0e-1]
        xlbl = f'forb(binary separation) [Hz]'        
        inv_axis = False
    else:
        raise ValueError(f"invalid keyword {distance_units=}. must be 'pc', 'rg', or 'forb'.")
    
    if fixedTime not in ['total','outer']:
        raise ValueError(f"keyword `fixedTime` must be 'total' or 'outer'.")

    if Tobs_yr is None and fobs_min is None:
        raise ValueError("keywords Tobs_yr and fobs_min cannot both be None.")
    
    if Tobs_yr is not None and fobs_min is not None:
        raise ValueError("one of Tobs_yr and fobs_min must be None.")
    
    # Make the plot
    if extra_panels:
        fig = plt.figure(figsize=(12,9))
        first_plot_index = 231
    else:
        fig = plt.figure(figsize=(12,4))
        first_plot_index = 131
        
    ax1 = fig.add_subplot(first_plot_index)
    plt.xscale('log')
    plt.xlim(xlim[0],xlim[1])
    plt.yscale('log')
    ax1.xaxis.set_inverted(inv_axis)
    plt.xlabel(xlbl)
    plt.ylabel('hardening tscale [yr]')
    #plt.ylim(1e4,1e10)

    ax2 = fig.add_subplot(first_plot_index+1)
    plt.xscale('log')
    plt.xlim(xlim[0],xlim[1])
    plt.yscale('log')
    ax2.xaxis.set_inverted(inv_axis)
    plt.xlabel(xlbl)
    plt.ylabel('hardening rate [cm/s]')
    #plt.ylim(1e4,1e10)

    ax3 = fig.add_subplot(first_plot_index+2)
    plt.xscale('log')
    plt.xlim(xlim[0],xlim[1])
    plt.yscale('log')
    ax3.xaxis.set_inverted(inv_axis)
    plt.xlabel(xlbl)
    plt.ylabel('cumulative time [yr]')

    if extra_panels:
        ax4 = fig.add_subplot(first_plot_index+3)
        plt.xscale('log')
        plt.xlim(xlim[0],xlim[1])
        plt.yscale('log')
        ax4.xaxis.set_inverted(inv_axis)
        plt.xlabel(xlbl)
        plt.ylabel(r's (hardening parameter) ')

        ax5 = fig.add_subplot(first_plot_index+4)
        plt.xscale('log')
        plt.xlim(xlim[0],xlim[1])
        plt.yscale('log')
        #ax5.xaxis.set_inverted(True)
        plt.xlabel(r'<s_inner> [pc Myr]$^{-1}$')
        plt.ylabel('cumulative time [yr]')

        #ax4 = fig.add_subplot(first_plot_index+3)
        #plt.xscale('log')
        #plt.xlim(xlim[0],xlim[1])
        #plt.yscale('log')
        #ax4.xaxis.set_inverted(True)
        #plt.xlabel(f'a_GW,crit [{distance_units}]')
        #plt.ylabel('time from fobs=1/30yr to ISCO [yr]')

        #ax5 = fig.add_subplot(first_plot_index+4)
        #plt.xscale('log')
        #plt.xlim(xlim[0],xlim[1])
        #plt.yscale('log')
        #ax5.xaxis.set_inverted(True)
        #plt.xlabel(f'a_GW,crit [{distance_units}]')
        #plt.ylabel('a(fobs=1/30yr) / a_GW,crit')

    
    cmap_arr = ['Blues', 'Oranges',  'Greens', 'Reds', 'Purples',
                'Greys', 'YlOrBr', 'YlOrRd', 'OrRd', 'PuRd', 'RdPu', 'BuPu',
                'GnBu', 'PuBu', 'YlGnBu', 'PuBuGn', 'BuGn', 'YlGn']*10
    fgw_ls = ['-','-.',':']
    lhandles = []
    flhandles = []
    
    nu_mrk = ['s','^','o','*']
    
    for n,sd in enumerate(sam_data):
        
        if fixedTime=='total':
            if len(sd) == 4:
                sam, hard, rads, dadt = sd
            elif len(sd) == 5:
                sam, hard, rads, dadt, gwb = sd
            else:
                raise ValueError(f"sam_data has unexpected length {len(sd)}. must be 4 or 5 for `fixedTime`='total'.")
                
            agw = calc_aGW_for_Fixed_Time_2PL(hard, sam)
            #print(f"{hard._norm.shape=} {agw.shape=}")
            
        else: 
            # fixedTime == 'outer'
            if len(sd) == 7:
                sam, hard, rads, dadt, agw, rzch, rzf = sd
            elif len(sd) == 8:
                sam, hard, rads, dadt, agw, rzch, rzf, gwb = sd
            else:
                raise ValueError(f"sam_data has unexpected length {len(sd)}. must be 7 or 8 for `fixedTime`='outer'")

        log.info(f"{dadt.min()=} {dadt.max()=}")
        log.info(f"{rads.min()=} {rads.max()=}")
        
        #print(f"{sam.mtot.shape=}, {sam.mrat.shape=}")
        mt_nskip = int((sam.mtot.size-1)/(max_to_plot-1)) if sam.mtot.size>max_to_plot else 1
        mr_nskip = int((sam.mrat.size-1)/(max_to_plot-1)) if sam.mrat.size>max_to_plot else 1
  
        times_evo = calc_cumulative_thard(sd, rads[0,0,0,0], rads[0,0,0,-1],fixedTime=fixedTime)
        #times_evo = -utils.trapz_loglog(-1.0 / dadt[:,:,0,:], rads[:,:,0,:], axis=2, cumsum=True)
        log.info(f"{times_evo.min()=} {times_evo.max()=}")
         
        cmap = plot._get_cmap(cmap_arr[n])
        colors = cmap(np.linspace(0.3, 1, max_to_plot+1))
        lw = np.arange(0.5,max_to_plot+1, 0.5)

        freqs, freqs_edges = utils.pta_freqs()

        if verbose:
            print(f"*** in plotting function: {rads.min()=} {rads.max()=}")
            print(f"{mt_nskip=}, {mr_nskip=}")
            print(f"{rads[0,0,0,0]=}, {rads[0,0,0,-1]=}")
            print(freqs_edges.min(),freqs_edges.max())
            if n==0:
                print('Mtot=', sam.mtot/MSOL)
                print('q=', sam.mrat)
                print('redz=', sam.redz)

        if fobs_min is None:
            fobs_min = 1 / (Tobs_yr*YR)
        
        i_plot = 0
        for i in np.arange(0,sam.mtot.size,mt_nskip):

            if distance_units == 'pc':
                xx_obs_max = sepa_emit(sam.mtot[i],fobs_min) / PC
            elif distance_units == 'rg':
                xx_obs_max = sepa_emit(sam.mtot[i],fobs_min) / (NWTG * sam.mtot[i] / SPLC**2)
            elif distance_units == 'forb':
                xx_obs_max = 0.5*fobs_min
                
            #print(f'Mtot = {sam.mtot[i]/MSOL:.2g}')


            #frst_min = utils.frst_from_fobs(freqs_edges.min(), sam.redz.min())
            #print(f"{len(colors)=}")
            flmi,= ax1.plot([xx_obs_max,xx_obs_max],[1e-4,1e10],ls='-',
                            alpha=0.7,color=colors[i_plot],label=f'frst={fobs_min:.2g}Hz')
            if i_plot==max_to_plot and n==len(sam_data)-1:
                flhandles += [flmi]
                    
            j_plot=0
            for j in np.arange(0,sam.mrat.size,mr_nskip):

                vorb = np.sqrt( NWTG * sam.mtot[i] / rads[i,j,0,:] )
                ix_gt_vorb = (-dadt[i,j,0,:] > vorb)
                
                if distance_units=='pc':
                    xx = rads[i,j,0,:]/PC
                elif distance_units=='rg':
                    xx = rads[i,j,0,:] / (NWTG * sam.mtot[i] / SPLC**2)
                else:
                    xx = 0.5 * freq_emit(sam.mtot[i], rads[i,j,0,:])
                    
                if fixedTime=='total':
                    l,= ax1.plot(xx, -rads[i,j,0,:]/dadt[i,j,0,:]/YR, 
                                 alpha=0.5, color=colors[i_plot], lw=lw[j_plot], 
                                 label=f'tau={hard._target_time/GYR}')
                    if i_plot==max_to_plot and j_plot==max_to_plot:
                        lhandles += [l]
                else: 
                    #print(f"*** thard(rchar) = {-rads[i,j,0,0]/dadt[i,j,0,0]/YR} yr")
                    ax1.plot(xx, -rads[i,j,0,:]/dadt[i,j,0,:]/YR, 
                             alpha=0.5, color=colors[i_plot], lw=lw[j_plot])

                if j_plot==max_to_plot and n==len(sam_data)-1:
                    ax2.plot(xx, -dadt[i,j,0,:], 
                             alpha=0.5, color=colors[i_plot], lw=lw[j_plot], 
                             label=f'mtot={sam.mtot[i]/MSOL:.2g}')
                else:
                    ax2.plot(xx, -dadt[i,j,0,:], 
                             alpha=0.5, color=colors[i_plot], lw=lw[j_plot],label=None)
                ax2.plot(xx[ix_gt_vorb], -dadt[i,j,0,ix_gt_vorb], 'mo', markersize=0.5)
                if fixedTime=='total':
                    ax2.plot([agw[i,j]/ (NWTG * sam.mtot[i] / SPLC**2),agw[i,j]/ (NWTG * sam.mtot[i] / SPLC**2)], 
                             [-5,10],'m')

                print(f"mt={sam.mtot[i]/MSOL:.2g}, mr={sam.mrat[j]:.2g}, tau_mrg={times_evo[i,j,-1]/YR:.2g}")
                ax3.plot(xx[:-1], times_evo[i,j,:]/YR, 
                         alpha=0.5, color=colors[i_plot], lw=lw[j_plot])
                
                j_plot += 1

                if extra_panels:

                    hard_param_s = -dadt[i,j,0,:]/rads[i,j,0,:]**2*PC*YR*1.0e6 # 1/(pc*Myr)
                    avg_hard_param_s = hard_param_s.mean()
                
                    if j_plot==max_to_plot and n==len(sam_data)-1:
                        ax4.plot(xx, hard_param_s, 
                                 alpha=0.5, color=colors[i_plot], lw=lw[j_plot],
                                 label=f'mtot={sam.mtot[i]/MSOL:.2g}')
                    else:
                        ax4.plot(xx, hard_param_s, 
                                 alpha=0.5, color=colors[i_plot], lw=lw[j_plot],label=None)
                    ax5.scatter(avg_hard_param_s, times_evo[i,j,-1]/YR)

            i_plot += 1
        
        if fixedTime=='total':
            ax1.plot(xlim, [hard._target_time/YR, hard._target_time/YR], '--', color='darkgray')
            if distance_units=='pc': ax1.plot([hard._rchar/PC, hard._rchar/PC], [1e-2,1e10],'k--')
            if distance_units=='pc': ax2.plot([hard._rchar/PC, hard._rchar/PC], [10,1e11],'k--')
    ax2.plot(xlim, [3.0e10,3.0e10], color='magenta')
    ax3.plot(xlim, [_AGE_UNIVERSE_GYR*1e9,_AGE_UNIVERSE_GYR*1e9], 'k:')
    leg2 = ax1.legend(handles=flhandles, loc='lower right')
    ax1.legend(handles=lhandles, loc='lower left')
    ax1.add_artist(leg2)
    ax2.legend(loc='lower left')
        
    if fixedTime=='total':
        plt.suptitle(f'ai={hard._sepa_init/PC:.2g}pc, '
                     f'rc={hard._rchar/PC:.2g}pc,'
                     f' nu_in={hard._gamma_inner}, nu_out={hard._gamma_outer}\n'
                     f'Mtot=({sam.mtot.min()/MSOL:.2g},{sam.mtot.max()/MSOL:.2g})Msun, '
                     f'q=({sam.mrat.min():.2g},{sam.mrat.max():.2g})')
    fig.subplots_adjust(wspace=0.3,top=0.85, right=0.95)
    

In [ ]:
def __compare_gwb(sams, dpops, gpf_flags=None, save=True, fname_extra='',
                  colors=None, lbl_extra=None, sam_colors=None,sam_lbls=None,
                  NLOUD=None, NREALS=None, TAU=None):

        
    LABEL_GW_FREQUENCY_YR = r"GW Frequency $[\mathrm{yr}^{-1}]$"
    LABEL_GW_FREQUENCY_HZ = r"GW Frequency $[\mathrm{Hz}]$"
    LABEL_GW_FREQUENCY_NHZ = r"GW Frequency $[\mathrm{nHz}]$"
    LABEL_SEPARATION_PC = r"Binary Separation $[\mathrm{pc}]$"
    LABEL_CHARACTERISTIC_STRAIN = r"GW Characteristic Strain"
    LABEL_HARDENING_TIME = r"Hardening Time $[\mathrm{Gyr}]$"
    LABEL_CLC0 = r"$C_\ell / C_0$"

    fig, ax = plot.figax(
        xlabel=LABEL_GW_FREQUENCY_YR,
        ylabel=LABEL_CHARACTERISTIC_STRAIN,
        ylim=(2.0e-17,2.0e-14)
    )

    frac = 0.50
    
    print(f"{len(sams)=}")
    if len(sams) > 0:
        sam_freqs = sams[0].PARS['freqs']
        xx = sam_freqs * YR
        if sam_colors is None: 
            sam_colors = ['k']*len(sams)
        for k,s in enumerate(sams):
            #print(f"{k=} {s.model_type=}")
            if s.gwb_sam is None:
                print("no plot generated for sam with invalid params:")
                #print([v for v in s.PARS.values()])
                continue
                
            if s.model_type == 'old':
                lw=1.5
                col='k'
                ls='-'
            elif s.model_type == 'old_rc100':
                lw=2.5
                col='k'
                ls=':' if len(dpops)==0 else '-'
            elif s.model_type == 'ph15':
                lw=2.5 if len(dpops)==0 else 2
                col='darkred' if len(dpops)==0 else 'k'
                ls='-.'
            elif s.model_type == 'old_2s':
                lw=2
                col='k'
                ls='--'
            elif s.model_type == 'astr_rc100':
                lw=2.5 if len(dpops)==0 else 3
                col='r' if len(dpops)==0 else 'k'
                ls='-'
            elif s.model_type == 'astr_nuo0':
                lw=2
                col='darkgray'
                ls='-.'
            elif s.model_type == 'astr':
                lw=2
                col='darkgray'
                ls='-' if len(dpops)==0 else ':'
            else:
                lw=2
                col='darkgray'
                ls='-'
                #raise ValueError(f"havent defined plot style for {s.model_type=}")

            #lbl = s.model_type
            lbl = ''
            if lbl_extra is not None:
                if len(lbl_extra) == len(sams):
                    lbl += lbl_extra[k]
            #gpf_lbls = [' (GMR)', ' (GPF)']
            #if len(gpf_flags) == len(sams):
            #    lbl += gpf_lbls[gpf_flags[k]] 
            #    ls = '--' if gpf_flags[k] else '-'

            if colors is not None and len(colors)>=len(sams):
                thiscol = colors[k]
                lw = (len(sams)-k)*0.5 + 0.5
            else:
                thiscol = col
            
            s_hctot = calc_hctot(s.gwb_sam)
            print(f"{s_hctot.min()=:.4g} {s_hctot.max()=:.4g}")
            #__draw_gwb(ax, xx, s_hctot, nsamp=0, color=sam_colors[k], label=lbl,
            __draw_gwb(ax, xx, s_hctot, nsamp=0, color=thiscol, label=lbl,
                       lw=lw, ls=ls, alpha=0.05, fracs=[frac])

                
    plt.legend(loc='lower left',fontsize=9) 
    ## new hardening model type = 0:
    if 'new_hard' in fname_extra or 'newhard' in fname_extra:
        suptitl = (f"{fname_extra}\n"
                     +f"tout={s.PARS['hard_outer_time']:.2g}, "
                     +f"rch9={s.PARS['hard_rchar_9']:.2g}pc, "
                     +f"alphch={s.PARS['hard_alpha_char']:.2g}, "       
                     +f"rgw9={s.PARS['hard_r_gw_crit_9']:.2g}{s.PARS['hard_gw_crit_units']}, "
                     +f"alphgw={s.PARS['hard_alpha_gw_crit']:.2g}, ")
        if s.hard._inner_model_type == 0:
            suptitl += f"nuin={s.PARS['hard_nu_inner']:.2g}"
        elif s.hard._inner_model_type == 1:
            suptitl += f"dadt={s.PARS['hard_dadt_rchar']:.2g}"
        else:
            raise NotImplementedError
        plt.suptitle(suptitl)
    else:
        plt.suptitle(fname_extra)
        
    if save:
        #plt.savefig('gwb_compare_ill_tng100_main.png',dpi=300)
        if fname_extra != '':
            fname = f'gwb_compare_nloud{NLOUD}_nreals{NREALS}_tau{TAU}_{fname_extra}.png'
        else:
            fname = f'gwb_compare_nloud{NLOUD}_nreals{NREALS}_tau{TAU}.png'            
        plt.savefig(fname, dpi=300)

    return

In [ ]:
from compare_sams import load_sams_from_pkl

In [ ]:
def calc_sam_dadt_from_pkl(sam, nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, 
                           num_steps=num_steps, verbose=False):

    # () start from the hardening model's initial separation
    rmax = sam.hard._rchar
    # (M,) end at the ISCO
    rmin = utils.rad_isco(s.sam.mtot)
    # Choose steps for each binary, log-spaced between rmin and rmax
    extr = np.log10([rmax * np.ones_like(rmin), rmin])
    radii = np.linspace(0.0, 1.0, num_steps)[np.newaxis, :]
    # (M, X)
    radii = extr[0][:, np.newaxis] + (extr[1] - extr[0])[:, np.newaxis] * radii
    radii = 10.0 ** radii
    # (M, Q, Z, X)
    mt, mr, rz, rads = np.broadcast_arrays(
        sam.sam.mtot[:, np.newaxis, np.newaxis, np.newaxis],
        sam.sam.mrat[np.newaxis, :, np.newaxis, np.newaxis],
        sam.sam.redz[np.newaxis, np.newaxis, :, np.newaxis],
        radii[:, np.newaxis, np.newaxis, :]
    )

    if verbose:
        print(sam.model_type)
        print(f"{sam.PARS['hard_outer_time']=}")
        print(sam.sam.mtot.shape, sam.sam.mrat.shape, sam.sam.redz.shape)
        print(sam.gwb_sam[0].shape,sam.gwb_sam[1].shape,sam.gwb_sam[2].shape,sam.gwb_sam[3].shape)
        print(f"{s.hard._rchar=} {sam.hard._rchar/PC=}")
        print(rmin.shape, rmin.min(), rmin.max())
        print(np.log10(sam.hard._rchar), np.log10(rmin), num_steps)

    dadt, agw_crit, rz_char, rz_final = s.hard.dadt(mt, mr, rz, rads)

    gpf_flags = [0]*len(sams_newhard_typ0_toutvar) 

    return sam.sam, sam.hard, rads, dadt, agw_crit, rz_char, rz_final, sam.gwb_sam


In [ ]:
zmin=1.0e-5
zmax=10
fomin=freqs_edges.min()
fomax=freqs_edges.max()
Mtot=1e9*MSOL
for z in [zmin,zmax]:
    for fo in [fomin,fomax]:
        for M in [1e4*MSOL,1e8*MSOL, 1e12*MSOL]:
            fr=utils.frst_from_fobs(fo,z)
            sp=sepa_emit(M,fr)
            print(f"{fo=:.4g}, {z=}, {M/MSOL=:.4g}, frst={fr:.4g}, "
                  f"sep={sp/PC:.4g}pc ={sp/(NWTG*M/SPLC**2):.4g}rg")
#print(f"{fomax=:.6g}, z={zmax}, frst={utils.frst_from_fobs(fomax,zmax):.6g}, sep={sepa_emit(Mtot,utils.frst_from_fobs(fomax,zmax))/PC:.6g}")
#print(f"{fomin=:.6g}, z={zmin}, frst={utils.frst_from_fobs(fomin,zmin):.6g}, sep={sepa_emit(Mtot,utils.frst_from_fobs(fomin,zmin))/PC:.6g}")
#print(f"{fomax=:.6g}, z={zmin}, frst={utils.frst_from_fobs(fomax,zmin):.6g}, sep={sepa_emit(Mtot,utils.frst_from_fobs(fomax,zmin))/PC:.6g}")


#fobs_min = utils.fobs_from_frst(freqs_edges.min(), sam.redz.max())
#fobs_max = utils.fobs_from_frst(freqs_edges.max(), sam.redz.min())
print(f"{1/(fomin*YR)=:.4g} yr")

In [ ]:
def tau_gw(a0,eta_norm,M):
    #print(f"{a0/PC=} {eta_norm=} {M=}")
    return 5 * a0**4 * SPLC**5 / (64 * NWTG**3 * (eta_norm) * M**3)

In [ ]:
tau_gw(1.0e16,1.0,1.0e8*MSOL)/YR

In [ ]:
for _M in np.logspace(4,12,9)*MSOL:
    print("")
    for q in (0.001,1.0):
        eta_norm = 4*q/(1+q)**2
        Tobs = 20*YR
        fobs = 1/Tobs
        apta = sepa_emit(_M, fobs)
        print(f"{_M/MSOL=:.2g} {q=:.2g} {apta/PC=:.4g}, tau/yr={tau_gw(apta,eta_norm,_M)/YR:.4g}")


In [ ]:
(5*SPLC**5)/(64*NWTG**(5/3)*np.pi**(8/3))*(1.0e9*MSOL)**(-5/3)*(YR)**(8/3)*(1/100)**(-8/3)/(4*0.001/(1.001)**2) / YR #*(1.0/(100*YR))**(-8/3)/GYR

In [ ]:
(13.7e9/8156)**(-3/5) * (1/100)**(-8/5) /(4*0.1/(1.1)**2)**(3/5)

In [ ]:
(1/15)**(-8/5)

In [ ]:
(13.7e9/8156)**(-3/5) * (1/15)**(-8/5)

In [ ]:
(4*0.001/(1.001)**2)**(3/5)

In [ ]:
sepa_emit(1e9*MSOL, 1/(20*YR))/PC 

In [ ]:
sepa_emit(1e9*MSOL, 1/(20*YR))/PC / .015

In [ ]:
1/ (sepa_emit(1e9*MSOL, 1/(20*YR))/PC)

In [ ]:
sepa_emit(1e4*MSOL, 1/(20*YR))/PC

In [ ]:
.02/.00122

In [ ]:
sepa_emit(1e9*MSOL, 1/(100*YR))/PC 

In [ ]:
sepa_emit(1e4*MSOL, 1/(100*YR))/PC 

In [ ]:
1/.1658

In [ ]:
.02/.00357

In [ ]:
# aGW
#10**2.5 * (NWTG * 1e9*MSOL / SPLC**2) * ((10**10.38)/1e9)**.75/PC
10**2.5 * (NWTG * 1e9*MSOL / SPLC**2) * ((10**10.04)/1e9)/PC

In [ ]:
# aPTA
sepa_emit((10**10.38)*MSOL, 1/(20*YR))/PC 

In [ ]:
def aPTA_eq_aGW(M, agw9rg, Tobs_yr, alphagw):
    sepa_emit(M*MSOL, 1/(Tobs_yr*YR))/PC - agw9rg * (NWTG * 1e9*MSOL / SPLC**2) * (M/1e9)**(alphagw+1)/PC

meq = scipy.optimize.fsolve(aPTA_eq_aGW, 1e10, args=(10**2.5,20,0))
print(meq, np.log10(meq))

In [ ]:
10**10.38/1e10

In [ ]:
NLOUD = 5
NREALS = 10
NFREQS = 40
num_steps = 100
sams_newhard_typ0_toutvar = load_sams_from_pkl(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, 
                                               gpf_flag=False, tau=None, data_dir=_SIM_MERGER_PATH, 
                                               fname_type='new_hardening_type0_toutvar')

dadt_sams_newhard_typ0_toutvar = []
for s in sams_newhard_typ0_toutvar: 
    dadt_sams_newhard_typ0_toutvar = ( dadt_sams_newhard_typ0_toutvar + 
                                       [(calc_sam_dadt_from_pkl(s, nloud=NLOUD, nreals=NREALS, 
                                                                nfreqs=NFREQS, num_steps=num_steps))] )



In [ ]:
calc_and_plot_dadt(dadt_sams_newhard_typ0_toutvar, fixedTime='outer', extra_panels=False)
calc_and_plot_dadt(dadt_sams_newhard_typ0_toutvar, fixedTime='outer', distance_units='rg', extra_panels=False)

In [ ]:
def calc_cumulative_thard(_sam_data, astart, astop, fixedTime='total'):

    if fixedTime not in ['total','outer']:
        raise ValueError(f"keyword `fixedTime` must be 'total' or 'outer'.")

    if fixedTime=='total':
            if len(_sam_data) == 4:
                _sam, _hard, _rads, _dadt = _sam_data
            elif len(_sam_data) == 5:
                _sam, _hard, _rads, _dadt, _gwb = _sam_data
            else:
                raise ValueError(f"sam_data has unexpected length {len(_sam_data)}. must be 4 or 5 for `fixedTime`='total'.")
    else:
        if len(_sam_data) == 7:
            _sam, _hard, _rads, _dadt, _agw_crit, _rz_char, _rz_final = _sam_data
        elif len(_sam_data) == 8:
            _sam, _hard, _rads, _dadt, _agw_crit, _rz_char, _rz_final, _gwb = _sam_data
        else:
            raise ValueError(f"sam_data has unexpected length {len(_sam_data)}. must be 7 or 8 for `fixedTime`='outer'")


    idx_avals = np.where((_rads[0,0,0,:]<=astart)&(_rads[0,0,0,:]>=astop))[0]
    
    _tevo = -utils.trapz_loglog(-1.0 / _dadt[:,:,0,idx_avals], _rads[:,:,0,idx_avals], 
                                axis=2, cumsum=True)
    return _tevo


def calc_aGW_for_Fixed_Time_2PL(_hard, _sam):
    """Calculate (circular) binary separation of transition from 'inner' power-law hardening to GW regime"""

    
    _mt, _mr = np.broadcast_arrays(
        _sam.mtot[:, np.newaxis],
        _sam.mrat[np.newaxis, :]
    )
    
    _m1, _m2 = utils.m1m2_from_mtmr(_mt, _mr)
    #print(f"{mt/MSOL=},{mr=}")
    #print(f"{m1/MSOL=},{m2/MSOL=}")
    dadt_gw_const = - (64/5.0) * NWTG**3 / SPLC**5 * _m1 * _m2 * _mt
    #print(f"{dadt_gw_const.shape=} {dadt_gw_const=}")
    dadt_innerPL_const = - _hard._norm * _hard._rchar**(_hard._gamma_inner-1)
    #print(f"{dadt_innerPL_const.shape=} {dadt_innerPL_const=}")
    
    aGW = ( dadt_gw_const / dadt_innerPL_const ) ** (1.0/(4.0-_hard._gamma_inner))
    #print(f"{aGW=}")
    
    return aGW
    
#def calc_aGW_for_Fixed_Time_2PL(norm, nu_inner, rchar, mtot, mrat):
#    """Calculate (circular) binary separation of transition from 'inner' power-law hardening to GW regime"""
#
#    
#    mt, mr = np.broadcast_arrays(
#        mtot[:, np.newaxis],
#        mrat[np.newaxis, :]
#    )
#    
#    m1, m2 = utils.m1m2_from_mtmr(mt, mr)
#    #print(f"{mt/MSOL=},{mr=}")
#    #print(f"{m1/MSOL=},{m2/MSOL=}")
#    dadt_gw_const = - (64/5.0) * NWTG**3 / SPLC**5 * m1 * m2 * mt
#    #print(f"{dadt_gw_const.shape=} {dadt_gw_const=}")
#    dadt_innerPL_const = - norm * rchar**(nu_inner-1)
#    #print(f"{dadt_innerPL_const.shape=} {dadt_innerPL_const=}")
#    
#    aGW = ( dadt_gw_const / dadt_innerPL_const ) ** (1.0/(4.0-nu_inner))
#    #print(f"{aGW=}")
#    
#    return aGW

def calc_model1_pars(sam, hard):

    if hard._inner_model_type != 1:
        raise ValueError(f"Function requires hard._inner_model_type=1, not {hard._inner_model_type}.")

    mt, mr, = np.broadcast_arrays(
        sam.mtot[:, np.newaxis],
        sam.mrat[np.newaxis, :]
    )
    m9 = mt / (1.0e9*MSOL) 
    if hard._gw_crit_units == 'rg':
        r9 = hard._r_gw_crit_9 * utils.gravitational_radius(1.0e9*MSOL) # convert to cm
    else:
        r9 = hard._r_gw_crit_9 * PC # convert to cm
    rgw_crit = r9 * m9**(hard._alpha_gw_crit+1)
    m1, m2 = utils.m1m2_from_mtmr(mt, mr)
    dadt_gw_crit = utils.gw_hardening_rate_dadt(m1, m2, rgw_crit)
    eta_norm = mr / np.square(1 + mr) * 4    
    dadt_phenom_rchar = hard._dadt_rchar * eta_norm
    tmp = np.log10(-dadt_phenom_rchar) - np.log10(-dadt_gw_crit)
    tmp /= (np.log10(hard._rchar) - np.log10(rgw_crit))
    #print(f"{m9.shape=} {m1.shape=} {rgw_crit.shape=} {m2.shape=} {dadt_gw_crit.shape=} {tmp.shape=}")
    nu_in = 1.0 - tmp
    #nu_in = 1 - ( np.log10(-dadt_gw_crit) - np.log10(-hard.dadt_rchar*1.0) ) / ( np.log10(rgw_crit) - np.log10(self._rchar) )

    return nu_in, rgw_crit, dadt_gw_crit, utils.gravitational_radius(mt), r9

def plot_param_space(sam_data):
    
    fig = plt.figure(figsize=(10,8))
    ax1 = fig.add_subplot(231)
    ax1.set_xlabel('log10(rGW_crit) [log10(pc)]')
    ax1.set_ylabel('log10(r_char) [log10(pc)]')
    ax2 = fig.add_subplot(232)
    ax2.set_xlabel('log10(rGW_crit/rISCO)')
    ax2.set_ylabel('nu_inner')
    ax2.set_ylim(-20,20)
    ax3 = fig.add_subplot(233)
    ax3.set_xlabel('alpha')
    ax3.set_ylabel('log10(r_gw_crit_9/Rg)')
    ax4 = fig.add_subplot(234)
    ax4.set_xlabel('log10(rchar/rGW_crit)')
    ax4.set_ylabel('nu_inner')
    #ax4.set_ylim(-10,10)
    ax5 = fig.add_subplot(235)
    ax5.set_xlabel('log10(r_char) [log10(pc)]')
    ax5.set_ylabel('log10(-dadt(rchar)) [log10(cm/s)]')
    ax6 = fig.add_subplot(236)
    ax6.set_xlabel('log10(rchar/rGW_crit)')
    ax6.set_ylabel('log10(-dadt(rchar)) [log10(cm/s)]')
    
    min_lg_rgw_pc = 1.0e20
    max_lg_rgw_pc = 1.0e-20
    min_lg_rch_pc = 1.0e20
    max_lg_rch_pc = 1.0e-20
    cmap = None
    x_alpha_valid = np.arange(-1.5,0.5,0.1)
    nu_inner_max = 10
    
    for s in sam_data:
        sam, hard, rads, dadt, agw_crit, rz_char, rz_final = s
        nu_in, rgw_crit, dadt_gw_crit, Rg, r9 = calc_model1_pars(sam, hard)
        #print(f"{hard._gw_crit_units=} {nu_in.shape=},{rgw_crit.shape=},{dadt_gw_crit.shape=},{Rg.shape=},{r9=}")
        risco = utils.rad_isco(sam.mtot)
        lg_risco_pc = np.log10(risco/PC)
        lg_rgw_pc = np.log10(rgw_crit/PC)        
        lg_rchar_pc = np.log10(hard._rchar/PC) 
        lg_negdadt = np.log10(-hard._dadt_rchar)        
        if cmap is None:
            cmap = plot._get_cmap('viridis')
            colors = cmap(np.linspace(0, 1, sam.mtot.size+1))
            mrksz = np.linspace(1,4,sam.mrat.size+1)
            print(f"{mrksz=}")
            for i in range(sam.mtot.size):
                ax1.plot([],[],color=colors[i],label=f"M={sam.mtot[i]/MSOL:.3g}",
                        marker='o',lw=0,ms=mrksz[-1])


        if lg_rgw_pc.max() > max_lg_rgw_pc:
            max_lg_rgw_pc = lg_rgw_pc.max()
        if lg_rgw_pc.min() < min_lg_rgw_pc:
            min_lg_rgw_pc = lg_rgw_pc.min()
        if lg_rchar_pc > max_lg_rch_pc:
            max_lg_rch_pc = lg_rchar_pc
        if lg_rchar_pc < min_lg_rch_pc:
            min_lg_rch_pc = lg_rchar_pc
            
        #if np.any(rgw_crit!=agw_crit[:,:,0,0]):
        #    print(f"{rgw_crit=}, {agw_crit=}")
        #    raise ValueError(f"mismatch in agw_crit from hard class & rgw_crit from calc_model1_pars.")
        for i in range(sam.mtot.size):
            for j in range(sam.mrat.size):

                    
                #lg(agw9)  = lg(risco) - (alpha+1)*lg(m9) # critical line for valid models
                #print(f"{Rg.shape=}")
                y_r9gtrisco_pc_valid = 10.0**(lg_risco_pc[i] - (x_alpha_valid+1)*np.log10(sam.mtot[i]/1.0e9/MSOL))
                y_r9gtrisco_risco_valid = y_r9gtrisco_pc_valid *PC / risco[i]
                #print(f"{np.log10(y_r9_Rg_valid)=}")
                y_r9ltrch_pc_valid = 10**(np.log10(0.5*hard._rchar) - (x_alpha_valid+1)*np.log10(sam.mtot[i]/1.0e9/MSOL))
                y_r9ltrch_risco_valid = y_r9ltrch_pc_valid / risco[i]
                #print(f"{y_r9ltrch_risco_valid.shape=}, {y_r9gtrisco_risco_valid.shape=}, {risco.shape=}")
                lgrdiff = np.log10(hard._rchar)-np.log10(rgw_crit[i,j])
                lgdadtdiff = np.log10(-hard._dadt_rchar) - np.log10(-dadt_gw_crit[i,j])    
                
                #(1-vmax)*lgrdiff + lgadotgw is a min or max depending if adotrch > or < dadt_gw_crit
                min_lgdadtrch_nuabsmax = (1-nu_inner_max)*lgrdiff + np.log10(-dadt_gw_crit[i,j])
                max_lgdadtrch_nuabsmax = (1+nu_inner_max)*lgrdiff + np.log10(-dadt_gw_crit[i,j])
                #print(f"{rgw_crit[i,j]/PC=} {hard._rchar/PC=}")
                #print(f"{lgrdiff=} {nu_inner_max=} {min_lgdadtrch_nuabsmax=} {max_lgdadtrch_nuabsmax=} {-dadt_gw_crit[i,j]=}")
                #print(f"{np.exp(lgadotrch_vmax)=} {np.exp(lgadotrch_vmin)=}")

                if j==0:
                    #tmp = 1 - ( np.log10(-dadt_gw_crit[i,j])-np.log10(-hard._dadt_rchar) ) / (np.log10(rgw_crit[i,j])-np.log10(hard._rchar))
                    # except for a few already-invalid cases with rgw>rch, increasing lg(-dadt(rch)) lowers nu_in 
                    # and decreasing lg(-dadt(rch)) increases nu_in (in both cases, the number itself, not its absolute value
                    # if nu_in is negative, and abs(nu_in)>vmax, we need to decrease dadt. 
                    # if nu_in is positive and >vmax, we need to increase dadt.
                    #use nu_in=0 case as divider. 
                    #if dadtrch>dadtgw, nu_in<0. need to make sure dadtrch<dadtrch_vmax.
                    #if dadtrch<dadtgw, nu_in>0. need to make sure dadtrch>dadtrch_vmax.
                    if nu_in[i,j] < -7:
                        print(f"nu_in={nu_in[i,j]:.3g}, M={sam.mtot[i]/MSOL:.3g},q={sam.mrat[j]},rch={hard._rchar/PC:.3g}, "
                              f"rgw={rgw_crit[i,j]/PC:.3g},alph={hard._alpha_gw_crit}, \n"
                              f"dadtrc={hard._dadt_rchar:.3g}, dadt_gw_crit={dadt_gw_crit[i,j]:.3g},{min_lgdadtrch_nuabsmax=:.3g}, {max_lgdadtrch_nuabsmax=:.3g}, "
                              f"lgdadtdiff={lgdadtdiff:.3g}, {lgdadtdiff/lgrdiff=:.3g}")

                #if np.abs(nu_in[i,j])>7 and np.abs(nu_in[i,j])<10:
                #    if (lg_rgw_pc[i,j] >= np.log10(0.5*hard._rchar/PC)) or (rgw_crit[i,j] <= risco[i]):
                #        pass
                #    else:
                #        print(f"WARNING: {nu_in[i,j]=} for M={sam.mtot[i]/MSOL:.3g},q={sam.mrat[j]},rch={hard._rchar/PC:.3g}, "
                #              f"rgw={rgw_crit[i,j]/PC:.3g},alph={hard._alpha_gw_crit}, \n"
                #              f"dadtrc={hard._dadt_rchar:.3g}, dadt_gw_crit={dadt_gw_crit[i,j]:.3g}, {lgadotrch_vmax=:.3g}, "
                #              f"lgadotdiff={lgadotdiff:.3g}, {lgadotdiff/lgrdiff=:.3g}")                        

                                        
                if rgw_crit[i,j] <= risco[i]:
                    ax1.plot(lg_rgw_pc[i,j], lg_rchar_pc,
                             marker='o',color='r',ms=mrksz[j]+1.5, lw=0)
                    ax2.plot(np.log10(rgw_crit[i,j]/risco[i]), nu_in[i,j],
                             marker='o',color='r',ms=mrksz[j]+1.5, lw=0)
                    ax3.plot(hard._alpha_gw_crit+i*0.05, np.log10(rgw_crit[i,j]/risco[i]), 
                             marker='o',color='r',ms=mrksz[j]+1.5, lw=0)
                    ax4.plot(np.log10(hard._rchar/rgw_crit[i,j]), nu_in[i,j],
                             marker='o',color='r',ms=mrksz[j]+1.5, lw=0)
                    ax5.plot(np.log10(hard._rchar/PC), np.log10(-hard._dadt_rchar),
                             marker='o',color='r',ms=mrksz[j]+1.5, lw=0)
                    ax6.plot(np.log10(hard._rchar/rgw_crit[i,j]), np.log10(-hard._dadt_rchar),
                             marker='o',color='r',ms=mrksz[j]+1.5, lw=0)
                    
                    #print(f"WARNING: rgw<risco for M={sam.mtot[i]/MSOL:.3g},q={sam.mrat[j]},rch={hard._rchar/PC:.3g}, "
                    #      f"rgw={rgw_crit[i,j]/PC:.3g},alph={hard._alpha_gw_crit},dadtrc={hard._dadt_rchar:.3g}.")
                #elif lg_rgw_pc[i,j] >= np.log10(0.5*hard._rchar/PC):
                elif lg_rgw_pc[i,j] >= np.log10(hard._rchar/PC):
                    ax1.plot(lg_rgw_pc[i,j], lg_rchar_pc,
                             marker='o',color='m',ms=mrksz[j]+1.5, lw=0)
                    ax2.plot(np.log10(rgw_crit[i,j]/risco[i]), nu_in[i,j],
                             marker='o',color='m',ms=mrksz[j]+1.5, lw=0)
                    ax3.plot(hard._alpha_gw_crit+i*0.05, np.log10(rgw_crit[i,j]/risco[i]), 
                             marker='o',color='m',ms=mrksz[j]+1.5, lw=0)
                    ax4.plot(np.log10(hard._rchar/rgw_crit[i,j]), nu_in[i,j],
                             marker='o',color='m',ms=mrksz[j]+1.5, lw=0)
                    ax5.plot(np.log10(hard._rchar/PC), np.log10(-hard._dadt_rchar),
                             marker='o',color='m',ms=mrksz[j]+1.5, lw=0)
                    ax6.plot(np.log10(hard._rchar/rgw_crit[i,j]), np.log10(-hard._dadt_rchar),
                             marker='o',color='m',ms=mrksz[j]+1.5, lw=0)
                    #print(f"WARNING: rgw>rchar for M={sam.mtot[i]/MSOL:.3g},q={sam.mrat[j]},rch={hard._rchar/PC:.3g}, "
                    #      f"rgw={rgw_crit[i,j]/PC:.3g},alph={hard._alpha_gw_crit},dadtrc={hard._dadt_rchar:.3g}.")

                    #if dadtrch>dadtgw, nu_in<0. need to make sure dadtrch<dadtrch_vmin.
                    #if dadtrch<dadtgw, nu_in>0. need to make sure dadtrch>dadtrch_vmax.

                #elif np.log10(-hard._dadt_rchar)>np.log10(-dadt_gw_crit[i,j]) and np.log10(-hard._dadt_rchar)>lgadotrch_vmax:
                elif np.log10(-hard._dadt_rchar)<min_lgdadtrch_nuabsmax or np.log10(-hard._dadt_rchar)>max_lgdadtrch_nuabsmax:
                    #np.log10(-hard._dadt_rchar) < np.log10(-dadt_gw_crit[i,j]) + (1-nu_inner_max)*lgrdiff:
                    #(1-vmax)*(lgrch-lgrgw) + lgadotgw >  lgadotrch
                    ax1.plot(lg_rgw_pc[i,j], lg_rchar_pc,
                             marker='o',color='y',ms=mrksz[j]+1.5, lw=0)
                    ax2.plot(np.log10(rgw_crit[i,j]/risco[i]), nu_in[i,j],
                             marker='o',color='y',ms=mrksz[j]+1.5, lw=0)
                    ax3.plot(hard._alpha_gw_crit+i*0.05, np.log10(rgw_crit[i,j]/risco[i]), 
                             marker='o',color='y',ms=mrksz[j]+1.5, lw=0)
                    ax4.plot(np.log10(hard._rchar/rgw_crit[i,j]), nu_in[i,j],
                             marker='o',color='y',ms=mrksz[j]+1.5, lw=0)
                    ax5.plot(np.log10(hard._rchar/PC), np.log10(-hard._dadt_rchar),
                             marker='o',color='y',ms=mrksz[j]+1.5, lw=0)
                    ax6.plot(np.log10(hard._rchar/rgw_crit[i,j]), np.log10(-hard._dadt_rchar),
                             marker='o',color='y',ms=mrksz[j]+1.5, lw=0)
                    
                    print(f"WARNING: {nu_in[i,j]=:.3g} for M={sam.mtot[i]/MSOL:.3g},q={sam.mrat[j]},rch={hard._rchar/PC:.3g}, "
                          f"rgw={rgw_crit[i,j]/PC:.3g},lgrdiff={lgrdiff:.3g}, alph={hard._alpha_gw_crit},\n"
                          f"dadtrc={hard._dadt_rchar:.3g}, dadt_gw_crit={dadt_gw_crit[i,j]:.3g}, "
                          f"lgdadtdiff={lgdadtdiff:.3g}, {lgdadtdiff/lgrdiff=:.3g}")
                #elif np.log10(-hard._dadt_rchar)<lgadotrch_vmin:
                ##elif np.log10(-hard._dadt_rchar)<np.log10(-dadt_gw_crit[i,j]) and np.log10(-hard._dadt_rchar)<lgadotrch_vmin:
                #    ax1.plot(lg_rgw_pc[i,j], lg_rchar_pc,
                #             marker='o',color='orange',ms=mrksz[j]+4, lw=0)
                #    ax2.plot(np.log10(rgw_crit[i,j]/risco[i]), nu_in[i,j],
                #             marker='o',color='orange',ms=mrksz[j]+4, lw=0)
                #    ax3.plot(hard._alpha_gw_crit+i*0.05, np.log10(rgw_crit[i,j]/risco[i]), 
                #             marker='o',color='orange',ms=mrksz[j]+4, lw=0)
                #    ax4.plot(np.log10(hard._rchar/rgw_crit[i,j]), nu_in[i,j],
                #             marker='o',color='orange',ms=mrksz[j]+4, lw=0)
                #    print(f"***WARNING: {nu_in[i,j]=:.3g} for M={sam.mtot[i]/MSOL:.3g},q={sam.mrat[j]},rch={hard._rchar/PC:.3g}, "
                #          f"rgw={rgw_crit[i,j]/PC:.3g},lgrdiff={lgrdiff:.3g}, alph={hard._alpha_gw_crit},\n"
                #          f"dadtrc={hard._dadt_rchar:.3g}, dadt_gw_crit={dadt_gw_crit[i,j]:.3g}, "
                #          f"lgadotdiff={lgadotdiff:.3g}, {lgadotdiff/lgrdiff=:.3g}")
                
                ax1.plot(lg_rgw_pc[i,j], lg_rchar_pc,
                         marker='o',color=colors[i],ms=mrksz[j], lw=0)
                ax1.plot([lg_risco_pc[i],lg_risco_pc[i]],[3,5],color=colors[i],lw=2,ls='--')
                
                ax2.plot(np.log10(rgw_crit[i,j]/risco[i]), nu_in[i,j],
                         marker='o',color=colors[i],ms=mrksz[j], lw=0)
                #ax2.plot(lg_rchar_pc, lg_negdadt,
                #         marker='o',color=col,ms=mrksz[j], lw=0) # density plot with max dadt

                #ax3.plot(hard._alpha_gw_crit+i*0.05, np.log10(hard._r_gw_crit_9*Rg[i,j]/risco[i]), 
                #         marker='o',color=col,ms=mrksz[j], lw=0)
                ax3.plot(hard._alpha_gw_crit+i*0.05, np.log10(rgw_crit[i,j]/risco[i]), 
                         marker='o',color=colors[i],ms=mrksz[j], lw=0)
                #if i==0 and j==0:
                #    print(f"{x_alpha_valid=},{np.log10(y_r9_Rg_valid)=}")
                if j==0:
                    ax3.plot(x_alpha_valid, np.log10(y_r9gtrisco_risco_valid), color=colors[i],lw=1,alpha=0.5)
                    ax3.plot(x_alpha_valid, np.log10(y_r9ltrch_risco_valid), color=colors[i],ls=':',lw=0.5,alpha=0.5)
                
                #ax4.plot(lg_rgw_pc[i,j], nu_in[i,j],
                #         marker='o',color=col,ms=mrksz[j], lw=0)
                ax4.plot(np.log10(hard._rchar/rgw_crit[i,j]), nu_in[i,j],
                         marker='o',color=colors[i],ms=mrksz[j], lw=0)

                ax5.plot(np.log10(hard._rchar/PC), np.log10(-hard._dadt_rchar),
                         marker='o',color=colors[i],ms=mrksz[j], lw=0)
                ax6.plot(np.log10(hard._rchar/rgw_crit[i,j]), np.log10(-hard._dadt_rchar),
                         marker='o',color=colors[i],ms=mrksz[j], lw=0)
                
    ax1.legend()
    fig.subplots_adjust()
    
    print(f"{max_lg_rgw_pc=}, {min_lg_rgw_pc=}, {max_lg_rch_pc=}, {min_lg_rch_pc=} (in pc)")
    #ax1.plot(rgw_crit, hard._rchar)
    #ax2.plot(hard._rchar, hard._dadt_rchar) # density plot with max dadt
    #plt.plot(rchar, hard._dadt_rchar) # density plot with min dadt
    #plt.plot(rchar, hard._dadt_rchar) # density plot with nu_in


def plot_allowed_param_space_model1(N=10, risco_in_rg=6.0, gw_crit_units='rg',
                                    nu_inner_max=10.0, speed_limit=SPLC,
                                    alpha_char=-1, beta_gw=0):

    # alphar_char=-1 : no mass scaling of rchar 
    # alpha_char=0 : linear mass scaling of rchar

    lg_speedlimit = np.log10(speed_limit)   # scalar
    lg_risco_in_rg = np.log10(risco_in_rg)  # scalar

    mtot_arr = np.logspace(4,12,N) * MSOL  # (Nmtot,)
    mrat_arr = np.logspace(-3,0,N)   # (Nmrat,)
    rchar_9_arr = np.logspace(1,5,N) * PC   # (Nrch,)
    alpha_gw_arr = np.linspace(-1,0,N)   # (Nalphgw,)
    dadt_rchar_arr = -np.logspace(5,9,N)   # (Ndadtrch,)
    
    if gw_crit_units == 'rg':
        # hard._r_gw_crit_9 is in units of [Rg]
        #r9_rg = hard._r_gw_crit_9
        r9_rg_arr = np.logspace(1,5,N)   # (N,)
        r9_cm_arr = r9_rg_arr * utils.gravitational_radius(1.0e9*MSOL) # convert to cm   # (N,)
    elif gw_crit_units == 'pc':
        # hard._r_gw_crit_9 is in units of [pc]
        #r9_cm = hard._r_gw_crit_9 * PC # convert to cm
        r9_cm_arr = np.logspace(-6,3,N) * PC  # (Nr9,)
        r9_rg_arr = r9_cm_arr / utils.gravitational_radius(1.0e9*MSOL)   # (N,)
    else:
        raise ValueError(f"Invalid {gw_crit_units=}, must be 'rg' or 'pc'.")
    
    mt, mr, alph_gw, rch9, dadt_rch, r9cm = np.broadcast_arrays(
        mtot_arr[:, np.newaxis, np.newaxis, np.newaxis, np.newaxis, np.newaxis],
        mrat_arr[np.newaxis, :, np.newaxis, np.newaxis, np.newaxis, np.newaxis],
        alpha_gw_arr[np.newaxis, np.newaxis, :, np.newaxis, np.newaxis, np.newaxis],
        rchar_9_arr[np.newaxis, np.newaxis, np.newaxis, :, np.newaxis, np.newaxis],
        dadt_rchar_arr[np.newaxis, np.newaxis, np.newaxis, np.newaxis, :, np.newaxis],
        r9_cm_arr[np.newaxis, np.newaxis, np.newaxis, np.newaxis, np.newaxis, :]
    )

    m9 = mt / (1.0e9*MSOL)   # varies with mtot

    r9rg = r9cm / utils.gravitational_radius(1.0e9*MSOL)

    rchar = rch9 * m9**(alpha_char+1)
    
    m1, m2 = utils.m1m2_from_mtmr(mt, mr)   # varies with mtot & mrat

    eta_norm = mr / np.square(1 + mr) * 4   # varies with mrat

    lgr9rg_min_allowed = lg_risco_in_rg - alph_gw * np.log10(m9) # varies with mtot & alpha_gw
    lgr9rg_max_allowed = np.log10(rchar/utils.gravitational_radius(1.0e9*MSOL)) - (alph_gw+1) * np.log10(m9) # varies with rchar, mtot, alpha_gw
    
    # critical gw transition radius in cm
    rgw_crit = np.zeros_like(mt) 
    dadt_gw_crit = np.zeros_like(mt) 

    rgw_crit = r9cm * m9**(alph_gw+1)   # varies with mtot, mrat, r9, & alpha_gw
    dadt_gw_crit = utils.gw_hardening_rate_dadt(m1, m2, rgw_crit)   # varies with mtot, mrat, r9, & alpha_gw

    tmp = np.log10(np.abs(dadt_rch)) - np.log10(np.abs(dadt_gw_crit))  
    tmp /= (np.log10(rchar) - np.log10(rgw_crit))  
    nu_in = 1.0 - tmp   # varies with mtot, r9, & alpha_gw (mrat variation normalized out by construction)
    #print(f"{mtot=}")
    print(f"{dadt_gw_crit.shape=}, {rgw_crit.shape=}, {rgw_crit.shape=}") 
    #print(f"{rgw_crit.shape=}, {rgw_crit=}")
    #print(f"{rgw_crit.shape=}, {nu_in=}")

    lg_rchar_in_rg = np.log10(rchar/utils.gravitational_radius(mt))   # varies with rchar and mtot

    log.info(f"{rchar.shape=} {rchar.min()/PC=} {rchar.max()/PC=}")
    log.info(f"{np.min(rchar/utils.gravitational_radius(mt))} {np.max(rchar/utils.gravitational_radius(mt))}")
    log.info(f"{lg_rchar_in_rg.min()=} {lg_rchar_in_rg.max()=}")

    # nuin max criterion
    #(1-vmax)*lgrdiff + lgadotgw is a min or max depending if adotrch > or < dadt_gw_crit
    lgrdiff = np.log10(rchar) - np.log10(rgw_crit) # varies with rchar, mtot, mrat, r9, & alpha_gw
    # varies with rchar, mtot, mrat, r9, & alpha_gw:
    min_lgdadtrch_nuabsmax = -1.0*np.log10(eta_norm) + (1-nu_inner_max)*lgrdiff + np.log10(-dadt_gw_crit)
    max_lgdadtrch_nuabsmax = -1.0*np.log10(eta_norm) + (1+nu_inner_max)*lgrdiff + np.log10(-dadt_gw_crit)


    
    # make subplots 
    cmap = plot._get_cmap('PuBuGn')
    colors = cmap(np.linspace(0.3, 1, N+1))
    lstyles = ['-','--',':','-.']*4
    
    fig = plt.figure(figsize=(12,9))
    ax1 = fig.add_subplot(221)
    plt.xlabel('log10(Mtot/Msun)')
    plt.ylabel('log10(r9/Rg)')
    plt.title(r'Min/max allowed values of r9[Rg] vs Mtot (varied $\alpha_GW$)')
    ax1.plot([np.log10(mtot_arr[0]/MSOL),np.log10(mtot_arr[-1]/MSOL)],
             [lg_risco_in_rg,lg_risco_in_rg],'k:',lw=5)  

    ax2 = fig.add_subplot(222)
    plt.xlabel('log10(Mtot/Msun)')
    plt.ylabel('log10(rGWcrit/Rg)')
    plt.title(r'Min/max allowed values of rGWcrit[Rg] vs Mtot')    
    ax2.plot([np.log10(mtot_arr[0]/MSOL),np.log10(mtot_arr[-1]/MSOL)],
             [lg_risco_in_rg,lg_risco_in_rg],'k:',lw=5)  


    if mtot_arr.size > 6:
        mt_ix_to_plot = np.array([0,(mt.shape[0]-1)//4,(mt.shape[0]-1)//2,
                                  3*(mt.shape[0]-1)//4,mt.shape[0]-1])
    else:
        mt_ix_to_plot = np.arange(mt.shape[0])
    print(f"{mt_ix_to_plot=}")

    # rcritGW > rISCO criterion       
    lgr9rg_range, nuin_range, lgdadtrchar_range, lgdadtrchar_absrange = allowed_param_range(mt, mr, alpha_char, 
                                                                                            rch9, alph_gw, beta_gw, r9rg,
                                                                                           inner_model_type=1)
    _min_lgr9rg, _max_lgr9rg = lgr9rg_range[0], lgr9rg_range[1]
    _min_lgdadtrchar, _max_lgdadtrchar = lgdadtrchar_range[0], lgdadtrchar_range[1]
    _absmin_lgdadtrchar, _absmax_lgdadtrchar = lgdadtrchar_absrange[0], lgdadtrchar_absrange[1]
    
    for i,a in enumerate(alpha_gw_arr):  
        #lgr9rg_min_allowed = lg_risco_in_rg - a * np.log10(m9)
        ax1.plot(np.log10(mt[:,0,i,0,0,0]/MSOL), lgr9rg_min_allowed[:,0,i,0,0,0],
                 label=fr"$\alpha_GW=${a:.2g}",color=colors[i])
        for j,r in enumerate(rchar_9_arr):
            # rcritGW < rchar criterion
            if i==0:
                ax1.plot(np.log10(mt[:,0,i,j,0,0]/MSOL), lgr9rg_max_allowed[:,0,i,j,0,0],
                         color=colors[i],ls=lstyles[j]) #,label=fr"rch9={r/PC:.2g}pc")
                ax2.plot(np.log10(mt[:,0,i,j,0,0]/MSOL), lg_rchar_in_rg[:,0,i,j,0,0], 
                         color=colors[i],ls=lstyles[j],label=fr"rch9[pc]={r/PC:.2g}pc")
            else:
                ax1.plot(np.log10(mt[:,0,i,j,0,0]/MSOL), lgr9rg_max_allowed[:,0,i,j,0,0],color=colors[i],ls=lstyles[j])
                ax2.plot(np.log10(mt[:,0,i,j,0,0]/MSOL), lg_rchar_in_rg[:,0,i,j,0,0],color=colors[i],ls=lstyles[j])

            # testing compared to new function
            for ix_m in mt_ix_to_plot:
                for ix_q in mt_ix_to_plot:
                    ax1.plot(np.log10(mt[ix_m,ix_q,i,j,0,0]/MSOL), _min_lgr9rg[ix_m,ix_q,i,j,0,0], 
                             '^', color='g',ls=lstyles[j],alpha=0.2)
                    ax1.plot(np.log10(mt[ix_m,ix_q,i,j,0,0]/MSOL), _max_lgr9rg[ix_m,ix_q,i,j,0,0], 
                             'o', color='g',ls=lstyles[j],alpha=0.2)

    ax1.legend()                
    ax2.legend()


    figb = plt.figure(figsize=(12,9))

    plot_index = 231
    for ix_m in mt_ix_to_plot:
        ax = figb.add_subplot(plot_index)
        plt.xlabel('log10(r9/Rg)')
        plt.ylabel('log10(-dadt[cm/s])')
        #plt.ylim(-50,50)
        plt.title(rf'log10(Mtot/Msun) = {np.log10(mtot_arr[ix_m]/MSOL)}')
        
        for ix_q in mt_ix_to_plot:
            for i,a in enumerate(alpha_gw_arr):  
                for j,r in enumerate(rchar_9_arr):                
                    if np.any(min_lgdadtrch_nuabsmax[ix_m,ix_q,i,j,0,:]>max_lgdadtrch_nuabsmax[ix_m,ix_q,i,j,0,:]):
                        print(f"WARNING: invalid dadt for logM={np.log10(mtot_arr[ix_m]/MSOL)}, "
                              f"mrat={mrat_arr[ix_q]}, log(rchar9[pc])={r/PC}, {a=}")
                        #print(f"    {rgw_crit[ix_m,:,i]=}  {dadt_gw_crit[ix_m,:,i]=}")
                        #print(f"    {min_lgdadtrch_nuabsmax=} > {max_lgdadtrch_nuabsmax=}")
                        
                    if plot_index == 231 and i==0 and ix_q==mt_ix_to_plot[0]: 
                        ax.plot(np.log10(r9_rg_arr), max_lgdadtrch_nuabsmax[ix_m,ix_q,i,j,0,:],
                                lw=2,color=colors[i],ls=lstyles[j],label=fr"rchar9[pc]={r/PC:.2g}pc")
                        ax.plot(np.log10(r9_rg_arr), min_lgdadtrch_nuabsmax[ix_m,ix_q,i,j,0,:],lw=1,color=colors[i],ls=lstyles[j]) 
                        print(f"{r9_rg_arr.shape=} {min_lgdadtrch_nuabsmax.shape=} {max_lgdadtrch_nuabsmax.shape=}")
                        ax.fill_between(np.log10(r9_rg_arr), min_lgdadtrch_nuabsmax[ix_m,ix_q,i,j,0,:], 
                                        np.minimum(lg_speedlimit,max_lgdadtrch_nuabsmax[ix_m,ix_q,i,j,0,:]),
                                        alpha=0.05, color=colors[i],ls=lstyles[j])    
                    elif plot_index == 232 and j==0 and ix_q==mt_ix_to_plot[0]:
                        ax.plot(np.log10(r9_rg_arr), max_lgdadtrch_nuabsmax[ix_m,ix_q,i,j,0,:],
                                lw=2,color=colors[i],ls=lstyles[j],label=fr"$\alpha_GW=${a:.2g}")
                        ax.plot(np.log10(r9_rg_arr), min_lgdadtrch_nuabsmax[ix_m,ix_q,i,j,0,:],
                                lw=1,color=colors[i],ls=lstyles[j])                                                
                        ax.fill_between(np.log10(r9_rg_arr), min_lgdadtrch_nuabsmax[ix_m,ix_q,i,j,0,:], 
                                        np.minimum(lg_speedlimit,max_lgdadtrch_nuabsmax[ix_m,ix_q,i,j,0,:]),
                                        alpha=0.05, color=colors[i],ls=lstyles[j])
                    else:
                        ax.plot(np.log10(r9_rg_arr), max_lgdadtrch_nuabsmax[ix_m,ix_q,i,j,0,:],
                                lw=2,color=colors[i],ls=lstyles[j])
                        ax.plot(np.log10(r9_rg_arr), min_lgdadtrch_nuabsmax[ix_m,ix_q,i,j,0,:],
                                lw=1,color=colors[i],ls=lstyles[j])

                    #print(_min_lgdadtrchar[ix_m,ix_q,i,j,0,:])
                    #print(_max_lgdadtrchar[ix_m,ix_q,i,j,0,:])
                    #print(f"r9rg={r9rg[ix_m,ix_q,i,j,0,:]}")
                    ax.plot(np.log10(r9rg[ix_m,ix_q,i,j,0,:]), _max_lgdadtrchar[ix_m,ix_q,i,j,0,:],'o',
                            markersize=j+3,color=colors[i],alpha=0.2)
                    ax.plot(np.log10(r9rg[ix_m,ix_q,i,j,0,:]), _min_lgdadtrchar[ix_m,ix_q,i,j,0,:],'^',
                            markersize=j+3,color=colors[i],alpha=0.2)
                    #ax.plot(np.log10(r9_rg_arr),'^',markersize=j+1,color=colors[i],alpha=0.2)                                          
                    #print(f"mt={mtot_arr[ix_m]/MSOL} mr={mrat_arr[ix_q]} {a=:g} {rc/PC=:g} ")
                    #      #f"{_min_lgr9rg=} {_max_lgr9rg=} {_min_lgdadtrchar=} {_max_lgdadtrchar=}")
        ax.plot([np.log10(r9_rg_arr[0]),np.log10(r9_rg_arr[-1])],[lg_speedlimit,lg_speedlimit],'m--',lw=3)
        if plot_index == 231 or plot_index == 232:
            ax.legend() 
        plot_index += 1


In [ ]:
(5*SPLC**6 *10*6*utils.gravitational_radius(1e8*MSOL)**3 / (16*NWTG**3*4*(1e8*MSOL)**3)-1)**(1/2)

In [ ]:
N=13
print(np.logspace(4,12,N)) # mtot
print(np.logspace(-3,0,N)) # mrat
print(np.logspace(-0.5,2.5,N)) # rch9[log10(pc)]
print(np.linspace(-0.5,0,N))   # (Nalphgw,)
print(np.linspace(-1,2,N))   # (Nnuin,)
    


In [ ]:
def plot_allowed_param_space_model0(Nmt=9, Nmr=4, Nrch=4, Nalgw=5, Nnu=7, Nr9=100,
                                    risco_in_rg=6.0, gw_crit_units='rg',
                                    nu_inner_max=10.0, speed_limit=SPLC,
                                    alpha_char=-2/3, beta_gw=0):

    nskip = 1
    alphskip = 1
    rch9skip = 1
    nuskip = 1
    
    
    # alphar_char=-1 : no mass scaling of rchar 
    # alpha_char=0 : linear mass scaling of rchar

    lg_speedlimit = np.log10(speed_limit)   # scalar
    lg_risco_in_rg = np.log10(risco_in_rg)  # scalar

    mtot_arr = np.logspace(4,12,Nmt) * MSOL  # (Nmt,)
    mrat_arr = np.logspace(-3,0,Nmr)   # (Nmr,)
    rchar_9_arr = np.logspace(-1,2,Nrch) * PC   # (Nrch,)
    alpha_gw_arr = np.linspace(-1,0,Nalgw)   # (Nalgw,)
    #alpha_gw_arr = np.linspace(-0.5,0,Nalgw)   # (Nalgw,)
    nuin_arr = np.linspace(-1,2,Nnu)   # (Nnuin,)
    
    if gw_crit_units == 'rg':
        # hard._r_gw_crit_9 is in units of [Rg]
        #r9_rg = hard._r_gw_crit_9
        r9_rg_arr = np.logspace(1,10,Nr9)   # (Nr9,)
        r9_cm_arr = r9_rg_arr * utils.gravitational_radius(1.0e9*MSOL) # convert to cm   # (N,)
    elif gw_crit_units == 'pc':
        # hard._r_gw_crit_9 is in units of [pc]
        #r9_cm = hard._r_gw_crit_9 * PC # convert to cm
        r9_cm_arr = np.logspace(-6,5,Nr9) * PC  # (Nr9,)
        r9_rg_arr = r9_cm_arr / utils.gravitational_radius(1.0e9*MSOL)   # (N,)
    else:
        raise ValueError(f"Invalid {gw_crit_units=}, must be 'rg' or 'pc'.")
    
    mt, mr, alph_gw, rch9, nuin, r9cm = np.broadcast_arrays(
        mtot_arr[:, np.newaxis, np.newaxis, np.newaxis, np.newaxis, np.newaxis],
        mrat_arr[np.newaxis, :, np.newaxis, np.newaxis, np.newaxis, np.newaxis],
        alpha_gw_arr[np.newaxis, np.newaxis, :, np.newaxis, np.newaxis, np.newaxis],
        rchar_9_arr[np.newaxis, np.newaxis, np.newaxis, :, np.newaxis, np.newaxis],
        nuin_arr[np.newaxis, np.newaxis, np.newaxis, np.newaxis, :, np.newaxis],
        r9_cm_arr[np.newaxis, np.newaxis, np.newaxis, np.newaxis, np.newaxis, :]
    )

    m9 = mt / (1.0e9*MSOL)   # varies with mtot

    r9rg = r9cm / utils.gravitational_radius(1.0e9*MSOL)

    rchar = rch9 * m9**(alpha_char+1)
    
    m1, m2 = utils.m1m2_from_mtmr(mt, mr)   # varies with mtot & mrat

    eta_norm = mr / np.square(1 + mr) * 4   # varies with mrat

    # varies with mtot, mrat, alpha_gw, & beta_gw:
    lgr9rg_min_allowed = lg_risco_in_rg - alph_gw * np.log10(m9) - beta_gw * np.log10(eta_norm) 
    # varies with rchar, mtot, mrat, alpha_gw, & beta_gw
    lgr9rg_max_allowed = ( np.log10(rchar/utils.gravitational_radius(1.0e9*MSOL)) - 
                           (alph_gw+1) * np.log10(m9) - beta_gw * np.log10(eta_norm) ) 
    
    # critical gw transition radius in cm
    rgw_crit = np.zeros_like(mt) 
    dadt_gw_crit = np.zeros_like(mt) 

    rgw_crit = r9cm * m9**(alph_gw+1) * eta_norm**beta_gw  # varies with mtot, mrat, r9, alpha_gw
    dadt_gw_crit = utils.gw_hardening_rate_dadt(m1, m2, rgw_crit)   # varies with mtot, mrat, r9, & alpha_gw

    
    #tmp = np.log10(np.abs(dadt_rch)) - np.log10(np.abs(dadt_gw_crit))  
    #tmp /= (np.log10(rchar) - np.log10(rgw_crit))  
    #nu_in = 1.0 - tmp   # varies with mtot, r9, & alpha_gw (mrat variation normalized out by construction)
    #print(f"{mtot=}")
    #print(f"{dadt_gw_crit.shape=}, {rgw_crit.shape=}, {rgw_crit.shape=}") 
    #print(f"{rgw_crit.shape=}, {rgw_crit=}")
    #print(f"{rgw_crit.shape=}, {nu_in=}")

    lg_rchar_in_rg = np.log10(rchar/utils.gravitational_radius(mt))   # varies with rchar and mtot

    log.info(f"{rchar.shape=} {rchar.min()/PC=} {rchar.max()/PC=}")
    log.info(f"{np.min(rchar/utils.gravitational_radius(mt))} {np.max(rchar/utils.gravitational_radius(mt))}")
    log.info(f"{lg_rchar_in_rg.min()=} {lg_rchar_in_rg.max()=}")

    ## nuin max criterion
    ##(1-vmax)*lgrdiff + lgadotgw is a min or max depending if adotrch > or < dadt_gw_crit
    #lgrdiff = np.log10(rchar) - np.log10(rgw_crit) # varies with rchar, mtot, mrat, r9, & alpha_gw
    ## varies with rchar, mtot, mrat, r9, & alpha_gw:
    #min_lgdadtrch_nuabsmax = -1.0*np.log10(eta_norm) + (1-nu_inner_max)*lgrdiff + np.log10(-dadt_gw_crit)
    #max_lgdadtrch_nuabsmax = -1.0*np.log10(eta_norm) + (1+nu_inner_max)*lgrdiff + np.log10(-dadt_gw_crit)

    
    # make subplots 
    cmap = plot._get_cmap('PuBuGn')
    colors = cmap(np.linspace(0.3, 1, Nalgw+1))
    lstyles = ['-','--',':','-.']*4
    
    fig = plt.figure(figsize=(12,9))
    ax1 = fig.add_subplot(221)
    plt.xlabel('log10(Mtot/Msun)')
    plt.ylabel('log10(r9/Rg)')
    plt.title(r'Min/max allowed values of r9[Rg] vs Mtot (varied $\alpha_GW$)')
    ax1.plot([np.log10(mtot_arr[0]/MSOL),np.log10(mtot_arr[-1]/MSOL)],
             [lg_risco_in_rg,lg_risco_in_rg],'k:',lw=5)  

    ax2 = fig.add_subplot(222)
    plt.xlabel('log10(Mtot/Msun)')
    plt.ylabel('log10(rGWcrit/Rg)')
    plt.title(r'Min/max allowed values of rGWcrit[Rg] vs Mtot')    
    ax2.plot([np.log10(mtot_arr[0]/MSOL),np.log10(mtot_arr[-1]/MSOL)],
             [lg_risco_in_rg,lg_risco_in_rg],'k:',lw=5)  


    if mtot_arr.size > 6:
        mt_ix_to_plot = np.array([0,(mt.shape[0]-1)//4,(mt.shape[0]-1)//2,
                                  3*(mt.shape[0]-1)//4,mt.shape[0]-1])
    else:
        mt_ix_to_plot = np.arange(mt.shape[0])
    print(f"{mt_ix_to_plot=}")

    # rcritGW > rISCO criterion       
    lgr9rg_range, nuin_range, lgdadtrchar_range, lgdadtrchar_absrange  = allowed_param_range(mt, mr, alpha_char, 
                                                 rch9, alph_gw, beta_gw, r9rg,
                                                 inner_model_type=0, 
                                                 nu_inner_absmax=nu_inner_max)
    _min_lgr9rg, _max_lgr9rg = lgr9rg_range[0], lgr9rg_range[1]
    _min_nuin, _max_nuin = nuin_range[0], nuin_range[1]
    #print(f"CHECK: {_min_nuin.shape=}")
    #_min_lgdadtrchar, _max_lgdadtrchar = lgdadtrchar_range[0], lgdadtrchar_range[1]
    #_absmin_lgdadtrchar, _absmax_lgdadtrchar = lgdadtrchar_absrange[0], lgdadtrchar_absrange[1]
    #print(f"CHECK: {lgr9rg_min_allowed=} {_min_lgr9rg=}")
    #print(f"CHECK: {lgr9rg_max_allowed=} {_max_lgr9rg=}")

    fac = 5 * SPLC**5 * speed_limit / (16 * NWTG**3)
    max_rchar_over_rgw = ( fac * rgw_crit**3 / (eta_norm * mt**3) - 1 )**(1.0/(1-nuin))
    max_rchar_over_rgw[max_rchar_over_rgw<1.0] = 1.0
    max_rchar_over_rgw[nuin>=1] = np.nan
    #max_rchar_over_rgw[r9rg!=r9rg] = np.nan
    max_rch9_over_rgw9 = max_rchar_over_rgw * m9**(alph_gw-alpha_char) * eta_norm**beta_gw

    
    for i,a in enumerate(alpha_gw_arr):  
        if i % alphskip != 0: 
            continue
        ax1.plot(np.log10(mt[:,0,i,0,0,0]/MSOL), lgr9rg_min_allowed[:,0,i,0,0,0],
                 label=fr"$\alpha_GW=${a:.2g}",color=colors[i])

        for j,r in enumerate(rchar_9_arr):
            if j % rch9skip != 0:
                continue
                
            # rcritGW < rchar criterion
            if i==0:
                ax1.plot(np.log10(mt[:,0,i,j,0,0]/MSOL), lgr9rg_max_allowed[:,0,i,j,0,0],
                         color=colors[i],ls=lstyles[j]) #,label=fr"rch9={r/PC:.2g}pc")
                ax2.plot(np.log10(mt[:,0,i,j,0,0]/MSOL), lg_rchar_in_rg[:,0,i,j,0,0], 
                         color=colors[i],ls=lstyles[j],label=fr"rch9[pc]={r/PC:.2g}pc")
            else:
                ax1.plot(np.log10(mt[:,0,i,j,0,0]/MSOL), lgr9rg_max_allowed[:,0,i,j,0,0],color=colors[i],ls=lstyles[j])
                ax2.plot(np.log10(mt[:,0,i,j,0,0]/MSOL), lg_rchar_in_rg[:,0,i,j,0,0],color=colors[i],ls=lstyles[j])
            if lgr9rg_max_allowed[:,0,i,j,0,0].min()>lgr9rg_min_allowed[:,0,i,j,0,0].max():
                ax1.fill_between(np.log10(mt[:,0,i,j,0,0]/MSOL), 
                                 lgr9rg_max_allowed[:,0,i,j,0,0].min(),lgr9rg_min_allowed[:,0,i,j,0,0].max(),
                                 alpha=0.2, color=colors[i],ls=lstyles[j])

            # testing compared to new function
            for ix_m in mt_ix_to_plot:
                for ix_q,q in enumerate(mrat_arr):
                    ax1.plot(np.log10(mt[ix_m,ix_q,i,j,0,0]/MSOL), _min_lgr9rg[ix_m,ix_q,i,j,0,0], 
                             '^', color='g',ls=lstyles[j],alpha=0.2)
                    ax1.plot(np.log10(mt[ix_m,ix_q,i,j,0,0]/MSOL), _max_lgr9rg[ix_m,ix_q,i,j,0,0], 
                             'o', color='g',ls=lstyles[j],alpha=0.2)

    ax1.legend()                
    ax2.legend()


    figb = plt.figure(figsize=(12,9))

    plot_index = 231
    for ix_m in mt_ix_to_plot:
        ax = figb.add_subplot(plot_index)
        plt.xlabel('log10(r9/Rg)')
        plt.ylabel('min(nu_inner)')
        #plt.ylabel('log10(-dadt[cm/s])')
        #plt.ylim(-50,50)
        plt.title(rf'log10(Mtot/Msun) = {np.log10(mtot_arr[ix_m]/MSOL)}')
        
        for ix_q,q in enumerate(mrat_arr):
            for i,a in enumerate(alpha_gw_arr):  
                for j,r in enumerate(rchar_9_arr):            
                    if i % alphskip != 0 or j % rch9skip != 0:
                        continue

                    #if np.any(min_lgdadtrch_nuabsmax[ix_m,ix_q,i,j,0,:]>max_lgdadtrch_nuabsmax[ix_m,ix_q,i,j,0,:]):
                    #    print(f"WARNING: invalid dadt for logM={np.log10(mtot_arr[ix_m]/MSOL)}, "
                    #          f"mrat={mrat_arr[ix_q]}, log(rchar9[pc])={r/PC}, {a=}")
                    #    #print(f"    {rgw_crit[ix_m,:,i]=}  {dadt_gw_crit[ix_m,:,i]=}")
                    #    #print(f"    {min_lgdadtrch_nuabsmax=} > {max_lgdadtrch_nuabsmax=}")
                        
                    if plot_index == 231 and i==0 and ix_q==mt_ix_to_plot[0]: 
                        ax.plot(np.log10(r9_rg_arr), _min_nuin[ix_m,ix_q,i,j,0,:],
                        #ax.plot(np.log10(r9_rg_arr), _min_nuin[:,:,i,j,0,:].max(axis=(0,1)),
                                lw=2,color=colors[i],ls=lstyles[j],label=fr"rchar9[pc]={r/PC:.2g}pc")
                        #ax.plot(np.log10(r9_rg_arr), max_lgdadtrch_nuabsmax[ix_m,ix_q,i,j,0,:],
                        #        lw=2,color=colors[i],ls=lstyles[j],label=fr"rchar9[pc]={r/PC:.2g}pc")
                        #ax.plot(np.log10(r9_rg_arr), min_lgdadtrch_nuabsmax[ix_m,ix_q,i,j,0,:],lw=1,color=colors[i],ls=lstyles[j]) 
                        #print(f"{r9_rg_arr.shape=} {min_lgdadtrch_nuabsmax.shape=} {max_lgdadtrch_nuabsmax.shape=}")
                        #ax.fill_between(np.log10(r9_rg_arr), min_lgdadtrch_nuabsmax[ix_m,ix_q,i,j,0,:], 
                        #                np.minimum(lg_speedlimit,max_lgdadtrch_nuabsmax[ix_m,ix_q,i,j,0,:]),
                        #                alpha=0.05, color=colors[i],ls=lstyles[j])    
                    elif plot_index == 232 and j==0 and ix_q==mt_ix_to_plot[0]:
                        ax.plot(np.log10(r9_rg_arr), _min_nuin[ix_m,ix_q,i,j,0,:],
                        #ax.plot(np.log10(r9_rg_arr), _min_nuin[:,:,i,j,0,:].max(axis=(0,1)),
                                lw=2,color=colors[i],ls=lstyles[j],label=fr"$\alpha_GW=${a:.2g}")
                        #ax.plot(np.log10(r9_rg_arr), min_lgdadtrch_nuabsmax[ix_m,ix_q,i,j,0,:],
                        #        lw=1,color=colors[i],ls=lstyles[j])                                                
                        #ax.fill_between(np.log10(r9_rg_arr), min_lgdadtrch_nuabsmax[ix_m,ix_q,i,j,0,:], 
                        #                np.minimum(lg_speedlimit,max_lgdadtrch_nuabsmax[ix_m,ix_q,i,j,0,:]),
                        #                alpha=0.05, color=colors[i],ls=lstyles[j])
                    else:
                        ax.plot(np.log10(r9_rg_arr), _min_nuin[ix_m,ix_q,i,j,0,:],
                        #ax.plot(np.log10(r9_rg_arr), _min_nuin[:,:,i,j,0,:].max(axis=(0,1)),
                                lw=2,color=colors[i],ls=lstyles[j])
                        #ax.plot(np.log10(r9_rg_arr), min_lgdadtrch_nuabsmax[ix_m,ix_q,i,j,0,:],
                        #        lw=1,color=colors[i],ls=lstyles[j])

                    #print(_min_lgdadtrchar[ix_m,ix_q,i,j,0,:])
                    #print(_max_lgdadtrchar[ix_m,ix_q,i,j,0,:])
                    #print(f"r9rg={r9rg[ix_m,ix_q,i,j,0,:]}")
                    #ax.plot(np.log10(r9rg[ix_m,ix_q,i,j,0,:]), _max_lgdadtrchar[ix_m,ix_q,i,j,0,:],'o',
                    #        markersize=j+3,color=colors[i],alpha=0.2)
                    #ax.plot(np.log10(r9rg[ix_m,ix_q,i,j,0,:]), _min_lgdadtrchar[ix_m,ix_q,i,j,0,:],'^',
                    #        markersize=j+3,color=colors[i],alpha=0.2)
                    #ax.plot(np.log10(r9_rg_arr),'^',markersize=j+1,color=colors[i],alpha=0.2)                                          
                    #print(f"mt={mtot_arr[ix_m]/MSOL} mr={mrat_arr[ix_q]} {a=:g} {rc/PC=:g} ")
                    #      #f"{_min_lgr9rg=} {_max_lgr9rg=} {_min_lgdadtrchar=} {_max_lgdadtrchar=}")
        #ax.plot([np.log10(r9_rg_arr[0]),np.log10(r9_rg_arr[-1])],[lg_speedlimit,lg_speedlimit],'m--',lw=3)
        if plot_index == 231 or plot_index == 232:
            ax.legend() 
        plot_index += 1

    figc = plt.figure(figsize=(12,9))

    ax1c = figc.add_subplot(231)
    plt.xlabel('log10(rgw9/Rg)')
    plt.ylabel(r'min($\nu_{\rm in}$))')
    ax2c = figc.add_subplot(232)
    plt.xlabel('log10(rgw9/pc)')
    plt.ylabel(r'min($\nu_{\rm in}$))')
    ax3c = figc.add_subplot(233)
    plt.ylim(-0.5,10)
    plt.xlabel('log10(rgw9/Rg)')
    plt.ylabel('log10(rchar/rgw)')
    ax4c = figc.add_subplot(234)
    plt.ylim(-0.5,10)
    plt.xlabel('log10(rgw9/Rg)')
    plt.ylabel('log10(rch9/rgw9)')
    ax5c = figc.add_subplot(235)
    #plt.ylim(-0.5,10)
    plt.xlabel('log10(rgw9/Rg)')
    plt.ylabel('log10(rch9/pc)')
    ax6c = figc.add_subplot(236)
    #plt.ylim(-0.5,10)
    plt.xlabel('log10(rgw9/pc)')
    plt.ylabel('log10(rch9/pc)')
    
    for i,a in enumerate(alpha_gw_arr):  
        if i % alphskip != 0:
            continue
        for j,r in enumerate(rchar_9_arr):            
            if j % rch9skip != 0:
                continue
            ax1c.plot(np.nanmax(lgr9rg_min_allowed[:,:,i,j,0,0],axis=(0,1)), 
                      np.nanmax(_min_nuin[:,:,i,j,0,0],axis=(0,1)), 'k^')
            ax1c.plot(np.nanmax(_min_lgr9rg[:,:,i,j,0,0],axis=(0,1)), 
                      np.nanmax(_min_nuin[:,:,i,j,0,0],axis=(0,1)), 'rs', markersize=2,alpha=0.5)
            ax1c.plot(np.nanmin(lgr9rg_max_allowed[:,:,i,j,0,-1],axis=(0,1)), 
                      np.nanmax(_min_nuin[:,:,i,j,0,-1],axis=(0,1)), 'b^')
            print(np.nanmin(lgr9rg_max_allowed[:,:,i,j,0,:],axis=(0,1)))
            print(np.nanmax(_min_nuin[:,:,i,j,0,:],axis=(0,1)))
            ax1c.plot(np.log10(r9_rg_arr), _min_nuin[:,:,i,j,0,:].max(axis=(0,1)),
                      lw=2,color=colors[i],ls=lstyles[j])
            ax2c.plot(np.log10(r9_cm_arr/PC), _min_nuin[:,:,i,j,0,:].max(axis=(0,1)),
                      lw=2,color=colors[i],ls=lstyles[j])
            
        for k,n in enumerate(nuin_arr):
            if k % nuskip != 0:
                continue
            if n >= 1:
                continue
                
            ax3c.plot(np.log10(r9_rg_arr), np.log10(max_rchar_over_rgw[:,:,i,0,k,:].min(axis=(0,1))),
                      lw=2,color=colors[i],ls=lstyles[k])
            ax3c.plot([np.log10(r9_rg_arr[0]),np.log10(r9_rg_arr[-1])], [0,0], lw=2, color='k', ls='--')
            if i==0:
                ax4c.plot(np.log10(r9_rg_arr), np.log10(max_rch9_over_rgw9[:,:,i,0,k,:].min(axis=(0,1))),
                          lw=2,color=colors[i],ls=lstyles[k],label=f'nu_in={n}')
            else:
                ax4c.plot(np.log10(r9_rg_arr), np.log10(max_rch9_over_rgw9[:,:,i,0,k,:].min(axis=(0,1))),
                          lw=2,color=colors[i],ls=lstyles[k])
            ax4c.plot([np.log10(r9_rg_arr[0]),np.log10(r9_rg_arr[-1])], [0,0], lw=2, color='k', ls='--')

            ax5c.plot(np.log10(r9_rg_arr), np.log10(max_rch9_over_rgw9[:,:,i,0,k,:].min(axis=(0,1))*r9_cm_arr/PC),
                      lw=2,color=colors[i],ls=lstyles[k])
            ax5c.plot(np.log10(r9_rg_arr), np.log10(r9_cm_arr/PC), 'k:',lw=3)
            ax6c.plot(np.log10(r9_cm_arr/PC), np.log10(max_rch9_over_rgw9[:,:,i,0,k,:].min(axis=(0,1))*r9_cm_arr/PC),
                      lw=2,color=colors[i],ls=lstyles[k])
            ax6c.plot(np.log10(r9_cm_arr/PC), np.log10(r9_cm_arr/PC), 'k:',lw=3)
    ax4c.legend()

def get_rchar_max(eta_norm, M, nu_in, rgw_crit_PC):

    x0 = 1.0e4 * rgw_crit_PC
    rchar_max_PC = scipy.optimize.fsolve(rchar_max_func, x0, 
                       args=(eta_norm, M, nu_in, rgw_crit_PC))

    return rchar_max_PC

def get_rch9_max(eta_norm, M, nu_in, rgw9_crit_PC, alpha_gw, beta_gw, alpha_ch):

    x0 = 1.0e4 * rgw9_crit_PC
    rch9_max_PC = scipy.optimize.fsolve(
        rchar9_max_func, x0, args=(eta_norm, M, nu_in, rgw9_crit_PC,
                                   alpha_gw, beta_gw, alpha_ch)
    )

    return rch9_max_PC

def rchar_max_func(x_PC, eta_norm, M, nu_in, rgw_crit_PC):
    const = 5 * SPLC**6 / (16 * NWTG**3)
    fac = (1 + (x_PC/rgw_crit_PC)**(4.0-nu_in))
    
    return const - ( eta_norm * M**3 / x_PC**3 ) * fac

def rchar9_max_func(x_PC, eta_norm, M, nu_in, rgw9_crit_PC, 
                    alpha_gw, beta_gw, alpha_ch):
    const = 5 * SPLC**6 / (16 * NWTG**3)
    m9 = M/(1.0e9*MSOL)
    rgw_crit_PC = rgw9_crit_PC * m9**(alpha_gw+1) * eta_norm**beta_gw
    fac = (1 + (x_PC * m9**(alpha_ch+1))/rgw_crit_PC)**(4.0-nu_in)
    
    return const - ( eta_norm * M**3 / (x_PC * m9**(alpha_ch+1))**3 ) * fac

def lgr9pc2rg(lgr9pc):
    return np.log10(10.0**(lgr9pc)*PC/utils.gravitational_radius(1.0e9*MSOL))
def lgr9rg2pc(lgr9rg):
    return np.log10(10.0**(lgr9rg)*utils.gravitational_radius(1.0e9*MSOL)/PC) 

In [ ]:
plot_allowed_param_space_model0(Nmt=9, Nmr=4, Nrch=4, Nalgw=3, Nnu=4, Nr9=100,beta_gw=+0.5)

In [ ]:
def paper_plot_param_space_with_rgw_limits_model0(isco_in_rg=6.0, gw_crit_units='rg',
                                                  nu_inner_max=10.0, speed_limit=SPLC,
                                                  lgmtot_range=[4.0,12.0],
                                                  lgmrat_range=[-3.0,0.0],
                                                  alpha_char=-2/3, beta_gw=0,
                                                  alpha_gw=[-1.0,-0.5,-0.25,0.0], 
                                                  rchar_9=[0.1,1.0,100.0],
                                                  fid_alphgw=-0.25, fid_rch9=10**0.5,
                                                  nuin=None, rgw9=None,
                                                  var_pars=['rgw9','nuin'],
                                                  Tobs_yr = 100.0,
                                                  N_mtot_plots=3):
    """
    This function aspires to be significantly less shitty than `plot_allowed_param_space_model0`.

    We still have work to do.
    """

    if not np.isscalar(alpha_char):
        raise ValueError("Only one value for `alpha_char` may be specified in this function.")
    if not np.isscalar(beta_gw):
        raise ValueError("Only one value for `beta_gw` may be specified in this function.")
    if not isinstance(alpha_gw,list):
        if np.isscalar(alpha_gw):
            alpha_gw = [alpha_gw]
        else:
            raise ValueError("Keyword `alpha_gw` must be a list or scalar.")
    if not isinstance(rchar_9,list):
        if 'rch9' not in var_pars:
            if np.isscalar(rchar_9):
                rchar_9 = [rchar_9]
            else:
                raise ValueError("Keyword `rchar_9` must be a list or scalar when 'rch9' not in var_pars.")

    if var_pars != ['rgw9','nuin'] or nuin is not None or rgw9 is not None:
        raise ValueError("This function currently only defined for the case var_pars=['rgw9','nuin'].")

    if N_mtot_plots > 6:
        raise ValueError("Maximum value of `N_mtot_plots` is 6.")
        
    #for v in var_pars:
    #    if v=='rgw9':
    #        if rgw9 is not None:
    #            raise ValueError("Keyword `rgw9` must be None if 'rgw9' in `var_pars`.")
    #        do_cool_stuff()
    #    elif v=='nuin':
    #        if nuin is not None:
    #            raise ValueError("Keyword `nuin` must be None if 'nuin' in `var_pars`.")
    #        do_cool_stuff()
    #    elif v=='rch9':
    #        # requires specifying either rgw9 or nuin
    #        raise NotImplementedError()
    #    elif v=='tout':
    #        raise NotImplementedError()
    #    else:
    #        raise ValueError(f"Undefined value {v} in `var_pars`.")

    
    mtot_arr = np.logspace(lgmtot_range[0],lgmtot_range[1],9) * MSOL
    nmskip = int(mtot_arr.size / N_mtot_plots) + 1
    mrat_arr = np.logspace(lgmrat_range[0],lgmrat_range[1],4)
    nqskip = 1
    mt, mr, = np.broadcast_arrays(
        mtot_arr[:, np.newaxis],
        mrat_arr[np.newaxis, :]
    )

    m9 = mt / (1.0e9*MSOL)   # varies with mtot

    m1, m2 = utils.m1m2_from_mtmr(mt, mr)   # varies with mtot & mrat

    eta_norm = mr / np.square(1 + mr) * 4   # varies with mrat

    lg_risco_in_rg = np.log10(isco_in_rg)
    
    rchar_9 = [r*PC for r in rchar_9] # convert to cm

    # make initial subplots 
    #cmap = plot._get_cmap('PuBuGn')
    #colors = cmap(np.linspace(0.3, 0.9, len(alpha_gw)+1))
    cmap = plot._get_cmap('viridis')
    colors = cmap(np.linspace(0.85, 0.05, len(alpha_gw)))
    lstyles = ['-','--',':','-.']*4

    ### figure a
    fig = plt.figure(figsize=(10,7),layout='tight')
    ax1 = fig.add_subplot(221, xlabel=r'log$_{10}(M_{tot}/M_{\odot})$', ylabel=r'log$_{10}(a_{GW,9}/R_g)$',
                          title=r'Min & max allowed values of $a_{GW,9}$ [R$_g$] vs $M_{tot}$')
    #plt.xlabel('log10(Mtot/Msun)')
    #plt.ylabel('log10(r9/Rg)')
    #plt.title(r'Min/max allowed values of r9[Rg] vs Mtot (varied $\alpha_GW$)')
    ax1.plot([np.log10(mtot_arr[0]/MSOL),np.log10(mtot_arr[-1]/MSOL)],
             [lg_risco_in_rg,lg_risco_in_rg],'k:',lw=4)  

    ax2 = fig.add_subplot(222)
    plt.xlabel(r'log$_{10}(M_{tot}/M_{\odot})$')
    plt.ylabel(r'log$_{10}(a_{GW}/R_g)$')
    plt.title(r'Min & max allowed values of $a_{GW}$ [R$_g$] vs $M_{tot}$')    
    ax2.plot([np.log10(mtot_arr[0]/MSOL),np.log10(mtot_arr[-1]/MSOL)],
             [lg_risco_in_rg,lg_risco_in_rg],'k:',lw=4)  

    frst_min = 1 / (Tobs_yr*YR)
    sepa_obs_max = sepa_emit(mtot_arr,frst_min) / utils.gravitational_radius(mtot_arr)
    #print(f"M={m/MSOL:.3g}, {sepa_obs_max=:.3g} Rg = {sepa_obs_max*utils.gravitational_radius(m)/PC:.3g} pc")
    flmi, = ax2.plot(np.log10(mtot_arr/MSOL),np.log10(sepa_obs_max),'k:',lw=4)
                      #alpha=0.7,color=colors[i_plot], lw=2,
                      #label=r'$f_{\rm obs}$'+f'=1/({Tobs_yr:g} yr)')
    
    ### figure b
    figb = plt.figure(figsize=(10,5),layout='tight')
    kwargs_b = {'xlabel':'log10(r9/Rg)', 'ylabel':r'min($\nu_{inner}$)'}
    figb_index = 231
    ax1b = figb.add_subplot(figb_index+0, title=rf'log10(Mtot/Msun) = {np.log10(mtot_arr[0]/MSOL)}', **kwargs_b)
    ax2b = figb.add_subplot(figb_index+1, title=rf'log10(Mtot/Msun) = {np.log10(mtot_arr[nmskip]/MSOL)}', **kwargs_b)
    ax3b = figb.add_subplot(figb_index+2, title=rf'log10(Mtot/Msun) = {np.log10(mtot_arr[nmskip*2]/MSOL)}', **kwargs_b)
    #ax4b = figb.add_subplot(figb_index+3, title=rf'log10(Mtot/Msun) = {np.log10(mtot_arr[nmskip*3]/MSOL)}', **kwargs_b)
    #ax5b = figb.add_subplot(figb_index+4, title=rf'log10(Mtot/Msun) = {np.log10(mtot_arr[nmskip*4]/MSOL)}', **kwargs_b)
    #ax6b = figb.add_subplot(figb_index+5, title=rf'log10(Mtot/Msun) = {np.log10(mtot_arr[0]/MSOL)}', **kwargs_b)

    ### figure c
    figc = plt.figure(figsize=(6,5),layout='tight')

    ax1c = figc.add_subplot(111, ylim=(-10.5,1.5), 
                            xlabel=r'log10($a_{GW,9}/R_g$)',ylabel=r'min($\nu_{inner}$)')
    secax1c = ax1c.secondary_xaxis('top', functions=(lgr9rg2pc, lgr9pc2rg),xlabel='log10($a_{GW,9}$/pc)')
    #ax2c = figc.add_subplot(232, xlabel='log10($a_{GW,9}$/pc)',ylabel=r'min($\nu_{inner}$)')
    
    # do this for each alpha_gw and rchar_9:
    alphgw_legend_vals = []
    rch9_legend_vals = []
    for j,alphgw in enumerate(alpha_gw):
        # varies with mtot, mrat, alpha_gw, & beta_gw:
        lgr9rg_min = np.maximum(lg_risco_in_rg, 
                                lg_risco_in_rg - alphgw * np.log10(m9) - beta_gw * np.log10(eta_norm))
        lgr9rg_strict_min = lgr9rg_min.max()
        lgrgwrg_min = np.log10( 10.0**lgr9rg_min * m9**alphgw * eta_norm**beta_gw)
        #print(f"{lgr9rg_min.shape=} {lgr9rg_min=}")
        #print(f"{lgr9rg_strict_min.shape=} {lgr9rg_strict_min=}")
        
        for k,rch9 in enumerate(rchar_9):
            rchar = rch9 * m9**(alpha_char+1)
            lg_rchar_in_rg = np.log10(rchar/utils.gravitational_radius(mt))
            
            # varies with rchar, mtot, mrat, alpha_gw, & beta_gw
            # rch/rgw>1 for all M,eta requires rgw9 < rch9 * m9^(alpha_char-alphgw) * eta_norm^(-beta_gw)
            # also requires rgw9 < rch9, need to check separately since both rch and rgw have m dependence
            r9cm_max = np.minimum(rch9, rch9 * m9**(alpha_char-alphgw) * eta_norm**(-beta_gw))
            lgr9rg_max = np.log10(r9cm_max / utils.gravitational_radius(1.0e9*MSOL))
            lgr9rg_strict_max = lgr9rg_max.min()
            #print(f"{lgr9rg_max-lgr9rg_min}")
    
            if lgr9rg_strict_max < lgr9rg_strict_min:
                print(f"WARNING: skipping invalid model: {alphgw=} {rch9/PC=}")
                continue
                #raise ValueError(f"invalid model: {alphgw=} {rch9/PC=}. add code to deal with this case.")

            #print(f"{alphgw=:.4g} {rch9/PC=:.4g} rch9/Rg={rch9/utils.gravitational_radius(1.0e9*MSOL):.4g} "
            #      f"{lgr9rg_strict_min=:.4g} {lgr9rg_strict_max=:.4g}")
            dlgr9 = (lgr9rg_strict_max-lgr9rg_strict_min)/50
            #print(f"{dlgr9=}")
            lgr9rg = np.arange(lgr9rg_strict_min+5*dlgr9,lgr9rg_strict_max-4*dlgr9,dlgr9)
            #print(lgr9rg)
            r9rg = 10.0**lgr9rg
            r9cm = r9rg * utils.gravitational_radius(1.0e9*MSOL)

            for ix_q in np.arange(0,mrat_arr.size,nqskip):
                if ix_q==0:
                    if alphgw not in alphgw_legend_vals:
                        ax1.plot(np.log10(mt[:,ix_q]/MSOL), lgr9rg_min[:,ix_q],
                                 label=r"$\alpha_{GW}$="+f"{alphgw:.2g}",color=colors[j])
                        alphgw_legend_vals += [alphgw]
                    if k==len(rch9_legend_vals) and rch9 not in rch9_legend_vals:
                        ax2.plot(np.log10(mt[:,ix_q]/MSOL), lgrgwrg_min[:,ix_q], 
                                 label=r"$a_{char,9}$="+f"{rch9/PC:.2g}pc",ls=lstyles[k],color=colors[j])
                        rch9_legend_vals += [rch9]
                else:
                    ax1.plot(np.log10(mt[:,ix_q]/MSOL), lgr9rg_min[:,ix_q], color=colors[j])
                    ax2.plot(np.log10(mt[:,ix_q]/MSOL), lgrgwrg_min[:,ix_q], color=colors[j])
                    
                ax1.plot(np.log10(mt[:,ix_q]/MSOL), lgr9rg_max[:,ix_q], color=colors[j],ls=lstyles[k])
                if np.abs(alphgw-fid_alphgw)<1.0e-6 and np.abs(rch9/PC-fid_rch9)<1.0e-6:
                    ax1.plot(np.log10(mt[-1,0]/MSOL), lgr9rg_min[-1,0], 'o', 
                             color=colors[j],lw=0,markersize=5)
                    ax1.plot(np.log10(mt[-1,0]/MSOL), lgr9rg_max[-1,0], 'o', 
                             color=colors[j],lw=0,markersize=5)
                    ax1.fill_between(np.log10(mt[:,ix_q]/MSOL), 
                                     lgr9rg_strict_max,lgr9rg_strict_min,
                                     alpha=0.15, color=colors[j],lw=0,zorder=0)
                    ax2.fill_between(np.log10(mt[:,ix_q]/MSOL), 
                                     lg_rchar_in_rg[:,ix_q],lgrgwrg_min[:,ix_q],
                                     alpha=0.15, color=colors[j],lw=0,zorder=0)
            
            ax2.plot(np.log10(mt/MSOL), lg_rchar_in_rg,color=colors[j],ls=lstyles[k])
                
            #if j==0:
            #    ax1.plot(np.log10(mt/MSOL), lgr9rg_max,
            #             color=colors[j],ls=lstyles[k]) #,label=fr"rch9={r/PC:.2g}pc")
            #    #ax2.plot(np.log10(mt[:,0,i,j,0,0]/MSOL), lg_rchar_in_rg[:,0,i,j,0,0], 
            #    #         color=colors[i],ls=lstyles[j],label=fr"rch9[pc]={r/PC:.2g}pc")
            #else:
            #if lgr9rg_max_allowed[:,0,i,j,0,0].min()>lgr9rg_min_allowed[:,0,i,j,0,0].max():
            #    ax1.fill_between(np.log10(mt[:,0,i,j,0,0]/MSOL), 
            #                     lgr9rg_max_allowed[:,0,i,j,0,0].min(),lgr9rg_min_allowed[:,0,i,j,0,0].max(),
            #                     alpha=0.2, color=colors[i],ls=lstyles[j])
            
            rgw_crit = np.zeros((mt.shape[0],mt.shape[1],lgr9rg.size))
            dadt_gw_crit = np.zeros_like(rgw_crit) 
            nuin_min = np.zeros_like(rgw_crit) 
            
            #print(f"{rgw_crit.shape=} {dadt_gw_crit.shape=}")
            for i in range(lgr9rg.size):
                rgw_crit[:,:,i] = r9cm[i] * m9**(alphgw+1) * eta_norm**beta_gw  # varies with mtot, mrat, r9, alpha_gw
                dadt_gw_crit[:,:,i] = utils.gw_hardening_rate_dadt(m1, m2, rgw_crit[:,:,i])   # varies with mtot, mrat, r9, & alpha_gw
                # calc nuin_min for each valid lgr9rg_arr
                lgrdiff = np.log10(rchar)-np.log10(rgw_crit[:,:,i])
                if np.any(lgrdiff<0): 
                    ixbad = np.where(lgrdiff<0)
                    print(f"{ixbad[0]=} {ixbad=}")
                    print(f"{alphgw=} {rch9/PC=} {r9cm[i]/PC=} {lgr9rg[i]=}")
                    print(f"rch={rchar[ixbad]/PC}pc, rgw={rgw_crit[:,:,i][ixbad]/PC}, lgrdiff={lgrdiff[ixbad]}")
                    print(f"mt={mt[ixbad]/MSOL},mr={mr[ixbad]},nu={nuin_min[:,:,i][ixbad]},dadtgw={dadt_gw_crit[:,:,i][ixbad]}")
                    raise ValueError(f"something went wrong {lgrdiff.min()} < 0")
                #print(f"{i=} {rchar[0,0]=} {rgw_crit[0,0,i]=} {lgrdiff[0,0]=}")
                #print(f"{lgrdiff.shape=}")
                nuin_min[:,:,i] = np.maximum( -1.0*nu_inner_max,
                                              1.0 - (np.log10(speed_limit) - np.log10(-dadt_gw_crit[:,:,i])) / lgrdiff )
                #if np.any(nuin_min[:,:,i] > 1):
                #    print(f"{i=} {alphgw=} {rch9/PC=} {lgr9rg[i]=} {r9cm[i]/PC=}")
                #    print(mt[0,0],mr[0,0],nuin_min[0,0,i],dadt_gw_crit[0,0,i],rgw_crit[0,0,i]/PC,lgrdiff[0,0])
                #    print(mt[0,-1],mr[0,-1],nuin_min[0,-1,i],dadt_gw_crit[0,-1,i],lgrdiff[0,-1])
                #    print(mt[-1,0],mr[-1,0],nuin_min[-1,0,i],dadt_gw_crit[-1,0,i],lgrdiff[-1,0])
                #    print(mt[-1,-1],mr[-1,-1],nuin_min[-1,-1,i],dadt_gw_crit[-1,-1,i],lgrdiff[-1,-1])
            #print(f"{nuin_min.shape=}")
            nuin_strict_min = nuin_min.max(axis=(0,1))
            #print(f"{nuin_min=}")
            #print(f"{nuin_strict_min=}")
            
            nuin_max = nu_inner_max
            nuin_strict_max = nu_inner_max
            if np.any(nuin_min > 1):
                raise ValueError(f"invalid model: {nuin_min.max()=} should have been caught earlier.")

            for ix_q in np.arange(0,mrat_arr.size,nqskip):
                ax1b.plot(lgr9rg, nuin_min[0,ix_q,:],
                          lw=2,color=colors[j],ls=lstyles[k],alpha=1-ix_q*0.25)#,label=fr"rchar9[pc]={r/PC:.2g}pc")
                ax2b.plot(lgr9rg, nuin_min[nmskip,ix_q,:],
                          lw=2,color=colors[j],ls=lstyles[k],alpha=1-ix_q*0.25)
                ax3b.plot(lgr9rg, nuin_min[nmskip*2,ix_q,:],
                          lw=2,color=colors[j],ls=lstyles[k],alpha=1-ix_q*0.25)
                #ax4b.plot(lgr9rg, nuin_min[nmskip*3,ix_q,:],
                #          lw=2,color=colors[j],ls=lstyles[k])
                #ax5b.plot(lgr9rg, nuin_min[nmskip*4,ix_q,:],
                #          lw=2,color=colors[j],ls=lstyles[k])
                #ax6b.plot(lgr9rg, nuin_min[nmskip*5,ix_q,:],
                #          lw=2,color=colors[j],ls=lstyles[k])
                
            if np.abs(alphgw+0.25)<1.0e-6 and np.abs(rch9/PC-3.0)<1.0e-6:
                shading = 0.45
                ms=6
            else:
                shading = 0.08
                ms=4
            ## changing this from lgr9rg (limited to the 'allowed' values if we force risco<rgw<rchar)
            ## to an array of unrestricted values 
            #print(nuin_strict_min.shape)
            lgr9rg_arr = np.linspace(-1,10,nuin_strict_min.size)
            # the dots were to mark the allowed range of lgr9rg, commenting
            #ax1c.plot(lgr9rg[0], nuin_strict_min[0], 'o',color=colors[j],lw=0,markersize=ms)
            #ax1c.plot(lgr9rg[-1], nuin_strict_min[-1], 'o',color=colors[j],lw=0,markersize=ms)
            ax1c.plot(lgr9rg_arr, nuin_strict_min, lw=2.5,color=colors[j],ls=lstyles[k])
            ax1c.fill_between(lgr9rg_arr, nuin_strict_min, 1.5, color=colors[j], lw=0, alpha=shading,zorder=0)
            #ax2c.plot(np.log10((10.0**lgr9rg)*utils.gravitational_radius(1.0e9*MSOL)/PC), 
            #          nuin_strict_min, lw=2,color=colors[j],ls=lstyles[k])
    ax1.legend(loc='lower left')    
    ax2.legend(loc='upper right')    
    fig.savefig('allowed_rgw_vs_Mtot_with_rgw_limits.png', dpi=300)
    figb.savefig('allowed_rgw_vs_nuin_per_Mtot_with_rgw_limits.png', dpi=300)
    figc.savefig('allowed_rgw_vs_nuin_with_rgw_limits.png', dpi=300)

In [ ]:
def paper_plot_param_space_model0(isco_in_rg=6.0, gw_crit_units='rg',
                                  nu_inner_max=10.0, speed_limit=SPLC,
                                  lgmtot_range=[4.0,12.0],
                                  lgmrat_range=[-3.0,0.0],
                                  alpha_char=-2/3, beta_gw=None,
                                  alpha_gw=[-1.0,-0.5,-0.25,0.0], 
                                  rchar_9=[0.1,1.0,100.0],
                                  fid_alphgw=-0.25, fid_rch9=10**0.5,
                                  nuin=None, rgw9=None,
                                  var_pars=['rgw9','nuin'],
                                  Tobs_yr = None, fobs_min=9.0e-10,
                                  N_mtot_plots=3, final_models_only=True):
    """
    Just the nu_inner vs r9 limits.
    """

    if not np.isscalar(alpha_char):
        raise ValueError("Only one value for `alpha_char` may be specified in this function.")

    if not isinstance(alpha_gw,list):
        if np.isscalar(alpha_gw):
            alpha_gw = [alpha_gw]
        else:
            raise ValueError("Keyword `alpha_gw` must be a list or scalar.")
    if beta_gw is not None:
        log.warning(f"Ignoring keyword value of {beta_gw=} to plot pre-defined models.")
        
    if not isinstance(rchar_9,list):
        if 'rch9' not in var_pars:
            if np.isscalar(rchar_9):
                rchar_9 = [rchar_9]
            else:
                raise ValueError("Keyword `rchar_9` must be a list or scalar when 'rch9' not in var_pars.")

    if var_pars != ['rgw9','nuin'] or nuin is not None or rgw9 is not None:
        raise ValueError("This function currently only defined for the case var_pars=['rgw9','nuin'].")

    if N_mtot_plots > 6:
        raise ValueError("Maximum value of `N_mtot_plots` is 6.")
        
    #mtot_arr = np.logspace(lgmtot_range[0],lgmtot_range[1],9) * MSOL
    mtot_arr = np.logspace(lgmtot_range[0],lgmtot_range[1],900) * MSOL
    nmskip = int(mtot_arr.size / N_mtot_plots) + 1
    mrat_arr = np.logspace(lgmrat_range[0],lgmrat_range[1],2)
    print(f"{mrat_arr=}")
    nqskip = 1
    mt, mr, = np.broadcast_arrays(
        mtot_arr[:, np.newaxis],
        mrat_arr[np.newaxis, :]
    )

    m9 = mt / (1.0e9*MSOL)   # varies with mtot

    m1, m2 = utils.m1m2_from_mtmr(mt, mr)   # varies with mtot & mrat

    eta_norm = mr / np.square(1 + mr) * 4   # varies with mrat

    lg_risco_in_rg = np.log10(isco_in_rg)
    
    rchar_9 = [r*PC for r in rchar_9] # convert to cm

    # ---- SET COLORS
    #cmap = plot._get_cmap('PuBuGn')
    #colors = cmap(np.linspace(0.3, 0.9, len(alpha_gw)+1))
    cmap = plot._get_cmap('viridis')
    #colors = [cmap(0.55),cmap(0.1)]*((len(alpha_gw)+1)//2)
    colors = [cmap(0.15),cmap(0.6)]*((len(alpha_gw)+1)//2)
    aobs_color = cmap(0.85)
    #color_arr = cmap(np.linspace(0.9, 0.1, 10))
    #colors = 
    lstyles = ['--','-','-.',':']*4


    if fobs_min is None:
        fobs_min = 1 / (Tobs_yr*YR)
    sepa_obs_max = sepa_emit(mtot_arr,fobs_min) / utils.gravitational_radius(mtot_arr)
    lg_sepa_obs_max_9 = np.log10( sepa_emit(1.0e9*MSOL,fobs_min) / 
                                  utils.gravitational_radius(1.0e9*MSOL) )

    mpl.rcParams.update({'font.size': 12})
     #np.log10( max(rchar_9)*1.02 / utils.gravitational_radius(1.0e9*MSOL)))
    if nu_inner_max==4:
        ylimit=(-4.9, 2.4)
    else:
        ylimit=(-10.5,2.4)

    agw9_xlimit=(lg_risco_in_rg-0.02, 5.75)    
    #agw_xlimit=(lg_risco_in_rg-0.03, 9)    
    agw_xlimit=(0, 9)    
    #xlimit=(3,4) #np.log10( max(rchar_9)*1.02 / utils.gravitational_radius(1.0e9*MSOL)))
    #ylimit=(-5,0)
    
    ### figure b
    figb, axsb = plt.subplots(nrows=3, ncols=1, sharex=True, figsize=[4,4.5])
    figb.subplots_adjust(hspace=0, left=0.2, right=0.95,top=0.95, bottom=0.12)
    #figb = plt.figure(figsize=(5,10),layout='tight')
    kwargs_b = {'ylabel':r'min( $\nu_{\rm in}$ )','xlim':agw_xlimit, 'ylim':ylimit}
    for i in range(3):
        axsb[i].set(**kwargs_b)
    axsb[2].set(xlabel=r'log$_{10}$($a_{\rm GW}/R_g$)')
    #axsb[0].set(title=rf'log10(Mtot/Msun) = {np.log10(mtot_arr[0]/MSOL)}', **kwargs_b)
    #axsb[1].set(title=rf'log10(Mtot/Msun) = {np.log10(mtot_arr[nmskip]/MSOL)}', **kwargs_b)
    #axsb[2].set(title=rf'log10(Mtot/Msun) = {np.log10(mtot_arr[nmskip*2]/MSOL)}', **kwargs_b)
    #plt.subplots_adjust(hspace=0)

    ### figure c
    figc = plt.figure(figsize=(6,4.5)) #,layout='tight')
    ax1c = figc.add_subplot(111, xlim=agw9_xlimit, ylim=ylimit, 
                            xlabel=r'log$_{10}$($a_{\rm GW,9}/R_g$)',ylabel=r'min( $\nu_{\rm in}$ )')
    secax1c = ax1c.secondary_xaxis('top', functions=(lgr9rg2pc, lgr9pc2rg),xlabel=r'log$_{10}$($a_{\rm GW,9}$/pc)')
    figc.subplots_adjust(left=0.15, right=0.95, top=0.88, bottom=0.12)  
    #ax2c = figc.add_subplot(232, xlabel='log10($a_{GW,9}$/pc)',ylabel=r'min($\nu_{inner}$)')
    
    # do this for each alpha_gw and rchar_9:
    mod_legend_vals = []
    r_legend_vals = []
    lgr9rg_arr = np.linspace(agw9_xlimit[0],agw9_xlimit[1],500)

    for j,alphgw in enumerate(alpha_gw):

        if alphgw == -0.25:
            beta_gw = +0.25
        elif alphgw == 0.0:
            beta_gw = 0.0
        else: 
            if final_models_only:
                continue
            else: beta_gw=+0.25
            
        for k,rch9 in enumerate(rchar_9):

            fidmod = True if alphgw==-0.25 and rch9==1.0*PC else False

            lgrch9rg = np.log10( rch9 / utils.gravitational_radius(1.0e9*MSOL) )
            rchar = rch9 * m9**(alpha_char+1)
            grav_radii = utils.gravitational_radius(mt)
            lg_rchar_in_rg = np.log10(rchar/grav_radii)
                
            r9rg = 10.0**lgr9rg_arr
            r9cm = r9rg * utils.gravitational_radius(1.0e9*MSOL)

            
            rgw_crit = np.zeros((mt.shape[0],mt.shape[1],lgr9rg_arr.size))
            dadt_gw_crit = np.zeros_like(rgw_crit) 
            nuin_min = np.zeros_like(rgw_crit) * np.nan
            
            for i in range(lgr9rg_arr.size):
                for im in range(mt.shape[0]):
                    for iq in range(mt.shape[1]):
                        rgw_crit[im,iq,i] = r9cm[i] * m9[im,iq]**(alphgw+1) * eta_norm[im,iq]**beta_gw
                        if rgw_crit[im,iq,i] < isco_in_rg * grav_radii[im,iq]: 
                            rgw_crit[im,iq,i] = isco_in_rg * grav_radii[im,iq]
                        if rgw_crit[im,iq,i] < rchar[im,iq]:
                            if rgw_crit[im,iq,i] >= isco_in_rg * grav_radii[im,iq]: 
                                dadt_gw_crit[im,iq,i] = utils.gw_hardening_rate_dadt(m1[im,iq], m2[im,iq], 
                                                                                     rgw_crit[im,iq,i])
                            # calc nuin_min for each lgr9rg_arr that has inner phase
                            lgrdiff = np.log10(rchar[im,iq])-np.log10(rgw_crit[im,iq,i])
                            if lgrdiff<0: 
                                print(f"{alphgw=} {rch9/PC=} {r9cm[i]/PC=} {lgr9rg_arr[i]=}")
                                raise ValueError(f"something went wrong. {lgrdiff=} < 0")
                            nuin_min[im,iq,i] = np.maximum( -1.0*nu_inner_max,
                                                            1.0 - (np.log10(speed_limit) - 
                                                                   np.log10(-dadt_gw_crit[im,iq,i])) / lgrdiff )
                            if nuin_min[im,iq,i] > 1:
                                print(f"{mt[im,iq]=} {mr[im,iq]=} {rgw_crit[im,iq,i]/PC=} {rchar[im,iq]/PC=} "
                                      f"dadt(rgw)={np.log10(-dadt_gw_crit[im,iq,i])} splim={np.log10(speed_limit)} {lgrdiff=}")
                                raise ValueError(f"something went wrong. {nuin_min[im,iq,i]=}")
            rgw_crit_in_rg = rgw_crit / grav_radii[...,np.newaxis]
            nuin_strict_min = np.nanmax(nuin_min, axis=(0,1))
            
            nuin_max = nu_inner_max
            nuin_strict_max = nu_inner_max

            linewt = 3 if fidmod else 2

            if (alphgw==-0.25) or (alphgw==0 and rch9==1*PC) or not final_models_only:
                for n in range(3):
                    for ix_q in np.arange(0,mrat_arr.size,nqskip):
                        #if rch9==1*PC:
                        #    print(f"{alphgw=} nuin_min={nuin_min[n*nmskip,ix_q,0]},{nuin_min[n*nmskip,ix_q,-1]} "
                        #          f"rgw={rgw_crit_in_rg[n*nmskip,ix_q,0]},{rgw_crit_in_rg[n*nmskip,ix_q,-1]}")
                        axsb[n].plot(np.log10(rgw_crit_in_rg[n*nmskip,ix_q,:]), nuin_min[n*nmskip,ix_q,:],
                                     lw=linewt,color=colors[j],ls=lstyles[k],alpha=1-ix_q*0.75)
                        #axsb[n].plot(lgr9rg_arr, nuin_min[n*nmskip,ix_q,:],
                        #             lw=linewt,color=colors[j],ls=lstyles[k],alpha=1-ix_q*0.75)

                    txt_loc_y = -4.5 if nu_inner_max==4 else -9
                    if int(np.log10(mtot_arr[n*nmskip]/MSOL)) == 4:
                        axsb[n].text(1.2,txt_loc_y,r"$M=10^4$M$_{\odot}$",fontsize=12)
                    elif int(np.log10(mtot_arr[n*nmskip]/MSOL)) == 8:
                        axsb[n].text(6.3,txt_loc_y,r"$M=10^8$M$_{\odot}$",fontsize=12)
                    elif int(np.log10(mtot_arr[n*nmskip]/MSOL)) == 12:
                        axsb[n].text(6.1,txt_loc_y,r"$M=10^{12}$M$_{\odot}$",fontsize=12)
                        
                    #axsb[n].text(2.5,0,(r"log$_{10}(M/M_{\odot}$)="+
                    #                     f"{np.log10(mtot_arr[n*nmskip]/MSOL)}"))
                    if j==0:
                        if k==0:
                            axsb[n].plot([lg_risco_in_rg,lg_risco_in_rg],[ylimit[0],ylimit[1]],
                                         ':',color='k',lw=4,zorder=3)
                            axsb[n].plot([np.log10(sepa_obs_max[n*nmskip]),np.log10(sepa_obs_max[n*nmskip])],
                                         [ylimit[0],ylimit[1]],':',color=aobs_color,lw=4,zorder=3)
                        for ix_q in np.arange(0,mrat_arr.size,nqskip):
                            axsb[n].plot([lg_rchar_in_rg[n*nmskip,ix_q],lg_rchar_in_rg[n*nmskip,ix_q]],
                                         [ylimit[0],ylimit[1]],color='k',ls=lstyles[k],
                                         lw=linewt-1,zorder=3)
            
            ## changing this from lgr9rg (limited to the 'allowed' values if we force risco<rgw<rchar)
            ## to an array of unrestricted values 
            #print(nuin_strict_min.shape)
            # the dots were to mark the allowed range of lgr9rg, commenting
            #ax1c.plot(lgr9rg[0], nuin_strict_min[0], 'o',color=colors[j],lw=0,markersize=ms)
            #ax1c.plot(lgr9rg[-1], nuin_strict_min[-1], 'o',color=colors[j],lw=0,markersize=ms)

            shading = 0.45 if fidmod else 0.15
            if j==0:
                if k==0:
                    lrisco, = ax1c.plot([lg_risco_in_rg,lg_risco_in_rg],[ylimit[0],ylimit[1]],':',color='k',lw=4,
                                        label=r"$r_{\rm ISCO}$",zorder=3)
                    r_legend_vals += [lrisco]
                    lrobs, = ax1c.plot([lg_sepa_obs_max_9,lg_sepa_obs_max_9],[ylimit[0],ylimit[1]],':',
                                       color=aobs_color,lw=4,label=r"$a$($f_{\rm obs,min}$)",zorder=3)
                    r_legend_vals += [lrobs]
                lrch, = ax1c.plot([lgrch9rg,lgrch9rg],[ylimit[0],ylimit[1]],color='k',ls=lstyles[k],
                                  lw=linewt-1,label=r"$r_{\rm char,9}$="+f"{rch9/PC:.2g}pc",zorder=3)
                r_legend_vals += [lrch]
                print(f"{alphgw=} {rch9=}")
                
            if (alphgw==-0.25 or rch9==1*PC) or not final_models_only:
                lbl = None
                if alphgw==-0.25 and rch9==1*PC:
                    lbl = r"Allowed A0 variants"
                elif alphgw==0.0 and rch9==1*PC:
                    lbl = r"Allowed B0 variants"
                if lbl is not None:
                    lmod,=ax1c.plot(lgr9rg_arr[lgr9rg_arr<=lgrch9rg], nuin_strict_min[lgr9rg_arr<=lgrch9rg], 
                                    lw=linewt,color=colors[j],ls=lstyles[k],label=lbl)
                    mod_legend_vals += [lmod]
                else:
                    lmod,=ax1c.plot(lgr9rg_arr[lgr9rg_arr<=lgrch9rg], nuin_strict_min[lgr9rg_arr<=lgrch9rg], 
                                    lw=linewt,color=colors[j],ls=lstyles[k])
                ax1c.plot(lgr9rg_arr[lgr9rg_arr>lgrch9rg], nuin_strict_min[lgr9rg_arr>lgrch9rg], 
                          lw=linewt-1,color=colors[j],ls=lstyles[k],alpha=0.2)
                ax1c.fill_between(lgr9rg_arr[lgr9rg_arr<=lgrch9rg], nuin_strict_min[lgr9rg_arr<=lgrch9rg], 
                                  ylimit[1], color=colors[j], lw=0, alpha=shading,zorder=2)

    lm,= ax1c.plot([2.5,2.5],[0.0,0.0],'ko',markersize=7,label="A0 & B0",zorder=3)
    mod_legend_vals += [lm]
    lm,= ax1c.plot([2.0,2.0],[2.0,2.0],'k^',markersize=7,label="Agas & Bgas",zorder=3)
    mod_legend_vals += [lm]
    lm,= ax1c.plot([3.5,3.5],[-1.0,-1.0],'k*',markersize=8,label="Astar & Bstar",zorder=3)
    mod_legend_vals += [lm]

    leg2 = ax1c.legend(handles=r_legend_vals, loc='lower left',fontsize=11)
    mod_leg_loc = 'lower center' if nu_inner_max==10 else 'upper right'
    ax1c.legend(handles=mod_legend_vals, loc=mod_leg_loc,fontsize=11)
    ax1c.add_artist(leg2)

    #ax1c.legend(loc='lower left')    
    #ax2.legend(loc='upper right')    
    figb.savefig('allowed_rgw_vs_nuin_per_Mtot_highMres.png', dpi=300)
    if mtot_arr.size<100:
        figc.savefig('allowed_rgw_vs_nuin_lowMres.png', dpi=300)
    else:
        figc.savefig('allowed_rgw_vs_nuin.png', dpi=300)


In [ ]:
paper_plot_param_space_model0(alpha_gw=[-0.25,0],rchar_9=[0.1,1.0,10],
                              alpha_char=-2/3, nu_inner_max=4.0) #,Tobs_yr=20) #,final_models_only=False)


In [ ]:
paper_plot_param_space_with_rgw_limits_model0(alpha_gw=[-0.25],rchar_9=[10**-0.5,1.0,10**0.5,1000],
                                             beta_gw=0.25,alpha_char=-2.0/3,Tobs_yr=20)
#paper_plot_param_space_model0(alpha_gw=[-0.25],rchar_9=[0.1,1.0,10,100],beta_gw=0)
#paper_plot_param_space_model0(alpha_gw=[-1,-0.5,-0.25,0],rchar_9=[3.0],beta_gw=0)
#paper_plot_param_space_model0(alpha_gw=[-1,-0.25,0],rchar_9=[0.1,3.0,100],beta_gw=0)
#paper_plot_param_space_model0(alpha_gw=[-0.25],rchar_9=[3.0],beta_gw=0)


In [ ]:
paper_plot_param_space_model0(alpha_gw=[0],rchar_9=[10**-0.5,10**0.5,1000],fid_alphgw=0, fid_rch9=10**0.5,
                              beta_gw=0,alpha_char=0,Tobs_yr=20)

In [ ]:
paper_plot_param_space_model0(alpha_gw=[-0.25],rchar_9=[10**-0.5,10**0.5,1000],beta_gw=0,alpha_char=-2/3,Tobs_yr=20)

In [ ]:
1000**(1/3-2/3)

In [ ]:
0.5623413251903516*(1000)**(3/4)

In [ ]:
_M, _eta = 1.0e9*MSOL, 0.1
get_rchar_max(4*_eta,_M,-0.5,100*utils.gravitational_radius(M)/PC)

In [ ]:
_M, _eta = 1.0e4*MSOL, 0.001
get_rch9_max(4*_eta,_M,-1,10000*utils.gravitational_radius(M)/PC, -0.25, 0, -2/3)

In [ ]:
plot_allowed_param_space(N=3, alpha_char=-0.5)

In [ ]:
plot_param_space(sam_newhard_data)

In [ ]:
5 * SPLC**5 / (16* NWTG**(5./3) * (2*np.pi)**(8./3)) / YR**(-8/3) * (1e9*MSOL)**(-5/3)/YR *.5**(-8/3)

In [ ]:
NWTG

In [ ]:
(1e9*MSOL)

In [ ]:
ayr=(np.sqrt(NWTG*1e9*MSOL)/(2*np.pi*(1/YR)))**(2/3)

In [ ]:
5*ayr**4*SPLC**5/(64*NWTG**3*(1e9*MSOL)**3*.25) / YR

In [ ]:
_AGE_UNIVERSE_GYR

In [ ]:
ztmp=np.logspace(-3,1,100)
Mmin100=(32624.292152148697/((cosmo.age(0).value-cosmo.age(ztmp).value)*1e9))**(3/5) / 100**(-8/5) * (1+ztmp)**(8/5)
Mmin20=(32624.292152148697/((cosmo.age(0).value-cosmo.age(ztmp).value)*1e9))**(3/5) / 20**(-8/5) * (1+ztmp)**(8/5)
Mmin15=(32624.292152148697/((cosmo.age(0).value-cosmo.age(ztmp).value)*1e9))**(3/5) / 15**(-8/5) * (1+ztmp)**(8/5)
plt.xscale('log')
plt.yscale('log')
plt.ylim(8e7,8e10)
plt.xlabel(r'$z_{\rm em}(f_{\rm obs,min})$')
plt.ylabel(r'Minimum $M_{\rm mrg,GW}$ [M$_{\odot}$]')
#plt.title(r'Minimum SMBHB mass able to merge by $z=0$,'+'\n'
#          r'starting from $z_{\rm em}(f_{\rm obs,min})$, via GW emission only')
plt.plot(ztmp,Mmin15*1e9,label=r'1/(15yr)',color='k',lw=2)
plt.plot(ztmp,Mmin20*1e9,label=r'1/(20yr)',color='b',ls='--',lw=2)
plt.plot(ztmp,Mmin100*1e9,label=r'1/(100yr)',color='c',ls=':',lw=2)
plt.legend(title=r'f$_{\rm obs,min}$ = f$_{\rm em}$(1+z$_{\rm em}$)')

In [ ]:
(32624.292152148697**0.6/ 20**(-8/5)

In [ ]:
nr = 100
sh = 4
gsmff = 2
gpff = 0
tau_out = 1.0
_rch9 = 100
_alphch = -0.5
nin = None
#xcr = 1000.0
#_gwc_units = 'rg'
r9 = 1000
_alphgw = -0.25
_betagw = 0
_gwc_units = 'rg'
_dadt_rch = -10.0**7 #cm/s
_in_mod_type = 1
_calc_gwb = False
    
sam_newhard_data = (get_sam_newhard_dadt(nrads=nr, shape=sh, gsmf_flag=gsmff, gpf_flag=gpff, 
                                          tau_outer=tau_out, rch9=_rch9, alphach=_alphch,
                                          nu_in=nin, rgw9=r9, alphagw=_alphgw, betagw=_betagw,
                                          dadt_rch=_dadt_rch, gwc_units=_gwc_units, in_mod_type=_in_mod_type,
                                          nreals=nr, calc_gwb=_calc_gwb),)

print(len(sam_newhard_data))
print(len(sam_newhard_data[0]))
print(len(sam_newhard_data[0][-1]))
#calc_and_plot_dadt(sam_newhard_data, distance_units='rg',fixedTime='outer')
#calc_and_plot_dadt(sam_newhard_data, distance_units='pc',fixedTime='outer')
if _calc_gwb:
    gwb = sam_newhard_data[0][-1]
    hc_ss, hc_bg, sspar, bgpar = gwb
    print(f"{hc_ss.min()} {hc_ss.max()}")
    print(f"{hc_bg.min()} {hc_bg.max()}")
    freqs, freqs_edges = utils.pta_freqs()
    print(f"{hc_ss.shape=} {hc_bg.shape=} {sspar.shape=}, {bgpar.shape=} {freqs.shape=}")
    print(f"{freqs.shape=} {hc_bg[:,0].shape=}")
    plt.xscale('log')
    plt.yscale('log')
    fig = plt.plot(freqs, hc_bg[:,3]) #np.sqrt(hc_ss.sum(axis=2)**2 + hc_bg**2))

calc_and_plot_dadt(sam_newhard_data, fixedTime='outer', extra_panels=True)
calc_and_plot_dadt(sam_newhard_data, fixedTime='outer', distance_units='rg', extra_panels=True)

In [ ]:
### HERE
nrad = 100
sh = 4
gsmff = 2
gpff = 0
tau_out = 1.0
_rch9 = 1.0
_dadt_rch = None #-10.0**7 #cm/s
_gwc_units = 'rg'
_in_mod_type = 0
_calc_gwb = False
nreal = 10

# ---- set base model here:
version='const-tin'
par_type='nu0'
# -------------------------

if version=='const-tin':
    _alphch = -0.5
    _alphgw = -0.25
    _betagw = 0.25
elif version=='const-rgw':
    _alphch = 0.0
    _alphgw = 0.0
    _betagw = 0.0
if par_type=='nu0':
    nin = 0
    rgw9 = 10.0**2.5
elif par_type=='star':
    nin = -1
    rgw9 = 10.0**3.5
elif par_type=='gas':
    nin = 2
    rgw9 = 10.0**2

# ---- set alt params here:
#rgw9=1e5
#_alphgw=-0.5
_alphch=-2/3
#_rch9=10**-0.5
#nin=-0.5
#_rch9=10

# -------------------------

freqs, freqs_edges = utils.pta_freqs()

sam_newhard_data = (get_sam_newhard_dadt(nrads=nrad, shape=sh, gsmf_flag=gsmff, gpf_flag=gpff, 
                                         tau_outer=tau_out, rch9=_rch9, alphach=_alphch,
                                         nu_in=nin, rgw9=rgw9, alphagw=_alphgw, betagw=_betagw,
                                         dadt_rch=_dadt_rch, gwc_units=_gwc_units, in_mod_type=_in_mod_type,
                                         #mtot_range=None, mrat_range=None,
                                         mtot_range=(1.0e4*MSOL, 1.0e12*MSOL), 
                                         #mtot_range=(1.0e7*MSOL, 1.0e10*MSOL), 
                                         mrat_range=(1e-3, 1.0),
                                         nreals=nreal, calc_gwb=_calc_gwb),)

#print(len(sam_newhard_data))
#print(len(sam_newhard_data[0]))
#print(len(sam_newhard_data[0][-1]))
#calc_and_plot_dadt(sam_newhard_data, distance_units='rg',fixedTime='outer')
#calc_and_plot_dadt(sam_newhard_data, distance_units='pc',fixedTime='outer')
if _calc_gwb:
    gwb = sam_newhard_data[0][-1]
    hc_ss, hc_bg, sspar, bgpar = gwb
    hctot = np.sqrt( np.sum(hc_ss**2,axis=2) + hc_bg**2 )
    print(f"{hc_ss.shape=} {hc_bg.shape=} {sspar.shape=}, {bgpar.shape=} {freqs.shape=}")
    print(f"{freqs.shape=} {hc_bg[:,0].shape=}")
    plt.xscale('log')
    plt.yscale('log')
    plt.grid()
    #fig = plt.plot(freqs, hc_bg[:,3]) #np.sqrt(hc_ss.sum(axis=2)**2 + hc_bg**2))
    fig = plt.plot(freqs, hctot) #np.sqrt(hc_ss.sum(axis=2)**2 + hc_bg**2))

print_extra_output=False
if print_extra_output:
    print(f"{freqs_edges.min()=}, {freqs_edges.max()=}")
    for Tobs in [1,15,20,30,50,100]:
        print(f"{Tobs=:.4g}yr, fmin={1/(Tobs*YR):.4g}")
        for MM in [4,6,9,12]:
            maxa = sepa_emit(10**MM*MSOL,1/(Tobs*YR))
            maxa_rg = maxa / utils.gravitational_radius(10**MM*MSOL)
            rgw = rgw9 * (10.0**MM/1e9)**(_alphgw) ## rgw9 in Rg, ignoring q<1 for the moment
            print(f"M=1e{MM}: max sepa,em = {maxa/PC:.4g}pc = {maxa_rg:.4}Rg. {rgw=:.4}Rg.")
#    #print(f"M=1e9: max sepa_emit={sepa_emit(1e9*MSOL,1/(Tobs*YR))/PC:.4g}")
#    #print(f"M=1e6: max sepa_emit={sepa_emit(1e6*MSOL,1/(Tobs*YR))/PC:.4g}")
#    #print(f"M=1e4: max sepa_emit={sepa_emit(1e4*MSOL,1/(Tobs*YR))/PC:.4g}")

    for MM in np.arange(4,12,1):
        ## converting rgw9 to pc, ignoring q<1 for the moment
        rch = _rch9*PC * (10.0**MM/1e9)**(_alphch+1)
        rgw = rgw9 * utils.gravitational_radius(1e9*MSOL) * (10.0**MM/1e9)**(_alphgw+1) 
        risco = utils.rad_isco(10**MM*MSOL)
        fem_rch = freq_emit(10**MM*MSOL, rch)
        fem_rgw = freq_emit(10**MM*MSOL, rgw)
        fem_risco = freq_emit(10**MM*MSOL, risco)
        print(f"M=1e{MM}: {rch/PC=:.2g} {fem_rch=:.2g} {rgw/PC=:.2g} {fem_rgw=:.2g} {risco/PC=:.2g} {fem_risco=:.2g}")

#print(np.linspace(4,12,6))
#calc_and_plot_dadt(sam_newhard_data, fixedTime='outer', extra_panels=True,
#                   fobs_min=0.9e-9, max_to_plot=4)
calc_and_plot_dadt(sam_newhard_data, fixedTime='outer', distance_units='rg', extra_panels=False,
                   fobs_min=0.9e-9, max_to_plot=4)
#calc_and_plot_dadt(sam_newhard_data, fixedTime='outer', distance_units='forb', extra_panels=True,
#                   fobs_min=0.9e-9, max_to_plot=4)

In [ ]:
### HERE
nrad = 100
sh = 10
gsmff = 2
gpff = 0
tau_out = 1.0
_rch9 = 1.0
_dadt_rch = None #-10.0**7 #cm/s
_in_mod_type = 0
_calc_gwb = False
nreal = 10

# ---- set base model here:
version='const-tin'
par_type='star'
# -------------------------

if version=='const-tin':
    _alphch = -0.5
    _alphgw = -0.25
    _betagw = 0.25
elif version=='const-rgw':
    _alphch = 0.0
    _alphgw = 0.0
    _betagw = 0.0
if par_type=='nu0':
    nin = 0
    rgw9 = 10.0**2.5
elif par_type=='star':
    nin = -1
    rgw9 = 10.0**3.5
elif par_type=='gas':
    nin = 2
    rgw9 = 10.0**2

# ---- set alt params here:
#rgw9=10**4.1
#_alphgw=-0.25
#_betagw=+0.5
_alphch=-2/3
#_rch9=10**-0.5
#nin=-4
#_rch9=10

# -------------------------

freqs, freqs_edges = utils.pta_freqs()

sam_newhard_data = (get_sam_newhard_dadt(nrads=nrad, shape=sh, gsmf_flag=gsmff, gpf_flag=gpff, 
                                         tau_outer=tau_out, rch9=_rch9, alphach=_alphch,
                                         nu_in=nin, rgw9=rgw9, alphagw=_alphgw, betagw=_betagw,
                                         dadt_rch=_dadt_rch, gwc_units=_gwc_units, in_mod_type=_in_mod_type,
                                         #mtot_range=None, mrat_range=None,
                                         mtot_range=(1.0e4*MSOL, 1.0e12*MSOL), 
                                         #mtot_range=(1.0e7*MSOL, 1.0e10*MSOL), 
                                         mrat_range=(1e-3, 1.0),
                                         nreals=nreal, calc_gwb=_calc_gwb),)

#print(len(sam_newhard_data))
#print(len(sam_newhard_data[0]))
#print(len(sam_newhard_data[0][-1]))
#calc_and_plot_dadt(sam_newhard_data, distance_units='rg',fixedTime='outer')
#calc_and_plot_dadt(sam_newhard_data, distance_units='pc',fixedTime='outer')
if _calc_gwb:
    gwb = sam_newhard_data[0][-1]
    hc_ss, hc_bg, sspar, bgpar = gwb
    hctot = np.sqrt( np.sum(hc_ss**2,axis=2) + hc_bg**2 )
    print(f"{hc_ss.shape=} {hc_bg.shape=} {sspar.shape=}, {bgpar.shape=} {freqs.shape=}")
    print(f"{freqs.shape=} {hc_bg[:,0].shape=}")
    plt.xscale('log')
    plt.yscale('log')
    plt.grid()
    #fig = plt.plot(freqs, hc_bg[:,3]) #np.sqrt(hc_ss.sum(axis=2)**2 + hc_bg**2))
    fig = plt.plot(freqs, hctot) #np.sqrt(hc_ss.sum(axis=2)**2 + hc_bg**2))

print_extra_output=False
if print_extra_output:
    print(f"{freqs_edges.min()=}, {freqs_edges.max()=}")
    for Tobs in [1,15,20,30,50,100]:
        print(f"{Tobs=:.4g}yr, fmin={1/(Tobs*YR):.4g}")
        for MM in [4,6,9,12]:
            maxa = sepa_emit(10**MM*MSOL,1/(Tobs*YR))
            maxa_rg = maxa / utils.gravitational_radius(10**MM*MSOL)
            rgw = rgw9 * (10.0**MM/1e9)**(_alphgw) ## rgw9 in Rg, ignoring q<1 for the moment
            print(f"M=1e{MM}: max sepa,em = {maxa/PC:.4g}pc = {maxa_rg:.4}Rg. {rgw=:.4}Rg.")
#    #print(f"M=1e9: max sepa_emit={sepa_emit(1e9*MSOL,1/(Tobs*YR))/PC:.4g}")
#    #print(f"M=1e6: max sepa_emit={sepa_emit(1e6*MSOL,1/(Tobs*YR))/PC:.4g}")
#    #print(f"M=1e4: max sepa_emit={sepa_emit(1e4*MSOL,1/(Tobs*YR))/PC:.4g}")

    for MM in np.arange(4,12,1):
        ## converting rgw9 to pc, ignoring q<1 for the moment
        rch = _rch9*PC * (10.0**MM/1e9)**(_alphch+1)
        rgw = rgw9 * utils.gravitational_radius(1e9*MSOL) * (10.0**MM/1e9)**(_alphgw+1) 
        risco = utils.rad_isco(10**MM*MSOL)
        fem_rch = freq_emit(10**MM*MSOL, rch)
        fem_rgw = freq_emit(10**MM*MSOL, rgw)
        fem_risco = freq_emit(10**MM*MSOL, risco)
        print(f"M=1e{MM}: {rch/PC=:.2g} {fem_rch=:.2g} {rgw/PC=:.2g} {fem_rgw=:.2g} {risco/PC=:.2g} {fem_risco=:.2g}")

print(np.linspace(4,12,6))
calc_and_plot_dadt(sam_newhard_data, fixedTime='outer', extra_panels=True,
                   fobs_min=0.9e-9, max_to_plot=4)
calc_and_plot_dadt(sam_newhard_data, fixedTime='outer', distance_units='rg', extra_panels=True,
                   fobs_min=0.9e-9, max_to_plot=10)
calc_and_plot_dadt(sam_newhard_data, fixedTime='outer', distance_units='forb', extra_panels=True,
                   fobs_min=0.9e-9, max_to_plot=4)

In [ ]:
nrad = 100
sh = 100
gsmff = 2
gpff = 0
tau_out = 1.0
_rch9 = 1.0
nin = 0
rgw9 = 10.0**3
#nin = -1
#rgw9 = 10.0**3.5
#nin = 2
#rgw9 = 10.0**2
#_alphch = 0
#_alphgw = 0
#_betagw = 0
_alphch = -0.5
_alphgw = -0.625
_betagw = 0.25
_gwc_units = 'rg'
_dadt_rch = None #-10.0**7 #cm/s
_in_mod_type = 0
_calc_gwb = True
nreal = 10

freqs, freqs_edges = utils.pta_freqs()

hctot=[]
for i,mmin in enumerate(np.arange(4,12)):
    sam_newhard_data = (get_sam_newhard_dadt(nrads=nrad, shape=sh, gsmf_flag=gsmff, gpf_flag=gpff, 
                                             tau_outer=tau_out, rch9=_rch9, alphach=_alphch,
                                             nu_in=nin, rgw9=rgw9, alphagw=_alphgw, betagw=_betagw,
                                             dadt_rch=_dadt_rch, gwc_units=_gwc_units, in_mod_type=_in_mod_type,
                                             #mtot_range=None, mrat_range=None,
                                             #mtot_range=(1.0e4*MSOL, 1.0e12*MSOL), 
                                             mtot_range=((10.0**mmin)*MSOL, (10.0**(mmin+1))*MSOL), 
                                             mrat_range=(1e-3, 1.0),
                                             nreals=nreal, calc_gwb=_calc_gwb),)
    print(f"{mmin=}")
    if _calc_gwb:
        gwb = sam_newhard_data[0][-1]
        hc_ss, hc_bg, sspar, bgpar = gwb
        hctot = np.sqrt( np.sum(hc_ss**2,axis=2) + hc_bg**2 )    
        print(f"{hc_ss.shape=} {hc_bg.shape=} {sspar.shape=}, {bgpar.shape=} {freqs.shape=}")
        print(f"{freqs.shape=} {hc_bg[:,0].shape=}")
        plt.xscale('log')
        plt.yscale('log')
        plt.grid()
        plt.plot(freqs, hctot,label=f"m={mmin}-{mmin+1}") #np.sqrt(hc_ss.sum(axis=2)**2 + hc_bg**2))
        #fig = plt.plot(freqs, hc_bg[:,3]) #np.sqrt(hc_ss.sum(axis=2)**2 + hc_bg**2))
plt.legend()    

In [ ]:
# alpha_char = -0.5
for x in [0.001222,0.001601,0.003572]:
    print(x*1e-5**(-0.5))
    print(x*1e-5**(-0.5)*PC/utils.gravitational_radius(1.0e9*MSOL))


In [ ]:
# alpha_char = 0
for x in [0.001222,0.001601,0.003572]:
    print(x*PC/utils.gravitational_radius(1.0e4*MSOL))
    print(x*PC/utils.gravitational_radius(1.0e4*MSOL)* utils.gravitational_radius(1.0e9*MSOL)/PC)

print("")
for x in [0.00567,0.00743,0.01658]:
    print(x*PC/utils.gravitational_radius(1.0e6*MSOL))
    print(x*PC/utils.gravitational_radius(1.0e6*MSOL)* utils.gravitational_radius(1.0e9*MSOL)/PC)


In [ ]:
1e8**.25

In [ ]:
0.166*PC/utils.gravitational_radius(1.0e9*MSOL)

In [ ]:
utils.gravitational_radius(1.0e9*MSOL)*2.553e6/PC

In [ ]:
nr = 100
sh = 4
gsmff = 2
gpff = 0
tau_out = 1.0
_rch9 = 100.0
nin = None
r9 = 1000
alph = -1.0
_gwc_units = 'rg'
_dadt_rc = -10.0**4 #cm/s
_in_mod_type = 1
_calc_gwb = False
    
sam_newhard_data = (get_sam_newhard_dadt(nrads=nr, shape=sh, gsmf_flag=gsmff, gpf_flag=gpff, 
                                          tau_outer=tau_out, rch9=_rch9, nu_in=nin, 
                                          rgw9=r9, alphagw=alph,
                                          dadt_rc=_dadt_rc, gwc_units=_gwc_units, in_mod_type=_in_mod_type,
                                          nreals=nr, calc_gwb=_calc_gwb),)

print(len(sam_newhard_data))
print(len(sam_newhard_data[0]))
print(len(sam_newhard_data[0][-1]))
#calc_and_plot_dadt(sam_newhard_data, distance_units='rg',fixedTime='outer')
#calc_and_plot_dadt(sam_newhard_data, distance_units='pc',fixedTime='outer')
if _calc_gwb:
    gwb = sam_newhard_data[0][-1]
    hc_ss, hc_bg, sspar, bgpar = gwb
    freqs, freqs_edges = utils.pta_freqs()
    print(f"{hc_ss.shape=} {hc_bg.shape=} {sspar.shape=}, {bgpar.shape=} {freqs.shape=}")
    print(f"{freqs[:-1].shape=} {hc_bg[:,0].shape=}")
    plt.xscale('log')
    plt.yscale('log')
    fig = plt.plot(freqs[:-1], hc_bg[:,3]) #np.sqrt(hc_ss.sum(axis=2)**2 + hc_bg**2))

calc_and_plot_dadt(sam_newhard_data, fixedTime='outer', extra_panels=True)
calc_and_plot_dadt(sam_newhard_data, fixedTime='outer', distance_units='rg', extra_panels=True)

In [ ]:
nr = 100
sh = 4
gsmff = 2
gpff = 0
tau_out = 1.0
rch = 10.0
nin = None
r9 = 10.0**3.5
alph = -0.25
_gwc_units = 'rg'
_dadt_rc = -10.0**6 #cm/s
_in_mod_type = 1
_calc_gwb = False
    
sam_newhard_data = (get_sam_newhard_dadt(nrads=nr, shape=sh, gsmf_flag=gsmff, gpf_flag=gpff, 
                                          tau_outer=tau_out, rc=rch, nu_in=nin, 
                                          rgw9=r9, alphagw=alph,
                                          dadt_rc=_dadt_rc, gwc_units=_gwc_units, in_mod_type=_in_mod_type,
                                          nreals=nr, calc_gwb=_calc_gwb),)

print(len(sam_newhard_data))
print(len(sam_newhard_data[0]))
print(len(sam_newhard_data[0][-1]))
#calc_and_plot_dadt(sam_newhard_data, distance_units='rg',fixedTime='outer')
#calc_and_plot_dadt(sam_newhard_data, distance_units='pc',fixedTime='outer')
if _calc_gwb:
    gwb = sam_newhard_data[0][-1]
    hc_ss, hc_bg, sspar, bgpar = gwb
    freqs, freqs_edges = utils.pta_freqs()
    print(f"{hc_ss.shape=} {hc_bg.shape=} {sspar.shape=}, {bgpar.shape=} {freqs.shape=}")
    print(f"{freqs[:-1].shape=} {hc_bg[:,0].shape=}")
    plt.xscale('log')
    plt.yscale('log')
    fig = plt.plot(freqs[:-1], hc_bg[:,3]) #np.sqrt(hc_ss.sum(axis=2)**2 + hc_bg**2))

calc_and_plot_dadt(sam_newhard_data, fixedTime='outer', extra_panels=True)
calc_and_plot_dadt(sam_newhard_data, fixedTime='outer', distance_units='rg', extra_panels=True)

In [ ]:
nr = 100
sh = 4
gsmff = 2
gpff = 0
tau_out = 1.0
_rch9 = 10.0
r9 = 1000
alph = -0.25
_gwc_units = 'rg'
_in_mod_type = 0
nin = -1.0
_calc_gwb = False
    
sam_newhard_data = (get_sam_newhard_dadt(nrads=nr, shape=sh, gsmf_flag=gsmff, gpf_flag=gpff, 
                                          tau_outer=tau_out, rch9=_rch9, nu_in=nin, 
                                          rgw9=r9, alphagw=alph,
                                          dadt_rc=_dadt_rc, gwc_units=_gwc_units, in_mod_type=_in_mod_type,
                                          nreals=nr, calc_gwb=_calc_gwb),)

print(len(sam_newhard_data))
print(len(sam_newhard_data[0]))
print(len(sam_newhard_data[0][-1]))
#calc_and_plot_dadt(sam_newhard_data, distance_units='rg',fixedTime='outer')
#calc_and_plot_dadt(sam_newhard_data, distance_units='pc',fixedTime='outer')
if _calc_gwb:
    gwb = sam_newhard_data[0][-1]
    hc_ss, hc_bg, sspar, bgpar = gwb
    freqs, freqs_edges = utils.pta_freqs()
    print(f"{hc_ss.shape=} {hc_bg.shape=} {sspar.shape=}, {bgpar.shape=} {freqs.shape=}")
    print(f"{freqs[:-1].shape=} {hc_bg[:,0].shape=}")
    plt.xscale('log')
    plt.yscale('log')
    fig = plt.plot(freqs[:-1], hc_bg[:,3]) #np.sqrt(hc_ss.sum(axis=2)**2 + hc_bg**2))

calc_and_plot_dadt(sam_newhard_data, fixedTime='outer', extra_panels=True)
calc_and_plot_dadt(sam_newhard_data, fixedTime='outer', distance_units='rg', extra_panels=True)

#lg(risco) < lg(agw9) + (alpha+1)*lg(m9)
#agw9 * (M/M9)^(alpha+1) < 0.5 rchar
#(alpha+1)*lg(m9) + lg(agw9) = lg(0.5*rchar)
# vmax>0:
(1-vin) > (lgadotrch-lgadotgw)/(lgrch-lgrgw)
(1-vin)*(lgrch-lgrgw) >  lgadotrch - lgadotgw
(1-vmax)*(lgrch-lgrgw) + lgadotgw >  lgadotrch
# vmax < 0:
(1-vin) > (lgadotrch-lgadotgw)/(lgrch-lgrgw)
(1-vmax) = (lgadotrch_vmax - lgadotgw)/(lgrch-lgrgw)


# 
(1-/+np.abs(vin))*lgrdiff + lgadotgw = lgadotrch 
if vin > 0:
    # corresponds to adotrch < adotgw
    # larger adotrch means vin closer to 0
    # smaller adotrch means vin more positive
    # thus, implementing a vmax>0 gives gives a min value of adot(rchar) in this case
    (1-vin)*lgrdiff + lgadotgw = lgadotrch 
    (1-vmax)*lgrdiff + lgadotgw = lgadotrch_vmax
    # if vin>vmax, lgadotrch is smaller. to avoid this, need to put lower bound on lgadotrch     
elif vin < 0:
    # corresponds to adotrch > adotgw
    # larger adotrch means vin more negative
    # smaller adotrch means vin closer to 0
    # thus, implementing a vmax<0 gives a max value of adot(rchar) in this case
    (1-vin)*lgrdiff + lgadotgw = lgadotrch
    (1-vmax)*lgrdiff + lgadotgw = lgadotrch_vmax
    # if np.abs(vin)>vmax, lgadotrch is bigger. to avoid this, need to cap lgadotrch 
elif vmax = 0:
    # special case where adots are equal

so the 1-np.abs(vmax) relation gives the min, and 1+np.abs(vmax) gives the max


In [ ]:
_in_mod_type = 1
nr = 100
sh = 4
gsmff = 2
gpff = 0
tau_out = 1.0
nin = None
_gwc_units = 'rg'
rch = 10.0
snd = ()
for rch in np.logspace(2,6,3):
    for r9 in np.logspace(1,5,3): 
        for alph in np.arange(-1.5,0.5,0.5):
            for adotrc in -1.0*np.logspace(5,9,3):
                print(f"\n{r9=}, {alph=}, {adotrc=}\n")

                #nu_in, rgw_crit, dadt_gw_crit = calc_model1_pars(sam, hard)

                snd += (get_sam_newhard_dadt(nrads=nr, shape=sh, gsmf_flag=gsmff, gpf_flag=gpff, 
                                            tau_outer=tau_out, rc=rch, nu_in=nin, 
                                            rgw9=r9, alphagw=alph,
                                            dadt_rc=adotrc, gwc_units=_gwc_units, in_mod_type=_in_mod_type,
                                            nreals=nr, calc_gwb=False),)
print(len(snd))

In [ ]:
nr = 100
sh = 4
gsmff = 2
gpff = 0
tau_out = 10.0
rch = 10.0
#nin = -0.5
nin = None
r9 = 10000.0
_gwc_units = 'rg'
alph = -0.5
#_dadt_rc=None
_dadt_rc = -1.0e7 #cm/s
_in_mod_type = 1
    

sam_newhard_data = (get_sam_newhard_dadt(nrads=nr, shape=sh, gsmf_flag=gsmff, gpf_flag=gpff, 
                                          tau_outer=tau_out, rc=rch, nu_in=nin, 
                                          rgw9=r9, alphagw=alph,
                                          dadt_rc=_dadt_rc, gwc_units=_gwc_units, in_mod_type=_in_mod_type,
                                          nreals=nr, calc_gwb=False),)

#print(len(sam_newhard_data))
print(len(sam_newhard_data[0]))
sam, hard, rads, dadt, agw, rzch, rzf = sam_newhard_data[0]
print(f"PARAMS: nu_in={hard._nu_inner}, r9={hard._r_gw_crit_9}, alph={hard._alpha_gw_crit}")
nu_in, rgw_crit, dadt_rgw_crit = calc_model1_pars(sam, hard)
print(f"{len(sam.mtot)=}")
print(f"{nu_in.shape=}, {nu_in.min()=}, {nu_in.max()}")
#for i in range(len(sam.mtot)):
#    for j in range(len(sam.mrat)):
#        print(f"{sam.mtot[i]/MSOL=} {sam.mrat[j]=} {nu_in[i][j]=}")
#calc_and_plot_dadt(sam_newhard_data, fixedTime='outer', extra_panels=True)
calc_and_plot_dadt(sam_newhard_data, fixedTime='outer', distance_units='rg', extra_panels=True)

plot_param_space(sam, hard)

In [ ]:
nr = 100
sh = 10
gsmff = 2
gpff = 0
ai = 1.0e4
rch=10.0
t=1.0
nin=0.0
nout=2.5
sam, hard, rads, dadt = get_sam_dadt(nrads=nr, shape=sh, gsmf_flag=gsmff, gpf_flag=gpff, 
                                     ainit=ai, tau=t, rc=rch, nu_in=nin, nu_out=nout)

print(f"{hard._norm.shape=}, {sam.mtot.shape=}, {sam.mrat.shape=}")
#print(hard._norm, sam.mtot/MSOL, sam.mrat)
#agw_test = calc_aGW_for_Fixed_Time_2PL(hard._norm, hard._gamma_inner, hard._rchar, sam.mtot, sam.mrat)
agw_test = calc_aGW_for_Fixed_Time_2PL(hard, sam)
print(f"{agw_test.shape=} {agw_test.min()=} {agw_test.max()=}")
#print(agw_test/PC)

times_evo = -utils.trapz_loglog(-1.0 / dadt[:,:,0,:], rads[:,:,0,:], axis=2, cumsum=True)
print(f"{dadt.shape=}, {rads.shape=}, {times_evo.shape=}")
tt = times_evo/GYR
#tt = times_evo[-1, :]/GYR
fig, ax = plot.figax(scale='log')
ax.xaxis.set_inverted(True)
print(utils.stats(tt))
#kale.dist1d(tt, density=True)
for m in range(rads.shape[0]):
    for q in range(rads.shape[1]):
        plt.plot(rads[m,q,0,:-1]/PC,tt[m,q,:])
plt.show()

In [ ]:
nr = 100
sh = 2
gsmff = 2
gpff = 0
ai = 1.0e4
#rch=10.0
#t=1.0
#nin=0.0
#nout=2.5
cm = plot._get_cmap('tab20')
colors = cm(np.linspace(0, 1, 1000))

fig,(ax0,ax1,ax2) = plt.subplots(figsize=(10,3),ncols=3)
mod_count = 0 
sparse_mod_count = 0 
#for t in np.array([0.1,10]):
#    for no in np.array([0.0,2.5]):
#        for ni in np.array([-1,0.5]):
#            for rch in np.array([10,100.0]):
bad_t = []
bad_nu = []
for t in np.logspace(-2.5,1.5,10):
    for no in np.array([2.5]):
        for ni in np.arange(-2,2,0.25):
            for rch in np.array([100.0]):
                sam, hard, rads, dadt = get_sam_dadt(nrads=nr, shape=sh, gsmf_flag=gsmff, gpf_flag=gpff, 
                                                     ainit=ai, tau=t, rc=rch, nu_in=ni, nu_out=no);

                if np.abs(dadt).max() > SPLC:
                    print(f"dadt>c: {t=} {no=} {ni=} {rch}")
                    bad_t.append(t)
                    bad_nu.append(ni)

                if mod_count % 10 == 0:
                    agw_test = calc_aGW_for_Fixed_Time_2PL(hard, sam)
                    _color = colors[sparse_mod_count]
                    #print(f"{hard._norm.shape=}, {sam.mtot.shape=}, {sam.mrat.shape=}")
                    #print(f"{agw_test.shape=} {agw_test.min()=} {agw_test.max()=}")

                    mt, mr = np.broadcast_arrays(
                        sam.mtot[:, np.newaxis],
                        sam.mrat[np.newaxis, :]
                    )
                    rg = NWTG * mt / SPLC**2
                    for j in range(sam.mrat.size):
                        #print(f"mt={np.log10(mt[:,j]/MSOL)} {np.log10(agw_test[:,j]/( NWTG * sam.mtot / SPLC**2))}")
                        ax0.plot(np.log10(mt[:,j]/MSOL), np.log10(agw_test[:,j]/rg[:,j]),
                                 lw=1.25-0.5*j,color=_color)
                        #plt.plot(np.log10(mr[j,:]), np.log10(agw_test[j,:]/rg),lw=1.25-0.25*j,alpha=0.5)

                    for k in range(sam.mtot.size):
                        #print(f"mr={np.log10(mr[k,:]/MSOL)} {np.log10(agw_test[k,:]/( NWTG * sam.mtot[k] / SPLC**2))}")
                        ax1.plot(np.log10(mr[k,:]), np.log10(agw_test[k,:]/rg[k,:]),
                                 lw=1.25-0.5*k,color=_color)

                    ax0.plot(np.log10(mt[:,0]/MSOL),-0.25*np.log10(mt[:,0]/MSOL)+3,'k:',lw=3)
                    ax1.plot(np.log10(mr[0,:]),0.25*np.log10(mr[0,:])+3,'k:',lw=3)
                    sparse_mod_count += 1
                    
                mod_count += 1

print(f"{mod_count=}")
#print(len(bad_mods),len(bad_mods[0]))
print(bad_mods)
#print(np.log10(bad_mods[:][0]))
#print(bad_mods[:][1])
#ax2.set_xlim(-2.5,1.5)
#ax2.set_ylim(-2,2)
#for tt,nn in zip(bad_t,bad_nu):
print(len(bad_t),len(bad_nu))
ax2.set_xlabel(r'$\tau$')
ax2.set_ylabel(r'$\nu_{\rm inner}$')
ax2.scatter(np.log10(bad_t),bad_nu)
#print(mt,mr,agw_test)
#calc_and_plot_dadt(sam_data, distance_units='rg',fixedTime='total')



In [ ]:
a = np.arange(10)
b = np.logspace(-2,2,10)
f = []
for i in range(10):
    f.append([a[i],b[i]])
for tt, nn in zip([a],[b]):
    print(tt,nn)
print(x,y in zip(f))

In [ ]:
print(agw_test.shape, sam.mtot.shape, sam.mrat.shape)
plt.xscale('log')
plt.yscale('log')
for i in range(sam.mrat.size):
    plt.scatter(sam.mtot/MSOL,agw_test[:,i]/PC)
    plt.plot(sam.mtot/MSOL,agw_test[:,i]/PC)

In [ ]:
alphaGW = np.arange(0,1.5,0.01)
dadt_index = 3*(1-alphaGW)
tscale_index = 4*alphaGW - 3
plt.plot(alphaGW, alphaGW,label='aGW')
plt.plot(alphaGW, dadt_index,label='dadt_GW')
plt.plot(alphaGW, tscale_index,label='tscale_GW')
plt.plot([0.75,0.75],[-3,3],'k:')
plt.plot([6/7,6/7],[-3,3],'k:')
plt.plot([1,1],[-3,3],'k:')
plt.plot([0,1.5],[0,0],'k:')
plt.plot([0,1.5],[1,1],'k:')
plt.legend()

In [ ]:
#calc_and_plot_dadt(nrads=100, shape=10, gsmf_flag=2, gpf_flag=0, tau=1.0, 
#                   ainit=1.0e4, rc=100.0, nu_in=-1.0, nu_out=+2.5)

nr = 100
sh = 4
gsmff = 1
gpff = 1
ai = 1.0e4
sam_data = []
_calc_gwb = False
#for t in [0.1,1.0,10.0]:
for t in [0.1]:
    for rch in [100]:
        #for nin in [-0.5]:
        #for nin in [-2.0, -1.0, -0.5, -0.1]:
        for nin in [-0.75]:
            for nout in [2.5]:
                sam_data = sam_data + [(get_sam_dadt(nrads=nr, shape=sh, gsmf_flag=gsmff, 
                                                     gpf_flag=gpff, ainit=ai, 
                                                     tau=t, rc=rch, nu_in=nin, nu_out=nout,
                                                     nreals=nr, calc_gwb=_calc_gwb))]

print(len(sam_data))
print(len(sam_data[0]))
##calc_and_plot_dadt(sam_data)
calc_and_plot_dadt(sam_data, distance_units='rg',fixedTime='total')
calc_and_plot_dadt(sam_data, distance_units='pc',fixedTime='total')
#calc_and_plot_dadt(sam_newhard_data, distance_units='pc',fixedTime='outer')
if _calc_gwb:
    gwb = sam_data[0][-1]
    hc_ss, hc_bg, sspar, bgpar = gwb
    freqs, freqs_edges = utils.pta_freqs()
    print(f"{hc_ss.shape=} {hc_bg.shape=} {freqs.shape=}")
    fig = plot.plot_gwb(freqs, hc_bg) #np.sqrt(hc_ss.sum(axis=2)**2 + hc_bg**2))


In [ ]:
print(10**2.5 * NWTG * 1e9*1.989e33/SPLC**2 / PC)
print(10**3.75 * NWTG * 1e4*1.989e33/SPLC**2 / PC)
print(10 * NWTG * 1e12*1.989e33/SPLC**2 / PC)

In [ ]:
print( PC / (NWTG * 1e9*1.989e33/SPLC**2 ))

In [ ]:
print(np.log10(10**2.5 *1e-5**-0.25))
print(np.log10(10**2.5 *1e3**-0.25 *(1e-3/(1.001**2))**.25))

In [ ]:
1.0 * 1e-5**(1/3)

In [ ]:
1.0 * 1e3**(1/3)

In [ ]:
1.0 * (10**9.3/1e9)**(1/3)

In [ ]:
def __draw_gwb(ax, xx, gwb, nsamp=10, color=None, label=None, alpha=0.25, **kwargs):
    if color is None:
        color = ax._get_lines.get_next_color()

    kw_plot = kwargs.pop('plot', {})
    kw_plot.setdefault('color', color)
    hh = __draw_med_conf(ax, xx, gwb, plot=kw_plot, label=label, **kwargs)
    if (nsamp is not None) and (nsamp > 0):
        nsamp_max = gwb.shape[1]
        idx = np.random.choice(nsamp_max, np.min([nsamp, nsamp_max]), replace=False)
        for ii in idx:
            ax.plot(xx, gwb[:, ii], color=color, alpha=alpha, lw=1.0, ls='-')

    return hh
    
def __draw_med_conf(ax, xx, vals, fracs=[0.50, 0.90], weights=None, plot={}, 
                    fill={}, filter=False, label=None, lw=1.0, ls='-'):
    #plot.setdefault('alpha', 0.75)
    #fill.setdefault('alpha', 0.2)
    plot.setdefault('alpha', 0.75)
    fill.setdefault('alpha', 0.1)
    percs = np.atleast_1d(fracs)
    assert np.all((0.0 <= percs) & (percs <= 1.0))

    # center the target percentages into pairs around 50%, e.g.  68 ==> [16,84]
    inter_percs = [[0.5-pp/2, 0.5+pp/2] for pp in percs]
    # Add the median value (50%)
    inter_percs = [0.5, ] + np.concatenate(inter_percs).tolist()
    # Get percentiles; they go along the last axis
    if filter:
        rv = [
            kale.utils.quantiles(vv[vv > 0.0], percs=inter_percs, weights=weights)
            for vv in vals
        ]
        rv = np.asarray(rv)
    else:
        rv = kale.utils.quantiles(vals, percs=inter_percs, weights=weights, axis=-1)

    med, *conf = rv.T
    # plot median
    hh, = ax.plot(xx, med, **plot, lw=lw, ls=ls, label=label)

    # Reshape confidence intervals to nice plotting shape
    # 2*P, X ==> (P, 2, X)
    conf = np.array(conf).reshape(len(percs), 2, xx.size)

    kw = dict(color=hh.get_color())
    kw.update(fill)
    fill = kw

    # plot each confidence interval
    for lo, hi in conf:
        gg = ax.fill_between(xx, lo, hi, **fill)

    return (hh, gg)


In [ ]:

#print(sam_data[1][1]._gamma_inner)
#print(sam_data[2][1]._gamma_inner)
calc_and_plot_dadt([sam_data[1],sam_data[2]], distance_units='rg',fixedTime='total',max_to_plot=3)
calc_and_plot_dadt([sam_data[1],sam_data[2]], distance_units='pc',fixedTime='total',max_to_plot=3)
#calc_and_plot_dadt(sam_data, distance_units='pc',fixedTime='total',max_to_plot=3)

#c_arr = 4*['g','c','m','b','k']
c_arr = 4*['orangered','steelblue']
gpf_flags = [1]*len(sam_data) 

LABEL_GW_FREQUENCY_YR = r"GW Frequency $[\mathrm{yr}^{-1}]$"
LABEL_GW_FREQUENCY_HZ = r"GW Frequency $[\mathrm{Hz}]$"
LABEL_GW_FREQUENCY_NHZ = r"GW Frequency $[\mathrm{nHz}]$"
LABEL_SEPARATION_PC = r"Binary Separation $[\mathrm{pc}]$"
LABEL_CHARACTERISTIC_STRAIN = r"GW Characteristic Strain"
LABEL_HARDENING_TIME = r"Hardening Time $[\mathrm{Gyr}]$"
LABEL_CLC0 = r"$C_\ell / C_0$"

fig, ax = plot.figax(
    xlabel=LABEL_GW_FREQUENCY_YR,
    ylabel=LABEL_CHARACTERISTIC_STRAIN,
    ylim=(2.0e-17,2.0e-14)
)
freqs, freqs_edges = utils.pta_freqs()
xx = freqs * YR
#for i,s in enumerate(sam_data): #[sam_data[1],sam_data[2]]:
for i,s in enumerate([sam_data[1],sam_data[2]]):
    gwb = s[-1]
    hc_ss, hc_bg, sspar, bgpar = gwb
    hctot = np.sqrt( np.sum(hc_ss**2,axis=2) + hc_bg**2 )
    __draw_gwb(ax, xx, hctot, nsamp=0, color=c_arr[i+1], 
               label=f"nu_inner={sam_data[i+1][1]._gamma_inner}",
               lw=2, ls='-', alpha=0.05, fracs=[0.5])
ax.legend()
#for s in sam_data:
#    #if i == 0 or i == 2: continue
    
#    print(s[1]._gamma_inner)
#    calc_and_plot_dadt([s], distance_units='rg',fixedTime='total')
#    calc_and_plot_dadt([s], distance_units='pc',fixedTime='total')
    


In [ ]:
### NG15 PHENOM-LIKE MODEL

nr = 100
#sh = 100
#sh = None
sh = 4
gsmff = 1
gpff = 1
ai = 1.0e4
sam_data = []
_calc_gwb = False
#for t in [0.1,1.0,10.0]:
for t in [0.1,11]:
    for rch in [100]:
        for nin in [-0.5]:
            for nout in [2.5]:
                sam_data = sam_data + [(get_sam_dadt(nrads=nr, shape=sh, gsmf_flag=gsmff, 
                                                     gpf_flag=gpff, ainit=ai, 
                                                     tau=t, rc=rch, nu_in=nin, nu_out=nout,
                                                     nreals=nr, calc_gwb=_calc_gwb))]

print(len(sam_data))
print(len(sam_data[0]))
##calc_and_plot_dadt(sam_data)
calc_and_plot_dadt(sam_data, distance_units='rg',fixedTime='total',extra_panels=True)
calc_and_plot_dadt(sam_data, distance_units='pc',fixedTime='total',extra_panels=True)
#calc_and_plot_dadt(sam_newhard_data, distance_units='pc',fixedTime='outer')
if _calc_gwb:
    gwb = sam_data[0][-1]
    hc_ss, hc_bg, sspar, bgpar = gwb
    freqs, freqs_edges = utils.pta_freqs()
    print(f"{hc_ss.shape=} {hc_bg.shape=} {freqs.shape=}")
    fig = plot.plot_gwb(freqs, hc_bg) #np.sqrt(hc_ss.sum(axis=2)**2 + hc_bg**2))


In [ ]:
calc_and_plot_dadt(sam_data)

In [ ]:
calc_and_plot_dadt(sam_data, distance_units='rg')

In [ ]:
nr = 100
sh = 4
gsmff = 2
gpff = 0
tau_out = 1.0
rch = 100.0
nin = -0.4
r9 = 1000
_gwc_units = 'rg'
alph = -0.25
_dadt_rc = None
_in_mod_type = 0
_calc_gwb = False
    
sam_newhard_data = (get_sam_newhard_dadt(nrads=nr, shape=sh, gsmf_flag=gsmff, gpf_flag=gpff, 
                                          tau_outer=tau_out, rch9=_rch9, nu_in=nin, 
                                          rgw9=r9, alphagw=alph,
                                          dadt_rc=_dadt_rc, gwc_units=_gwc_units, in_mod_type=_in_mod_type,
                                          nreals=nr, calc_gwb=_calc_gwb),)

print(len(sam_newhard_data))
#print(len(sam_newhard_data[0]))
print(len(sam_newhard_data[0][-1]))
#calc_and_plot_dadt(sam_newhard_data, distance_units='rg',fixedTime='outer')
#calc_and_plot_dadt(sam_newhard_data, distance_units='pc',fixedTime='outer')
if _calc_gwb:
    gwb = sam_newhard_data[0][-1]
    hc_ss, hc_bg, sspar, bgpar = gwb
    freqs, freqs_edges = utils.pta_freqs()
    print(f"{hc_ss.shape=} {hc_bg.shape=} {sspar.shape=}, {bgpar.shape=} {freqs.shape=}")
    print(f"{freqs[:-1].shape=} {hc_bg[:,0].shape=}")
    plt.xscale('log')
    plt.yscale('log')
    fig = plt.plot(freqs[:-1], hc_bg[:,3]) #np.sqrt(hc_ss.sum(axis=2)**2 + hc_bg**2))

calc_and_plot_dadt(sam_newhard_data, fixedTime='outer', extra_panels=True)
calc_and_plot_dadt(sam_newhard_data, fixedTime='outer', distance_units='rg', extra_panels=True)

In [ ]:
#calc_and_plot_dadt(nrads=100, shape=10, gsmf_flag=2, gpf_flag=0, tau=1.0, 
#                   ainit=1.0e4, rc=100.0, nu_in=-1.0, nu_out=+2.5)

nrad = 100
nreal = 10
sh = 4
gsmff = 2
gpff = 0
ai = 1.0e4
sam_data = []
_calc_gwb = False
#for t in [0.1,1.0,10.0]:
t = 1.0
rch = 10
nin = -1.0
nout = 2.5

if _calc_gwb:
    sam, hard, rads, dadt, gwb = get_sam_dadt(nrads=nrad, shape=sh, gsmf_flag=gsmff, 
                                              gpf_flag=gpff, ainit=ai, 
                                              tau=t, rc=rch, nu_in=nin, nu_out=nout,
                                              nreals=nreal, calc_gwb=_calc_gwb)
    hc_ss, hc_bg, sspar, bgpar = gwb
    freqs, freqs_edges = utils.pta_freqs()
    print(f"{hc_ss.shape=} {hc_bg.shape=} {freqs.shape=}")
    hc_tot = np.sqrt( np.sum(hc_ss**2, axis=-1) + hc_bg**2 ) 
    fig = plot.plot_gwb(freqs, hc_tot)
else:
    sam, hard, rads, dadt = get_sam_dadt(nrads=nrad, shape=sh, gsmf_flag=gsmff, 
                                         gpf_flag=gpff, ainit=ai, 
                                         tau=t, rc=rch, nu_in=nin, nu_out=nout,
                                         nreals=nreal, calc_gwb=_calc_gwb)
##calc_and_plot_dadt(sam_data)
calc_and_plot_dadt(sam_data, distance_units='rg',fixedTime='total')
calc_and_plot_dadt(sam_data, distance_units='pc',fixedTime='total')
#calc_and_plot_dadt(sam_newhard_data, distance_units='pc',fixedTime='outer')


In [ ]:
nrad = 100
nreal = 100
sh = 100
gsmff = 2
gpff = 0
tau_out = 10.0
rch = 100.0
nin = -1.0
r9 = 1000.0
_gwc_units = 'rg'
#r9 = 0.05
#_gwc_units = 'pc'
alph = -0.25
_dadt_rc=None
#_dadt_rc = -1.0e7 #cm/s
_in_mod_type = 0
_calc_gwb = True

if _calc_gwb:
    #sam, newhard, rads, dadt, agw_cr, rzch, rzf, gwb 
    sam_newhard_data = (get_sam_newhard_dadt(nrads=nrad, shape=sh, gsmf_flag=gsmff, gpf_flag=gpff, 
                                             tau_outer=tau_out, rc=rch, nu_in=nin, 
                                             rgw9=r9, alphagw=alph,
                                             dadt_rc=_dadt_rc, gwc_units=_gwc_units, 
                                             in_mod_type=_in_mod_type,
                                             nreals=nreal, calc_gwb=_calc_gwb),)

    print(len(sam_newhard_data[0]))
    gwb = sam_newhard_data[0][-1]
    hc_ss, hc_bg, sspar, bgpar = gwb
    freqs, freqs_edges = utils.pta_freqs()
    print(f"{hc_ss.shape=} {hc_bg.shape=} {sspar.shape=}, {bgpar.shape=} {freqs.shape=}")
    print(f"{freqs.shape=} {hc_bg[:,0].shape=}")
    hc_tot = np.sqrt( np.sum(hc_ss**2, axis=-1) + hc_bg**2 )
    print("hc_bg=",hc_bg)
    fig = plot.plot_gwb(freqs, hc_tot)


else:
    #sam, newhard, rads, dadt, agw_cr, rzch, rzf 
    sam_newhard_data = (get_sam_newhard_dadt(nrads=nrad, shape=sh, gsmf_flag=gsmff, gpf_flag=gpff, 
                                            tau_outer=tau_out, rc=rch, nu_in=nin, 
                                            rgw9=r9, alphagw=alph,
                                            dadt_rc=_dadt_rc, gwc_units=_gwc_units, 
                                            in_mod_type=_in_mod_type,
                                            nreals=nreal, calc_gwb=_calc_gwb),)

##calc_and_plot_dadt(sam_newhard_data, distance_units='rg',fixedTime='outer')
##calc_and_plot_dadt(sam_newhard_data, distance_units='pc',fixedTime='outer')

#calc_and_plot_dadt(sam_newhard_data, fixedTime='outer', extra_panels=True)
#calc_and_plot_dadt(sam_newhard_data, fixedTime='outer', distance_units='rg', extra_panels=True)



In [ ]:
nrad = 100
nreal = 10
sh = 4
gsmff = 2
gpff = 0
tau_out = 1.0
rch = 100.0
nin = -1.0
r9 = 1000.0
_gwc_units = 'rg'
#r9 = 0.05
#_gwc_units = 'pc'
alph = -0.25
_dadt_rc=None
#_dadt_rc = -1.0e7 #cm/s
_in_mod_type = 0
_calc_gwb = True

if _calc_gwb:
    #sam, newhard, rads, dadt, agw_cr, rzch, rzf, gwb 
    sam_newhard_data = (get_sam_newhard_dadt(nrads=nrad, shape=sh, gsmf_flag=gsmff, gpf_flag=gpff, 
                                             tau_outer=tau_out, rc=rch, nu_in=nin, 
                                             rgw9=r9, alphagw=alph,
                                             dadt_rc=_dadt_rc, gwc_units=_gwc_units, 
                                             in_mod_type=_in_mod_type,
                                             nreals=nreal, calc_gwb=_calc_gwb),)

    #print(len(sam_newhard_data[0]))
    gwb = sam_newhard_data[0][-1]
    hc_ss, hc_bg, sspar, bgpar = gwb
    freqs, freqs_edges = utils.pta_freqs()
    #print(f"{hc_ss.shape=} {hc_bg.shape=} {sspar.shape=}, {bgpar.shape=} {freqs.shape=}")
    #print(f"{freqs.shape=} {hc_bg[:,0].shape=}")
    hc_tot = np.sqrt( np.sum(hc_ss**2, axis=-1) + hc_bg**2 )
    #print("hc_bg=",hc_bg)
    fig = plot.plot_gwb(freqs, hc_tot)


else:
    #sam, newhard, rads, dadt, agw_cr, rzch, rzf 
    sam_newhard_data = (get_sam_newhard_dadt(nrads=nrad, shape=sh, gsmf_flag=gsmff, gpf_flag=gpff, 
                                            tau_outer=tau_out, rc=rch, nu_in=nin, 
                                            rgw9=r9, alphagw=alph,
                                            dadt_rc=_dadt_rc, gwc_units=_gwc_units, 
                                            in_mod_type=_in_mod_type,
                                            nreals=nreal, calc_gwb=_calc_gwb),)

calc_and_plot_dadt(sam_newhard_data, fixedTime='outer', extra_panels=False)
calc_and_plot_dadt(sam_newhard_data, fixedTime='outer', distance_units='rg', extra_panels=False)

In [ ]:
nrad = 100
nreal = 10
sh = 4
gsmff = 2
gpff = 0
tau_out = 1.0
rch = 100.0
#rch = 1.0
#r9 = 1000.0
_gwc_units = 'rg'
#r9 = 0.05
#_gwc_units = 'pc'
alph = -0.25
#_dadt_rc=None
#_dadt_rc = -1.0e7 #cm/s
_in_mod_type = 1
_calc_gwb = False
sam_newhard_data = []

dadt_arr=np.array([-10.0, -10.0**2.5, -1.0e4, -10.0**5.5, -1.0e7])
alph_arr = np.array([-0.5,-0.333,-0.25,-0.1,0.0])
r9_arr = np.array([30,100,300,1000,3000])
rch_arr = np.array([1.0,10.0,100.0,1000.0])
valid_arr = np.zeros((nin_arr.size,alph_arr.size,r9_arr.size)).astype('int')
print(valid_arr.shape)
#for i,da in enumerate(dadt_arr):
for i,rch in enumerate(rch_arr):
    for j,alph in enumerate(alph_arr):
        #for k,r9 in enumerate(r9_arr):
        tmp = get_sam_newhard_dadt(nrads=nrad, shape=sh, gsmf_flag=gsmff, gpf_flag=gpff, 
                                            tau_outer=tau_out, rc=rch, 
                                            rgw9=r9, alphagw=alph,
                                            dadt_rc=_da, gwc_units=_gwc_units, 
                                            in_mod_type=_in_mod_type,
                                            nreals=nreal, calc_gwb=_calc_gwb)

        sam_newhard_data = sam_newhard_data + [(tmp)]

calc_and_plot_dadt(sam_newhard_data, fixedTime='outer', extra_panels=False)
calc_and_plot_dadt(sam_newhard_data, fixedTime='outer', distance_units='rg', extra_panels=False)



In [ ]:
nrad = 100
nreal = 10
sh = 4
gsmff = 2
gpff = 0
tau_out = 1.0
rch = 100.0
#rch = 1.0
nin = -1.0
r9 = 1000.0
_gwc_units = 'rg'
#r9 = 0.05
#_gwc_units = 'pc'
alph = -0.25
_dadt_rc=None
#_dadt_rc = -1.0e7 #cm/s
_in_mod_type = 0
_calc_gwb = False
sam_newhard_data = []

nin_arr = np.array([-1.5, -1.0, -0.5, 0.0, 0.5])
alph_arr = np.array([-0.5,-0.333,-0.25,-0.1,0.0])
r9_arr = np.array([30,100,300,1000,3000])
valid_arr = np.zeros((nin_arr.size,alph_arr.size,r9_arr.size)).astype('int')
print(valid_arr.shape)
for i,nin in enumerate(nin_arr):
    for j,alph in enumerate(alph_arr):
        for k,r9 in enumerate(r9_arr):
            tmp = get_sam_newhard_dadt(nrads=nrad, shape=sh, gsmf_flag=gsmff, gpf_flag=gpff, 
                                            tau_outer=tau_out, rc=rch, nu_in=nin, 
                                            rgw9=r9, alphagw=alph,
                                            dadt_rc=_dadt_rc, gwc_units=_gwc_units, 
                                            in_mod_type=_in_mod_type,
                                            nreals=nreal, calc_gwb=_calc_gwb)

            sam_newhard_data = sam_newhard_data + [(tmp)]

calc_and_plot_dadt(sam_newhard_data, fixedTime='outer', extra_panels=False)
calc_and_plot_dadt(sam_newhard_data, fixedTime='outer', distance_units='rg', extra_panels=False)



In [ ]:
#fig, axs = plt.subplots(2,3)
#print(valid_arr)
mtyp=['.','+','o','^','s']
col=['darkorange','m','g','b','k']
plt.xlabel('log10(r9)')
plt.ylabel('alpha')
for i,n in enumerate(nin_arr):
    for j,al in enumerate(alph_arr):
        for k,r9 in enumerate(r9_arr):
            if valid_arr[i,j,k]:
                if j==0 and k==r9_arr.size-1:
                    plt.scatter(np.log10(r9_arr[k])+0.05*i,alph_arr[j],marker=mtyp[i],color=col[i],label=f"nuin={n}")
                else:
                    plt.scatter(np.log10(r9_arr[k])+0.05*i,alph_arr[j],marker=mtyp[i],color=col[i])
plt.legend()
plt.suptitle(f'param combos for which all dadt<{SPEED_LIMIT/SPLC}c (rch={rch}pc)')

In [ ]:
print(np.vstack([nin_arr,alph_arr,r9_arr]).shape,valid_arr.shape)
data = np.vstack([nin_arr[1:2],alph_arr,r9_arr]).T
print(valid_arr.T.flatten().shape)
figure = corner.corner(data)

In [ ]:
# Set up the parameters of the problem.
ndim, nsamples = 3, 50000

# Generate some fake data.
np.random.seed(42)
data1 = np.random.randn(ndim * 4 * nsamples // 5).reshape(
    [4 * nsamples // 5, ndim]
)
data2 = 4 * np.random.rand(ndim)[None, :] + np.random.randn(
    ndim * nsamples // 5
).reshape([nsamples // 5, ndim])
data = np.vstack([data1, data2])

# Plot it.
figure = corner.corner(
    data,
    labels=[
        r"$x$",
        r"$y$",
        r"$\log \alpha$",
        r"$\Gamma \, [\mathrm{parsec}]$",
    ],
    quantiles=[0.16, 0.5, 0.84],
    show_titles=True,
    title_kwargs={"fontsize": 12},
)

print(data.shape, data.min(), data.max())
print(data1.shape, data1.min(), data1.max())
print(data2.shape, data2.min(), data2.max())

In [ ]:
nr = 100
sh = 4
gsmff = 2
gpff = 0
tau_out = 10.0
rch = 100.0
#nin = -0.5
nin = None
#r9 = 1000.0
#_gwc_units = 'rg'
r9 = 0.05
_gwc_units = 'pc'
alph = -1
#_dadt_rc=None
_dadt_rc = -1.0e7 #cm/s
_in_mod_type = 1
_calc_gwb = False
    
sam_newhard_data = (get_sam_newhard_dadt(nrads=nr, shape=sh, gsmf_flag=gsmff, gpf_flag=gpff, 
                                          tau_outer=tau_out, rc=rch, nu_in=nin, 
                                          rgw9=r9, alphagw=alph,
                                          dadt_rc=_dadt_rc, gwc_units=_gwc_units, in_mod_type=_in_mod_type,
                                          nreals=nr, calc_gwb=_calc_gwb),)

print(len(sam_newhard_data))
print(len(sam_newhard_data[0]))
print(len(sam_newhard_data[0][-1]))
#calc_and_plot_dadt(sam_newhard_data, distance_units='rg',fixedTime='outer')
#calc_and_plot_dadt(sam_newhard_data, distance_units='pc',fixedTime='outer')
if _calc_gwb:
    gwb = sam_newhard_data[0][-1]
    hc_ss, hc_bg, sspar, bgpar = gwb
    freqs, freqs_edges = utils.pta_freqs()
    print(f"{hc_ss.shape=} {hc_bg.shape=} {sspar.shape=}, {bgpar.shape=} {freqs.shape=}")
    print(f"{freqs[:-1].shape=} {hc_bg[:,0].shape=}")
    plt.xscale('log')
    plt.yscale('log')
    fig = plt.plot(freqs[:-1], hc_bg[:,3]) #np.sqrt(hc_ss.sum(axis=2)**2 + hc_bg**2))

calc_and_plot_dadt(sam_newhard_data, fixedTime='outer', extra_panels=True)
calc_and_plot_dadt(sam_newhard_data, fixedTime='outer', distance_units='rg', extra_panels=True)



In [ ]:
calc_and_plot_dadt(sam_newhard_data, fixedTime='outer',distance_units='rg')

In [ ]:
nr = 100
sh = 4
gsmff = 2
gpff = 0
ai = 1.0e4
sam_data = []
for t in [1.0]:
    for rch in [100]:
        for nin in [0.0,0.5,1.0]:
            for nout in [0.0,+2.5]:
                sam_data = sam_data + [(get_sam_dadt(nrads=nr, shape=sh, gsmf_flag=gsmff, 
                                                     gpf_flag=gpff, ainit=ai, 
                                                     tau=t, rc=rch, nu_in=nin, nu_out=nout))]

print(len(sam_data))
print(len(sam_data[0]))
calc_and_plot_dadt(sam_data,max_to_plot=20)
calc_and_plot_dadt(sam_data,distance_units='rg',max_to_plot=20)

In [ ]:
1e4*PC/(0.001*SPLC)/YR

In [ ]:
nr = 100
sh = 4
gsmff = 2
gpff = 0
ai = 1.0e4
sam_data = []
for t in [1]:
    for rch in [10,100.0]:
        for nin in [-1.0]:
            for nout in [0.0,+2.5]:
                sam_data = sam_data + [(get_sam_dadt(nrads=nr, shape=sh, gsmf_flag=gsmff, 
                                                     gpf_flag=gpff, ainit=ai, 
                                                     tau=t, rc=rch, nu_in=nin, nu_out=nout))]

print(len(sam_data))
print(len(sam_data[0]))
calc_and_plot_dadt(sam_data,max_to_plot=20)

In [ ]:
nr = 100
sh = 4
gsmff = 2
gpff = 0
ai = 1.0e4
sam_data = []
for t in [0.1,1.0,10.0]:
    for rch in [10]:
        for nin in [-1.0]:
            for nout in [2.5]:
                sam_data = sam_data + [(get_sam_dadt(nrads=nr, shape=sh, gsmf_flag=gsmff, 
                                                     gpf_flag=gpff, ainit=ai, 
                                                     tau=t, rc=rch, nu_in=nin, nu_out=nout))]

print(len(sam_data))
print(len(sam_data[0]))
calc_and_plot_dadt(sam_data)

In [ ]:
nr = 100
sh = 4
gsmff = 2
gpff = 0
ai = 1.0e4
sam_data = []
for t in [0.1,1.0,10.0]:
    for rch in [100]:
        for nin in [-1.0]:
            for nout in [2.5]:
                sam_data = sam_data + [(get_sam_dadt(nrads=nr, shape=sh, gsmf_flag=gsmff, 
                                                     gpf_flag=gpff, ainit=ai, 
                                                     tau=t, rc=rch, nu_in=nin, nu_out=nout))]

print(len(sam_data))
print(len(sam_data[0]))
calc_and_plot_dadt(sam_data)

In [ ]:
nr = 100
sh = 4
gsmff = 2
gpff = 0
ai = 1.0e4
sam_data = []
for t in [0.1,1.0,10.0]:
    for rch in [10]:
        for nin in [-1.0]:
            for nout in [1.0]:
                sam_data = sam_data + [(get_sam_dadt(nrads=nr, shape=sh, gsmf_flag=gsmff, 
                                                     gpf_flag=gpff, ainit=ai, 
                                                     tau=t, rc=rch, nu_in=nin, nu_out=nout))]

print(len(sam_data))
print(len(sam_data[0]))
calc_and_plot_dadt(sam_data)

In [ ]:
nr = 100
sh = 4
gsmff = 2
gpff = 0
ai = 1.0e4
sam_data = []
for t in [0.1,1.0,10.0]:
    for rch in [100]:
        for nin in [-0.5]:
            for nout in [2.0]:
                sam_data = sam_data + [(get_sam_dadt(nrads=nr, shape=sh, gsmf_flag=gsmff, 
                                                     gpf_flag=gpff, ainit=ai, 
                                                     tau=t, rc=rch, nu_in=nin, nu_out=nout))]

print(len(sam_data))
print(len(sam_data[0]))
calc_and_plot_dadt(sam_data)

In [ ]:
calc_and_plot_dadt(nrads=100, shape=10, gsmf_flag=2, gpf_flag=0, tau=3.0, 
                   ainit=1.0e4, rc=100.0, nu_in=-1.0, nu_out=+1.5)

In [ ]:
calc_and_plot_dadt(nrads=100, shape=10, gsmf_flag=2, gpf_flag=0, tau=1.0, 
                   ainit=1.0e4, rc=0.0, nu_in=-1.0, nu_out=+2.5)

In [ ]:
calc_and_plot_dadt(nrads=100, shape=10, gsmf_flag=2, gpf_flag=0, tau=1.0, 
                   ainit=1.0e4, rc=10.0, nu_in=-1.0, nu_out=+2.5)

calc_and_plot_dadt(nrads=100, shape=10, gsmf_flag=2, gpf_flag=0, tau=3.0, 
                   ainit=1.0e3, rc=10.0, nu_in=-1.0, nu_out=+2.5)

In [ ]:
calc_and_plot_dadt(nrads=100, shape=10, gsmf_flag=2, gpf_flag=0, tau=1.0, 
                   ainit=1.0e3, rc=10.0, nu_in=-1.0, nu_out=+2.5)

In [ ]:
STEPS = 100

# () start from the hardening model's initial separation
rmax = hard._sepa_init
# (M,) end at the ISCO
rmin = utils.rad_isco(sam.mtot)
# rmin = hard._TIME_TOTAL_RMIN * np.ones_like(sam.mtot)
# Choose steps for each binary, log-spaced between rmin and rmax
extr = np.log10([rmax * np.ones_like(rmin), rmin])
radii = np.linspace(0.0, 1.0, STEPS)[np.newaxis, :]
# (M, X)
radii = extr[0][:, np.newaxis] + (extr[1] - extr[0])[:, np.newaxis] * radii
radii = 10.0 ** radii
# (M, Q, Z, X)
mt, mr, rz, rads = np.broadcast_arrays(
    sam.mtot[:, np.newaxis, np.newaxis, np.newaxis],
    sam.mrat[np.newaxis, :, np.newaxis, np.newaxis],
    sam.redz[np.newaxis, np.newaxis, :, np.newaxis],
    radii[:, np.newaxis, np.newaxis, :]
)
# (X, M*Q*Z)
#mt, mr, rz, rads = [mm.reshape(-1, STEPS).T for mm in [mt, mr, rz, rads]]
print(f'{sam.mtot.shape=}, {sam.mrat.shape=}, {sam.redz.shape=}, {radii.shape=}')
print(f'{mt.shape=}, {mr.shape=}, {rz.shape=}, {rads.shape=}')

# (X, M*Q*Z) --- `Fixed_Time.dadt` will only accept this shape
dadt = hard.dadt(mt, mr, rads)
print(f"{dadt.shape=}")


In [ ]:
##pta_dur = 16.03 * YR
#nfreqs = 40
##hifr = nfreqs/pta_dur
##pta_cad = 1.0 / (2 * hifr)
#fobs_cents = holo.utils.nyquist_freqs(pta_dur, pta_cad)
#fobs_edges = holo.utils.nyquist_freqs_edges(pta_dur, pta_cad)

nreals = 10
nloud = 1

freqs, freqs_edges = utils.pta_freqs()
gwb = sam.gwb(freqs, hard, realize=nreals, loudest=nloud, params=True)

In [ ]:
print(utils.stats(rads[-1,-1,-1,:]/dadt[-1,-1,-1,:]/GYR))
plt.xscale('log')
plt.yscale('log')
nskip = 1
cmap = plot._get_cmap('viridis')
colors = cmap(np.linspace(0, 1, int(sam.mtot.size/nskip)+1))
lw = np.arange(0,int(sam.mrat.size/nskip)+1, 0.1)

for i in np.arange(0,sam.mtot.size,nskip):
    for j in np.arange(0,sam.mrat.size,nskip):
        plt.plot(rads[i,j,0,:]/PC, -rads[i,j,0,:]/dadt[i,j,0,:]/GYR, 
                 alpha=0.5, color=colors[i], lw=lw[j])

In [ ]:
tt = times_evo[-1,-1,-1, :]/GYR
fig, ax = plot.figax(scale='lin')
print(utils.stats(tt))
kale.dist1d(tt, density=True)
plt.show()

In [ ]:
nbins = [5, 10, 123, 0]
_, fit_lamp, fit_plaw, fit_med_lamp, fit_med_plaw = holo.librarian.fit_spectra(fobs_cents, gwb, nbins=nbins)

In [ ]:
num_snaps = len(nbins)
fig, axes = plt.subplots(figsize=[10, 5], ncols=2)
for med, fits, ax in zip([fit_med_lamp, fit_med_plaw], [fit_lamp, fit_plaw], axes):
    for ii, nn in enumerate(nbins):
        if np.all(fits[:, ii] == 0.0):
            continue
        color = ax._get_lines.get_next_color()
        kale.dist1d(fits[:, ii], ax=ax, label=str(nn), color=color)
        ax.axvline(med[ii], ls='--', color=color)
    
    ax.legend()
    
plt.show()


In [ ]:
fig = plot.plot_gwb(fobs_cents, gwb)
ax = fig.axes[0]

xx = fobs_cents * YR
yy = 1e-15 * np.power(xx, -2.0/3.0)
ax.plot(xx, yy, 'r-', alpha=0.5, lw=1.0, label="$10^{-15} \cdot f_\\mathrm{yr}^{-2/3}$")

fits = holo.librarian.get_gwb_fits_data(fobs_cents, gwb)

for ls, idx in zip([":", "--"], [1, -1]):
    med_lamp = fits['fit_med_lamp'][idx]
    med_plaw = fits['fit_med_plaw'][idx]
    yy = (10.0 ** med_lamp) * (xx ** med_plaw)
    label = fits['fit_nbins'][idx]
    label = 'all' if label in [0, None] else label
    ax.plot(xx, yy, color='k', ls=ls, alpha=0.5, lw=2.0, label=str(label) + " bins")

label = fits['fit_label'].replace(" | ", "\n")
fig.text(0.99, 0.99, label, fontsize=6, ha='right', va='top')

ax.legend()
plt.show()


In [ ]:
fig = plot.plot_gwb(fobs_cents, gwb)
ax = fig.axes[0]

xx = fobs_cents * YR
yy = np.median(gwb, axis=-1)
ax.plot(xx, yy, 'k:')

for nn in [5, 10, None]:
    xx, amp, gamma = holo.librarian.fit_powerlaw(fobs_cents, np.median(gwb, axis=-1), nn)
    ax.plot(xx, amp * (xx ** gamma), ls='--')

plt.show()


In [ ]:
fig = plot.plot_gwb(fobs_cents, gwb, nsamp=None)
ax = fig.axes[0]

xx = fobs_cents * YR
yy = np.median(gwb, axis=-1)
ax.plot(xx, yy, 'k-')

nreals = gwb.shape[1]

fits = np.zeros((nreals, 2))
for nn in range(nreals):
    yy = gwb[:, nn]
    xx, *fits[nn, :] = holo.librarian.fit_powerlaw(fobs_cents, yy, 5)
    cc, = ax.plot(xx, fits[nn, 0] * (xx ** fits[nn, 1]), ls='--', alpha=0.5)
    cc = cc.get_color()
    ax.plot(fobs_cents*YR, yy, color=cc, alpha=0.5)

plt.show()

draw_fits = fits.copy()
draw_fits[:, 0] = np.log10(draw_fits[:, 0])

kale.corner(draw_fits.T)
plt.show()


In [ ]:
hard_time=-2.2957907176750907
hard_gamma_inner=-1.3335554512862717
gsmf_phi0=-2.802178096487384
gsmf_mchar0=11.704311872442908
gsmf_alpha0=-1.7179504809027346
gpf_zbeta=2.397456708546681
gpf_qgamma=0.4609649227136603
gmt_norm=0.5765308121579338
gmt_zbeta=-0.26777937808636665
mmb_amp=8.301258575486393
mmb_plaw=0.4785954601355894
mmb_scatter=0.12386778329303819

hard_time = (10.0 ** hard_time) * GYR
gmt_norm = gmt_norm * GYR
mmb_amp = (10.0 ** mmb_amp) * MSOL

gsmf = holo.sam.GSMF_Schechter(phi0=gsmf_phi0, mchar0_log10=gsmf_mchar0, alpha0=gsmf_alpha0)
gpf = holo.sam.GPF_Power_Law(qgamma=gpf_qgamma, zbeta=gpf_zbeta)
gmt = holo.sam.GMT_Power_Law(time_norm=gmt_norm, zbeta=gmt_zbeta)
mmbulge = holo.host_relations.MMBulge_KH2013(mamp=mmb_amp, mplaw=mmb_plaw, scatter_dex=mmb_scatter)

sam = holo.sam.Semi_Analytic_Model(
    gsmf=gsmf, gpf=gpf, gmt=gmt, mmbulge=mmbulge,
    shape=20
)
hard = holo.hardening.Fixed_Time.from_sam(
    sam, hard_time, gamma_sc=hard_gamma_inner,
    progress=False
)
pta_dur = 16.03 * YR
nfreqs = 40
hifr = nfreqs/pta_dur
pta_cad = 1.0 / (2 * hifr)
fobs_cents = holo.utils.nyquist_freqs(pta_dur, pta_cad)
fobs_edges = holo.utils.nyquist_freqs_edges(pta_dur, pta_cad)
gwb = sam.gwb(fobs_edges, realize=10, hard=hard)

plot.plot_gwb(fobs_cents, gwb)
plt.show()


In [ ]:
#SHAPE = None
SHAPE = 30
TIME = 1.0 * GYR

sam = holo.sam.Semi_Analytic_Model(shape=SHAPE)
hard = holo.hardening.Fixed_Time.from_sam(sam, TIME, interpolate_norm=False)

In [ ]:
STEPS = 100

# () start from the hardening model's initial separation
rmax = hard._sepa_init
# (M,) end at the ISCO
rmin = utils.rad_isco(sam.mtot)
# rmin = hard._TIME_TOTAL_RMIN * np.ones_like(sam.mtot)
# Choose steps for each binary, log-spaced between rmin and rmax
extr = np.log10([rmax * np.ones_like(rmin), rmin])
rads = np.linspace(0.0, 1.0, STEPS)[np.newaxis, :]
# (M, X)
rads = extr[0][:, np.newaxis] + (extr[1] - extr[0])[:, np.newaxis] * rads
rads = 10.0 ** rads
# (M, Q, Z, X)
mt, mr, rz, rads = np.broadcast_arrays(
    sam.mtot[:, np.newaxis, np.newaxis, np.newaxis],
    sam.mrat[np.newaxis, :, np.newaxis, np.newaxis],
    sam.redz[np.newaxis, np.newaxis, :, np.newaxis],
    rads[:, np.newaxis, np.newaxis, :]
)
# (X, M*Q*Z)
mt, mr, rz, rads = [mm.reshape(-1, STEPS).T for mm in [mt, mr, rz, rads]]
# (X, M*Q*Z) --- `Fixed_Time.dadt` will only accept this shape
dadt = hard.dadt(mt, mr, rads)
# Integrate (inverse) hardening rates to calculate total lifetime to each separation
times_evo = -utils.trapz_loglog(-1.0 / dadt, rads, axis=0, cumsum=True)


In [ ]:
tt = times_evo[-1, :]/GYR
fig, ax = plot.figax(scale='lin')
print(utils.stats(tt))
kale.dist1d(tt, density=True)
plt.show()

In [ ]:
import numpy as np
test_alpha = np.arange(-1,1.1,0.1)
print(test_alpha)

In [ ]:
import matplotlib.pyplot as plt
plt.plot(test_alpha, test_alpha, label='agw[rg]')
plt.plot(test_alpha, test_alpha+1, label='agw[pc]')
plt.plot(test_alpha, -3*test_alpha, label='dadt')
plt.plot(test_alpha, 4*test_alpha+1, label='a/dadt')
plt.plot([-1,1], [-2,-2], 'k', lw=4)
plt.plot([-1,1], [2,2], 'k', lw=4)
plt.plot([-1,1], [-1,-1], 'gray', lw=4)
plt.plot([-1,1], [1,1], 'gray',lw=4)
plt.plot([-2/3,-2/3],[-2,2], 'k')
plt.plot([0.25,0.25],[-2,2], 'k')
plt.plot([-1/3,-1/3],[-1,1], 'gray')
plt.plot([0,0],[-1,1], 'gray')
plt.plot([0.5,0.5],[-3,3], 'k:')
plt.legend()

In [ ]:
#suite_type = 'new_hardening_type0_toutvar'
suite_type = 'new_hardening_type0_foovar'
varied_values = dict(
            tout=[],
            nuivar=[],
            r9=[],
            alph=[]
        )
varName = [m for m in varied_values.keys() if m in suite_type]
print(f"{varName=}, {varied_values.keys()=}")
if len(varName)==1:         
    varName = varName[0]
else:
    print('blerg.')
print(varName)